# LSM-ECG thesis: does a proper liquid help? Y vs Y-off
**Y** is Maass's liquid built by his recipe ((c), Maass, Natschläger & Markram 2002): the 15 × 3 × 3 cube of 135 neurons. **Y-off** is the same liquid with its recurrent synapses removed. The question: does Y do better than Y-off?

* **Stage "build"** (Maass's own input, no ECG): (e)'s gain measure, and (c)'s four-segment memory task at Maass's own weights. Stop point: Y must keep the older segments better than Y-off on every seed.
* **Stage "ecg_setup"** (DS1 beats, labels unused): Y on config 11's front end. The input is scaled so Y's input-receiving neurons fire as they do on Maass's own input (`Y_INPUT_RULE`); the gain is chosen by (e)'s rank measure (Legenstein & Maass 2007).
* **Stage "potential"** (DS1 only): **A.** memory on ECG, Y vs Y-off (no labels). **B.** quick classification with (a)'s readout on one repetition of the authors' 4 folds: snapshot (R + 300 ms) and whole-beat readouts, for Y, Y-off, X (the paper's liquid, bugs fixed) and the input alone. **The agreed rule:** Y has potential if its snapshot Macro-F1 beats Y-off's on every seed.

* **Stages "claim3" and "claim7"** (no liquid simulation): (a)'s claim that robust normalisation helps, tested on the replication's own runs (`REPLICATION_TABLE`); and (a)'s explanation that Multi-threshold coding works through the liquid, tested by repeating the encoder comparison with no liquid (DS1 CV, 8 settings, 12 splits).

The new runs never use DS2. Every step saves its result in `WORK_DIR/ytest/`, so a restarted session continues where it stopped; delete a step's file to redo it. The summary is `WORK_DIR/ytest/summary_ytest.md`.

## 0. Constants (edit here)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CONSTANTS — the only cell you normally need to edit                 ║
# ╚══════════════════════════════════════════════════════════════════════╝
# The liquid test: Y vs Y-off. Its results go to WORK_DIR/ytest/ (the replication's output folder can be reused).

# ---- Machine -----------------------------------------------------------
MACHINE = "local"                  # "kaggle" or "local" (your own Linux machine; set its paths below)

MACHINES = {
    "kaggle": {
        "INPUT_DIR":   "/kaggle/input/datasets/tienthinh22/mitdb-1-0-0/mit-bih-arrhythmia-database-1.0.0",
        "WORK_DIR":    "/kaggle/working",
        "SCRATCH_DIR": "/kaggle/working/scratch",
        "RESUME_DIRS": ["/kaggle/input/datasets/tienthinh22/mitdb-common"],
        "SESSION_BUDGET_HOURS": 11.3,
        "DISK_LIMIT_GB": 19.0,
    },
    "local": {
        "INPUT_DIR":   "~/lsm_ecg/mitdb-1.0.0/mit-bih-arrhythmia-database-1.0.0",
        "WORK_DIR":    "~/lsm_ecg/output",          # any output folder; results go to WORK_DIR/ytest/
        "SCRATCH_DIR": None,
        "RESUME_DIRS": [],
        "SESSION_BUDGET_HOURS": 240.0,
        "DISK_LIMIT_GB": 250.0,
    },
}
_M = MACHINES[MACHINE]
INPUT_DIR            = _M["INPUT_DIR"]
WORK_DIR             = _M["WORK_DIR"]
SCRATCH_DIR          = _M["SCRATCH_DIR"] or WORK_DIR.rstrip("/") + "/scratch"
RESUME_DIRS          = _M["RESUME_DIRS"]
AUTO_RESUME_ROOT     = "/kaggle/input"
SESSION_BUDGET_HOURS = _M["SESSION_BUDGET_HOURS"]
DISK_LIMIT_GB        = _M["DISK_LIMIT_GB"]

# ---- This notebook ------------------------------------------------------
# One question: does a liquid built by Maass's recipe (Y, the 15 x 3 x 3 cube) do better than the same liquid with its
# recurrent synapses removed (Y-off)? Three stages, each saved in WORK_DIR/ytest/ and skipped once done:
#   build      : Y on Maass's own input (20 Hz Poisson): the gain measure of (e) and (c)'s four-segment memory task
#   ecg_setup  : Y on ECG beats (config 11's front end, DS1, labels unused): input size and the gain choice
#   potential  : A. memory on ECG (no labels); B. quick classification on DS1 (4 folds), Y vs Y-off (and X, input only)
#   claim3     : (a)'s "robust normalisation helps", tested on the replication's own runs (needs REPLICATION_TABLE)
#   claim7     : (a)'s "Multi-threshold works through the liquid": the encoder comparison with no liquid (DS1 CV)
# The new runs never touch DS2 (claim 3 reads the replication's DS2 runs).
Y_STAGES             = ["tests", "build", "ecg_setup", "potential", "claim3", "claim7"]
REPLICATION_TABLE    = None        # the replication's output folder (the one with runs/ inside), or its run table
                                   # tidy.csv; None skips claim 3 and the with-liquid half of claim 7
Y_SEEDS              = (1, 2, 3)   # liquid seeds for Y and Y-off (code choice: 3 seeds, as in the plan's ground rules)
X_SEEDS              = (2026, 69, 420)   # the replication's seeds for X (the paper's liquid, bugs fixed)
Y_INPUT_RULE         = "match_maass_rate"   # ECG input size: input neurons fire as on Maass's input; or "maass_per_spike"
Y_CV_REP             = 1           # B uses one repetition (4 folds) of the authors' grouped 4-fold CV on DS1
MAASS_SCRIPT_URL     = "https://raw.githubusercontent.com/brian-team/brian2/master/examples/frompapers/Maass_Natschlaeger_Markram_2002.py"   # (d), Brian2 examples
MAASS_SCRIPT_PATH    = None        # or a local copy of (d)'s maass2002.py (only the tests use it)

# ---- Session control ---------------------------------------------------
SELF_TEST            = False
SMOKE_TEST           = False       # True = tiny end-to-end test of the logic (few beats, coarse grid); the numbers mean nothing
BENCHMARK            = False
ANALYSIS_RESERVE_MIN = 25
RETRY_FAILED         = True
OLD_ANALYSIS         = False       # True = also rebuild the replication's tables and figures

# ---- Software ----------------------------------------------------------
PIN_TORCH_240  = False             # runs on the installed PyTorch
AUTHORS_REPO   = "https://github.com/Richard97Code/LSM-ECG-Classification.git"
AUTHORS_COMMIT = "02e45c4f3232886aed16a40d77de1f851ef4fcdd"

# ---- Experiment scope --------------------------------------------------
SEEDS = list(X_SEEDS)               # read by the reused cells

OLD_GROUPS = ["checks", "brian2", "main_final", "ablation_final", "rr_final", "baseline_final", "gain_final",
              "main_cv", "gain_cv", "ablation_cv", "rr_cv", "baseline_cv", "nn_tune", "retune", "kstale_final",
              "kstale_cv", "huang_final", "huang_cv", "meanrr_final", "meanrr_cv", "elmrr_final", "elmrr_cv",
              "rrliqmean_final", "rrliqmean_cv", "meanrr_nn", "hybrid", "working_diag", "working_final", "working_cv"]
GROUP_ORDER = OLD_GROUPS                     # read by the reused cells only; this notebook runs no replication group
GROUPS_ENABLED = {g: False for g in GROUP_ORDER}
SHARE = "BOTH"; SHARE_GROUPS = {"A": [], "B": []}
MERGE_DIRS = []

# replication settings that its cells still read (unchanged values)
GAINS = [1, 2, 3, 5, 8, 13, 21, 34]; DENSE_LAMBDAS = [1.5, 2.5]; RETUNE_TRIALS = 30
RETUNE_GRID = {"gain": [1, 2, 3, 5, 8, 13], "lambda_val": [0.8, 1.5, 2.5], "num_levels": [50, 100, 200, 400]}
NN_EPOCHS = 15; BRIAN2_BEATS = 10; NN_TUNE_TRIALS = 30; NN_PARAM_CAP = 35000; NN_TUNE_MAX_EPOCHS = 30
NN_TUNE_CHECKPOINTS = [5, 10, 15, 20, 25, 30]
HUANG_MATRICES = 15; HUANG_DIM = 50; HUANG_C = 1.0; HUANG_GAMMAS = [1.3, 1 / (2 * 1.3 ** 2)]
HUANG_THRESHOLD = 0.8; HUANG_SVM_CACHE_MB = 500; HYBRID_RULES = ["replace", "add"]
WORKING_LAMBDA = 2.0; WORKING_GAINS = [1, 3, 10, 30, 100, 300, 1000]; WORKING_DIAG_BEATS = 300
WORKING_T_PERTURB = 150; WORKING_MAX_FIRING = 0.2
SAVE_STATES = []

# ---- Performance -------------------------------------------------------
SIM_BATCH_CV    = 4096
SIM_BATCH_FINAL = 2000
NEW_SIM_BATCH   = 500      # beats per GPU batch for the new simulator: the same on every machine (plan, determinism)
PIPELINE        = True
FIT_WORKERS     = "auto"
CORE_SEARCH     = False
RAM_RESERVE_GB  = 4.0
READOUT_JOBS    = 4


## 1. Environment

In [ ]:
NOTEBOOK_BUILD = "5f162d0f696d"   # fingerprint of this build's code cells (do not edit)
# Environment: installs, imports, versions, device. Smoke-test overrides.
import os, sys, subprocess, time, json, platform, shutil, glob, hashlib, traceback, math, re, gc, random
import importlib, importlib.util, datetime, itertools, warnings, contextlib, io, heapq, collections
warnings.filterwarnings("ignore")
SESSION_START = time.time()
ON_KAGGLE = os.path.isdir("/kaggle/working") and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
NOTEBOOK_BUILD = globals().get("NOTEBOOK_BUILD", "dev")      # fingerprint of the code cells, set by the build
if SHARE is None:
    SHARE = "BOTH"
if SHARE != "BOTH" and SHARE not in SHARE_GROUPS:
    raise ValueError(f"SHARE must be 'BOTH', 'A' or 'B', not {SHARE!r}")
MERGE_MODE = bool(MERGE_DIRS)
if MERGE_MODE and SHARE != "BOTH":
    print(f"merge mode: SHARE = {SHARE!r} ignored (the merged result is checked against the whole plan)")
    SHARE = "BOTH"
if MACHINE == "kaggle" and not ON_KAGGLE:
    print("NOTE: MACHINE = 'kaggle' but this does not look like a Kaggle session (paths may not exist).")
INPUT_DIR, WORK_DIR, SCRATCH_DIR = (os.path.abspath(os.path.expanduser(p)) for p in (INPUT_DIR, WORK_DIR, SCRATCH_DIR))
if isinstance(RESUME_DIRS, (list, tuple)):
    RESUME_DIRS = [os.path.abspath(os.path.expanduser(p)) for p in RESUME_DIRS]
os.makedirs(os.path.join(WORK_DIR, "logs"), exist_ok=True)
import threading
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # deterministic cuBLAS
SESSION_TAG = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
LOG_PATH = os.path.join(WORK_DIR, "logs", f"session_{SESSION_TAG}.log")

class _Tee:
    """Everything the notebook prints (all cells, all threads, warnings and tracebacks) also goes to LOG_PATH,
    flushed line by line, so `tail -f WORK_DIR/logs/latest.log` shows it live from another terminal."""
    def __init__(self, stream, path):
        self._s, self._f, self._lock = stream, open(path, "a", buffering=1), threading.Lock()
    def write(self, text):
        with self._lock:
            try:
                self._f.write(text); self._f.flush()
            except Exception:
                pass
        return self._s.write(text)
    def flush(self):
        self._s.flush()
    def __getattr__(self, name):
        return getattr(self._s, name)

if getattr(sys.stdout, "_lsm_tee", False):           # notebook re-run in the same kernel: switch to the new log
    for _t in (sys.stdout, sys.stderr):
        _t._f = open(LOG_PATH, "a", buffering=1)
else:
    sys.stdout, sys.stderr = _Tee(sys.stdout, LOG_PATH), _Tee(sys.stderr, LOG_PATH)
    sys.stdout._lsm_tee = sys.stderr._lsm_tee = True
try:                                                    # a crash's traceback goes into the log file too
    def _exc_to_log(shell, etype, value, tb, tb_offset=None):
        with open(LOG_PATH, "a") as f:
            f.write("\n" + "".join(traceback.format_exception(etype, value, tb)))
        shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
    get_ipython().set_custom_exc((Exception,), _exc_to_log)
except NameError:
    pass
_latest = os.path.join(WORK_DIR, "logs", "latest.log")
try:
    if os.path.islink(_latest) or os.path.exists(_latest):
        os.remove(_latest)
    os.symlink(os.path.basename(LOG_PATH), _latest)
except OSError:
    pass
print(f"session {SESSION_TAG}: everything printed is also written to {LOG_PATH} (and {_latest})")

N_CPU = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
if PIPELINE:
    # one fit per core: the maths libraries must not start their own thread pools inside each fit
    for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        os.environ[_v] = "1"

import importlib.metadata as _md

def _ver(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None

def _pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-3000:]

# Kaggle's core packages must not change version mid-session (a NumPy swap after it is loaded breaks imports),
# so every install is constrained to the versions already present; pip then picks compatible package versions.
CORE = ("numpy", "scipy", "pandas", "matplotlib", "torch")
CORE_BEFORE = {p: _ver(p) for p in CORE}
INSTALL_REPORT = []
if not ON_KAGGLE:
    # Your own machine: you install the packages (in a virtual environment); here we only check.
    _need = {"numpy": None, "scipy": None, "pandas": None, "matplotlib": None, "torch": None, "snntorch": "0.9.4",
             "wfdb": None, "tqdm": None, "scikit-learn": "1.7.2", "joblib": None, "threadpoolctl": None, "brian2": None}
    _miss = [p for p in _need if _ver(p) is None]
    if _miss:
        raise RuntimeError(f"missing packages {_miss}: pip install them in the virtual environment, activate it, then run the notebook")
    for p, want in _need.items():
        if want and _ver(p) != want:
            print(f"WARNING: {p} {_ver(p)} installed, {want} expected")
    INSTALL_REPORT.append({"package": "(local machine: installed by hand)", "status": "checked", "tail": ""})
    if _ver("brian2") is None:
        print("WARNING: Brian2 is not installed; the brian2 check will be marked FAILED, everything else runs")
if ON_KAGGLE:
    os.makedirs(SCRATCH_DIR, exist_ok=True)
    def _constraints(skip=()):
        path = os.path.join(SCRATCH_DIR, "pip_constraints.txt")
        with open(path, "w") as f:
            f.write("\n".join(f"{p}=={v}" for p, v in CORE_BEFORE.items() if v and p not in skip) + "\n")
        return path
    if PIN_TORCH_240 and sys.version_info[:2] <= (3, 12):
        code, out = _pip("torch==2.4.0", "-c", _constraints(skip=("torch",)))
        INSTALL_REPORT.append({"package": "torch==2.4.0", "status": "ok" if code == 0 else "FAILED", "tail": out})
    elif PIN_TORCH_240:
        INSTALL_REPORT.append({"package": "torch==2.4.0", "status": "skipped",
                               "tail": f"Python {platform.python_version()} has no torch 2.4.0; using Kaggle's torch {CORE_BEFORE['torch']}"})
    for pkg, need in [("snntorch==0.9.4", "required"), ("wfdb", "required"), ("tqdm", "required"),
                      ("scikit-learn==1.7.2", "preferred"), ("brian2", "optional")]:
        code, out = _pip(pkg, "-c", _constraints())
        INSTALL_REPORT.append({"package": pkg, "need": need, "status": "ok" if code == 0 else "FAILED", "tail": out})
    for r in INSTALL_REPORT:
        print(f"install {r['package']:<22} -> {r['status']}" + (f"  ({r['tail']})" if r["status"] == "skipped" else ""))
    bad = [r for r in INSTALL_REPORT if r["status"] == "FAILED" and r.get("need") == "required"]
    if bad:
        raise RuntimeError("Required install failed (is Internet ON in the notebook settings?):\n" + bad[0]["tail"])
    for r in INSTALL_REPORT:
        if r["status"] == "FAILED" and r.get("need") == "preferred":
            print(f"WARNING: {r['package']} could not be installed; using Kaggle's scikit-learn {_ver('scikit-learn')}")
        if r["status"] == "FAILED" and r.get("need") == "optional":
            print("WARNING: Brian2 could not be installed; the brian2 check will be marked FAILED, everything else runs")
    changed = {p: (CORE_BEFORE[p], _ver(p)) for p in CORE if _ver(p) != CORE_BEFORE[p]
               and not (p == "torch" and any(r["package"] == "torch==2.4.0" and r["status"] == "ok" for r in INSTALL_REPORT))}
    if changed:
        raise RuntimeError(f"pip changed core packages {changed}. Restart the session and run again.")

import numpy as np, pandas as pd, torch, snntorch, sklearn, scipy, wfdb, joblib
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedGroupKFold
from scipy import stats as sstats
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TORCH_PINNED = torch.__version__.split("+")[0] == "2.4.0"

def gpu_supported(cap, arch_list):
    """Can this PyTorch build run on a GPU of compute capability cap=(major, minor)? A GPU runs code built for its own
    major version and a lower or equal minor (sm_61 runs sm_60), or PTX (compute_XY) at or below its capability."""
    for a in arch_list:
        kind, _, num = a.partition("_")
        if not num.isdigit():
            continue
        maj, mnr = int(num[:-1]), int(num[-1])
        if kind == "sm" and maj == cap[0] and mnr <= cap[1]:
            return True
        if kind == "compute" and (maj, mnr) <= tuple(cap):
            return True
    return False

if DEVICE.type == "cuda":
    _cap = torch.cuda.get_device_capability(0)
    _gpu = torch.cuda.get_device_name(0)
    _archs = torch.cuda.get_arch_list()
    _hint = (f"Install a PyTorch build that includes it: on Python <= 3.12 'pip install torch==2.4.0' (the authors' version); "
             f"otherwise a CUDA 12.6 build, 'pip install \"torch<2.15\" --index-url https://download.pytorch.org/whl/cu126'. "
             f"Older GPUs (e.g. Pascal sm_6x) are not in the CUDA 12.8+ / 13.x builds.")
    if not gpu_supported(_cap, _archs):
        raise RuntimeError(f"PyTorch {torch.__version__} (CUDA {torch.version.cuda}) has no code for this GPU: {_gpu}, "
                           f"compute capability {_cap[0]}.{_cap[1]}; this build supports {_archs}. {_hint}")
    try:
        torch.ones(8, device=DEVICE).add_(1).sum().item()          # a real kernel launch, not just the device query
    except Exception as ex:
        raise RuntimeError(f"PyTorch cannot run on {_gpu} (capability {_cap[0]}.{_cap[1]}): {ex}. {_hint}")
    print(f"GPU: {_gpu}, compute capability {_cap[0]}.{_cap[1]}: supported by PyTorch {torch.__version__} (CUDA {torch.version.cuda})")
# Plain float32 everywhere: Ampere+ GPUs (e.g. RTX 3090) would otherwise run convolutions in TF32 (a T4 cannot)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)   # PyTorch's deterministic algorithms on
if PIPELINE:
    from threadpoolctl import threadpool_limits
    threadpool_limits(1)                 # same as the environment variables above, in case NumPy was already loaded
    torch.set_num_threads(max(1, N_CPU // 4) if DEVICE.type == "cpu" else 1)
import threading
READOUT_EPOCHS = 15            # authors' value (part of the replication, not a tuning knob)

def _ram_gb():
    try:
        with open("/proc/meminfo") as f:
            return round(int(f.readline().split()[1]) / 1e6, 1)
    except Exception:
        return None

ENV = {
    "python": platform.python_version(), "platform": platform.platform(),
    "torch": torch.__version__, "torch_pinned_2_4_0": TORCH_PINNED, "snntorch": snntorch.__version__,
    "sklearn": sklearn.__version__, "numpy": np.__version__, "scipy": scipy.__version__,
    "pandas": pd.__version__, "wfdb": wfdb.__version__, "device": str(DEVICE),
    "cpu_count": N_CPU, "ram_gb": _ram_gb(), "on_kaggle": ON_KAGGLE, "machine": MACHINE, "smoke_test": SMOKE_TEST,
    "pipeline": PIPELINE, "cpu_model": next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), "?")
                 if os.path.exists("/proc/cpuinfo") else platform.processor(),
}
if DEVICE.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    ENV.update({"gpu": p.name, "gpu_sm_count": p.multi_processor_count, "gpu_capability": f"{p.major}.{p.minor}",
                "gpu_mem_gb": round(p.total_memory / 1e9, 1), "cuda": torch.version.cuda})
try:
    import brian2
    ENV["brian2"] = brian2.__version__
except Exception as e:
    ENV["brian2"] = f"unavailable ({e.__class__.__name__})"

if SMOKE_TEST:
    SEEDS = SEEDS[:1]
    GAINS = [1, 21]
    DENSE_LAMBDAS = DENSE_LAMBDAS[:1]
    RETUNE_TRIALS = 2
    NN_EPOCHS = 1
    BRIAN2_BEATS = 2
    NN_TUNE_TRIALS = 2
    NN_TUNE_MAX_EPOCHS = 2
    NN_TUNE_CHECKPOINTS = [1, 2]
    HUANG_MATRICES = 2
    WORKING_DIAG_BEATS = 40
    SAVE_STATES = [s for s in SAVE_STATES if s["seed"] in SEEDS]
SMOKE_CONFIGS   = [11, 12, 13, 22]       # used only when SMOKE_TEST
SMOKE_CV_SPLITS = [(1, 1)]               # used only when SMOKE_TEST
SMOKE_PER_CLASS = {0: 10, 1: 5, 2: 5}    # beats kept per record and class when SMOKE_TEST

for k, v in ENV.items():
    print(f"{k:>20}: {v}")
if PIN_TORCH_240 and not TORCH_PINNED:
    print("NOTE: torch 2.4.0 is not active; the exact-replication comparison will still run but cannot match.")


## 2. The authors' code (pinned commit)

In [ ]:
# The authors' code, cloned at a pinned commit. Their functions are used directly;
# bugs are patched only inside the "fixed" liquid modes (see the liquid cell).
REPO_DIR = os.path.join(SCRATCH_DIR, "authors_repo")
os.makedirs(SCRATCH_DIR, exist_ok=True)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "-q", AUTHORS_REPO, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", AUTHORS_COMMIT], check=True)
ENV["authors_commit"] = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                                       capture_output=True, text=True).stdout.strip()
assert ENV["authors_commit"] == AUTHORS_COMMIT, ENV["authors_commit"]
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.preprocessing import signals as A_sig
from src.preprocessing import stratification as A_strat
from src.encoding import spike_coding as A_enc
from src.utils.reproducibility import seed_everything

def _load_script(name, rel):
    spec = importlib.util.spec_from_file_location(name, os.path.join(REPO_DIR, rel))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)          # only imports + definitions; their main blocks are guarded
    return mod

A03 = _load_script("authors_cv_script", "scripts/03_run_24_configurations_cv.py")
LSM3D = A03.LiquidStateMachine3D           # identical class in scripts 02 and 03 (checked)

AUTHORS_CONFIGS = pd.read_csv(os.path.join(REPO_DIR, "configs/all_24_configurations.csv"))
ENC_SHORT = {"multithreshold": "multi", "temporal_rate": "rate", "bounded_delay_latency": "latency"}
CONFIGS = {int(r.config_id): {"config_id": int(r.config_id), "filter": r.filtering, "norm": r.normalization,
                              "encoder": ENC_SHORT[r.encoding], "readout": "svm" if r.readout_loss == "hinge" else "lr",
                              "authors_csv": r.source_result_file}
           for r in AUTHORS_CONFIGS.itertuples()}
print(f"authors' code at {ENV['authors_commit'][:10]}; {len(CONFIGS)} configurations loaded")


## 3. The liquid code (lsm2)

In [ ]:
# The liquid code (lsm2: liquids, simulator, measures, gain rule, inputs) and its tests, written to SCRATCH_DIR
# and imported. The same files are shipped next to this notebook; this cell is generated from them.
LSM2_FILES = {}
LSM2_FILES['lsm2/config.py'] = r'''"""Every fixed number of the redesign plan, with its source or reason.

Sources (as labelled in the plan):
  (a) Perez-Lopez et al. 2026 and the authors' code, commit 02e45c4
  (c) Maass, Natschlager & Markram 2002, Section 5 and Appendix B
  (d) Schmitt 2022, Brian2 reproduction of (c)'s Figure 2 (maass2002.py)
  (e) Legenstein & Maass 2007
Numbers marked "plan" are the plan's own choices; the reason is in the plan (section named).
Numbers marked "code" were not fixed by the plan; their reason is given here and listed in NOTES.md.
"""
import math

# ---------------------------------------------------------------- time
FS = 360.0                          # MIT-BIH sampling rate (Hz), (a)
SAMPLE_MS = 1000.0 / FS             # 2.7778 ms, one ECG sample = one step of (a)'s liquid
BEAT_SAMPLES = 577                  # (a): beats are R-0.8 s .. R+0.8 s at 360 Hz
SIM_SAMPLES = 650                   # (a): num_sim_steps = 650 (input is zero after the beat)
R_SAMPLE = 288                      # plan, Readouts: the R peak is sample 288 of the 577-sample window
WIN, NWIN = 20, 32                  # (a): spike counts in 32 windows of 20 samples (film readout)
SNAPSHOT_SAMPLES = (288, 342, 396)  # plan, Readouts: R, R + 150 ms, R + 300 ms
STATE_TAU_MS = 30.0                 # (c): liquid states are spikes filtered with a 30 ms exponential
N_SUB_REFERENCE = 28                # plan, Time grid: 28 sub-steps per sample (0.0992 ms) is the reference grid

# ---------------------------------------------------------------- X: (a)'s liquid, both bugs fixed
X = dict(
    betaE=0.95, betaI=0.80,         # (a) leak per 2.778 ms step (about 54 / 12 ms)
    threshold=1.0,                  # (a) Uthr
    rho=0.9,                        # (a) target fill of the paper's input rule
    excit_ratio=0.8,                # (a) and (c)
    C={"EE": 0.3, "EI": 0.2, "IE": 0.4, "II": 0.1},   # (a) and (c)
    lam=0.8,                        # (a)
    mu_rsv={"EE": 0.020, "EI": 0.025, "IE": 0.050, "II": 0.040},   # (a) type means
    P_in=0.1,                       # (a) code line 691
    dims=(7, 7, 7),                 # (a) 343 neurons
)
I_TARGET = X["rho"] * (1 - X["betaE"]) * X["threshold"]     # 0.045: (a)'s rule, mean input per step
TYPES = ("EE", "EI", "IE", "II")    # pre-post

# ---------------------------------------------------------------- Maass (c), in the paper's units (threshold 1)
MAASS = dict(
    tau_m_ms=30.0,                  # (c) membrane time constant
    v_thresh_mV=15.0, v_reset_mV=13.5, I_b_nA=13.5, R_Mohm=1.0,   # (c); background 13.5 mV of 15
    refrac_ms={"E": 3.0, "I": 2.0},  # (c)
    tau_s_ms={"E": 3.0, "I": 6.0},   # (c) synaptic current decay by presynaptic type
    tau_in_ms=3.0,                   # plan: input synapses are excitatory, so 3 ms for every target
    delay_ms={"EE": 1.5, "EI": 0.8, "IE": 0.8, "II": 0.8},   # (c)
    A_nA={"EE": 30.0, "EI": 60.0, "IE": -19.0, "II": -19.0},  # (c)
    U={"EE": 0.5, "EI": 0.05, "IE": 0.25, "II": 0.32},          # (c)
    D_s={"EE": 1.1, "EI": 0.125, "IE": 0.7, "II": 0.144},       # (c)
    F_s={"EE": 0.05, "EI": 1.2, "IE": 0.02, "II": 0.06},        # (c)
    udf_spread=0.5,                  # (c): SD 50% of the mean
    lam=2.0,                         # (c)
    in_nA={"E": 18.0, "I": 9.0},     # (c) input amplitudes
    in_fraction=0.3,                 # (c) input injected into 30% of neurons
    dims=(15, 3, 3),                 # (c) 135-neuron column
    input_rate_hz=20.0,              # (d) (and (c)'s Figure 4 task)
)
TAP = MAASS["I_b_nA"] * MAASS["R_Mohm"] / MAASS["v_thresh_mV"]    # 0.9 of threshold
RESET_TAP = MAASS["v_reset_mV"] / MAASS["v_thresh_mV"]            # 0.9

def _push(A_nA, tau_s_ms, U=1.0):
    """Voltage push (in thresholds) of one current pulse A*exp(-t/tau_s) through R into a 30 ms membrane:
    A * U * tau_s * R / (tau_m * V_thresh), the tap's convention (plan, factors 9 and 10)."""
    return A_nA * 1e-9 * U * tau_s_ms * 1e-3 * MAASS["R_Mohm"] * 1e6 / (MAASS["tau_m_ms"] * 1e-3 * MAASS["v_thresh_mV"] * 1e-3)

_pre = {"EE": "E", "EI": "E", "IE": "I", "II": "I"}
MAASS_PUSH = {t: abs(_push(MAASS["A_nA"][t], MAASS["tau_s_ms"][_pre[t]], MAASS["U"][t])) for t in TYPES}
# EE 0.1, EI 0.02, IE 0.0633, II 0.0811 (plan, factor 3: ratios 1 / 0.2 / 0.63 / 0.81; factor 10: EE 0.1)
MAASS_RATIOS = {t: MAASS_PUSH[t] / MAASS_PUSH["EE"] for t in TYPES}
MAASS_IN_PUSH = {k: _push(MAASS["in_nA"][k], MAASS["tau_in_ms"]) for k in ("E", "I")}   # 0.12 / 0.06 (factor 9)
MAASS_EE_PUSH = MAASS_PUSH["EE"]   # 0.1 (factor 10)

# ---------------------------------------------------------------- the 11 factors (plan table)
FACTOR_NAMES = {
    1: "reach lambda", 2: "code-only normalisations", 3: "E/I balance", 4: "leak", 5: "operating point",
    6: "input wiring", 7: "fine timing", 8: "dynamic synapses", 9: "input size", 10: "weight scale",
    11: "size and shape",
}
ALL_FACTORS = frozenset(range(1, 12))
LEAK_MS_Y = 30.0                    # factor 4: (c)'s 30 ms for E and I

# ---------------------------------------------------------------- gain grid (plan, Choosing the recurrent gain; decision 5)
G_LOW = (0.1, 0.3)
G_COARSE = (1, 3, 10, 30, 100, 300, 1000)
G_EXTEND_STEP, G_EXTEND_MAX = 3.0, 3
G_FINE_STEP, G_FINE_K = 1.4, 3      # g* x 1.4^k, k = -3..3 (1.4^3 ~ 2.7 splits a 3x step into three)

# ---------------------------------------------------------------- level thresholds (plan, Correctness criteria)
TOL_1A = 1e-9                       # 1a, 1b open loop (float64)
TOL_1C = 1e-4                       # 1c float32 vs float64
LEVEL2_SE = 4.5                     # level 2 (decision 24)
BAND_HZ = (2.0, 40.0)               # level 3 firing band (decision 7)
HIGH_HZ, HIGH_FRAC = 100.0, 0.05    # at most 5% of neurons above 100 Hz (decision 7)
N_CAL_BEATS = 500                   # plan: 500 beats for level 3 and the rank scores (decision 8)
GEN_BEATS, GEN_COPIES = 4, 125      # decision 8: 4 beats x 125 jittered copies
JITTER_MS = 4.0                     # decision 8: (e)'s 4 ms jitter for its 30-channel patterns
STATE_READ_SAMPLE = 396             # decision 8: states read at R + 300 ms
START_TEST_SAMPLE = 576             # level 3: starting-state rule judged at sample 576
MEMORY_T_MS = (0, 25, 50, 100, 200, 400)   # level 4 lags
MEMORY_PASS_T_MS = 100              # level 4 pass lag (plan: well past 30-54 ms membranes and 3-6 ms currents)
MEMORY_MIN_GAIN = 0.05              # level 4 (decision 9)
MEMORY_WARMUP = 100                 # level 4: 278 ms warm-up
SURROGATE_TOL = 0.10                # level 4: surrogate spikes per sample within 10% of ECG's
E1D_N_TRY_MAX = 28                  # 1d: n = 1, 2, 3, ... up to the reference grid
E1E_T_CRIT = 4.2                    # 1e: t at 8 df, two-sided 0.3%
E1E_FRAC = 0.95                     # 1e: 95% of time points
E1E_RUNS = 5                        # 1e: 5 runs per side

SEEDS_SEARCH = (1, 2, 3)            # code: liquid seeds for the ladder search (the plan fixes 3 seeds, not their values)
SEEDS_CONFIRM = (101, 202, 303)     # plan, confirmation
SEEDS_RETRY = (404, 505, 606)       # plan, confirmation retry
SEEDS_TEST_REDESIGN = (707, 808, 909)   # plan, Maass check redesign
SEEDS_CLASSIFY = (2026, 69, 420)    # plan, ground rules

# ---------------------------------------------------------------- code choices the plan does not fix (see NOTES.md)
CODE = dict(
    twin_beats=100,          # twin-run cross-check per g and seed: a reported cross-check, 100 beats give a stable mean ratio
    twin_early=(289, 306),   # samples of the first 50 ms after the extra spike
    twin_late=(559, 576),    # samples of the last 50 ms of the beat
    memory_sequences=500,    # level 4: same count as level 3's beats; 250 train / 250 test sequences
    ridge_alphas=tuple(10.0 ** k for k in range(-3, 5)),   # level 4 inner-CV penalty grid
    ridge_inner_folds=5,     # level 4 inner CV (grouped by sequence)
    calib_tol=0.01,          # factor 5: bisection stops at 1% of the target rate (well inside seed-to-seed spread)
    calib_iters=40,          # factor 5: bisection cap
    search_split_seed=0,     # DS1 patients halved by a fixed random permutation of the 22 records
    sim_batch=500,           # beats per GPU batch (fixed on every machine: plan, determinism)
    x_update="equations",    # X's reset follows (a)'s Eqs. 17-19 (the plan). "authors_code" (the snnTorch neuron of
                             # (a)'s code) exists only so the tests can show the difference; the notebook never sets it
)
'''
LSM2_FILES['lsm2/liquid.py'] = r'''"""Building liquids: every ladder candidate (X plus any set of the 11 factors), Maass's two versions for the
Maass check (script and paper), and X converted from the authors' own class (for 1f and E1's paper liquid).

A Liquid holds the realised network and every time constant already converted to its grid (n sub-steps per
input step). Input weights are stored as a base matrix; the input size (paper's rule, factor-5 calibration or
factor 9) is a separate scalar `in_scale`, and the recurrent gain g a scalar given to the simulator.
"""
from dataclasses import dataclass, field
import math
import numpy as np

from . import config as C

TYPE_INDEX = {t: i for i, t in enumerate(C.TYPES)}     # EE 0, EI 1, IE 2, II 3


def round_half_up(x):
    return int(math.floor(x + 0.5))


def steps_of(ms, h_ms):
    """A time in ms as a whole number of sub-steps: rounded to the nearest, at least 1 (plan, Time grid)."""
    return max(1, int(round(ms / h_ms)))


@dataclass
class Liquid:
    # structure
    N: int
    dims: tuple
    is_exc: np.ndarray          # (N,) bool
    pos: np.ndarray             # (N, 3)
    pre: np.ndarray             # (S,) int
    post: np.ndarray            # (S,) int
    typ: np.ndarray             # (S,) int 0..3 (EE, EI, IE, II)
    w: np.ndarray               # (S,) signed base weight: a push in thresholds ("ours"), or J amplitude ("brian2")
    U: np.ndarray               # (S,) utilisation (dynamic synapses), else ones
    D_ms: np.ndarray            # (S,)
    F_ms: np.ndarray            # (S,)
    Ubar: np.ndarray            # (4,) nominal type means of U
    Win: np.ndarray             # (C, N) base input weights, >= 0
    in_neurons: np.ndarray      # (N,) bool: neurons with any input synapse
    # time grid and neuron model
    n_sub: int                  # sub-steps per input step
    h_ms: float                 # sub-step length
    beta: np.ndarray            # (N,) leak per sub-step
    tap: np.ndarray             # (N,) tap per sub-step ("ours"); 0 without factor 5
    reset_mode: str             # "subtract" or "value"
    reset_value: float
    start_lo: float             # starting voltages drawn uniformly in [start_lo, start_hi]; equal = fixed
    start_hi: float
    rest: float                 # resting level (0, or 0.9 with the tap): lower end of level 3's random starts
    gap: np.ndarray             # (N,) int minimum gap between spikes, in sub-steps
    clamp: bool                 # voltage clamped during the blocked sub-steps
    strict: bool                # threshold test u > 1 (Brian2 script) instead of u >= 1 (ours, (a)'s H)
    schedule: str               # "ours" or "brian2"
    syn_model: str              # "static", "markram" (two-state, (c)'s form) or "tsodyks3" ((d)'s three-state)
    delay: np.ndarray           # (4,) int sub-steps by type
    cur_a: object               # None (push at once) or (2,) decay per sub-step for E- and I-presynaptic currents
    in_a: object                # None or (N,) decay per sub-step of each neuron's input current
    # Brian2-schedule constants (script version only)
    J_b: float = 0.0
    kappa: object = None        # (N,) exact weight of the decaying input current in the membrane update
    syn_tau_ms: object = None   # (4,) current decay per type (tsodyks3)
    # bookkeeping
    factors: frozenset = frozenset()
    snn_quirk: bool = False     # (a)'s code (snnTorch Leaky, reset_delay=False): the threshold test subtracts 1 more
                                # when the stored voltage is above 1; u > 1 strict. Default: (a)'s Eqs. 17-19
    g_maass: float = None       # g at Maass's own scale (factor 10)
    in_scale_fixed: float = None   # input size fixed by the design (factor 9, Maass versions)
    in_split: tuple = (1.0, 1.0)   # base input means onto E / I neurons (2:1 with factors 5 or 9)
    meta: dict = field(default_factory=dict)

    @property
    def S(self):
        return len(self.pre)

    def ee_mean_push(self, g=1.0):
        """Mean isolated EE push per presynaptic spike, in thresholds (the plan's absolute g scale)."""
        m = self.typ == 0
        if not m.any():
            return 0.0
        iso = np.abs(self.w[m]) * (self.U[m] / self.Ubar[0] if self.syn_model == "markram" else 1.0)
        return float(g * iso.mean())


# ---------------------------------------------------------------- shared pieces
def _grid(dims):
    return np.stack(np.meshgrid(*[np.arange(d) for d in dims], indexing="ij"), -1).reshape(-1, 3).astype(np.float64)


def _type_matrix(is_exc):
    pe, qe = is_exc[:, None], is_exc[None, :]
    T = np.where(pe & qe, 0, np.where(pe & ~qe, 1, np.where(~pe & qe, 2, 3)))
    return T


def _draw_udf(rng_n, rng_u, T, mean_of_type, cap, bad_rule):
    """U, D or F for every (pre, post) pair: Normal(mean, 0.5 mean); draws at or below 0 replaced by
    uniform(0, 2 mean) ("redraw", plan decision 17) or by the mean ("mean", (d)); U capped at 1 if cap."""
    mean = np.vectorize(lambda t: mean_of_type[C.TYPES[t]])(T).astype(np.float64)
    z = rng_n.standard_normal(T.shape)
    v = mean + C.MAASS["udf_spread"] * mean * z
    repl = rng_u.random(T.shape) * 2 * mean
    if bad_rule == "redraw":
        v = np.where(v <= 0, repl, v)
    else:      # (d): U < 0 -> mean; D, F <= 0 -> mean
        v = np.where(v <= 0, mean, v)
    if cap:
        v = np.minimum(v, 1.0)
    return v


# ---------------------------------------------------------------- ladder candidates
def candidate_settings(factors):
    f = frozenset(factors)
    return {
        "dims": C.MAASS["dims"] if 11 in f else C.X["dims"],
        "lam": C.MAASS["lam"] if 1 in f else C.X["lam"],
        "mu": ({t: C.X["mu_rsv"]["EE"] * C.MAASS_RATIOS[t] for t in C.TYPES} if 3 in f else dict(C.X["mu_rsv"])),
        "normalise": 2 not in f,
        "tau_E_ms": C.LEAK_MS_Y if 4 in f else -C.SAMPLE_MS / math.log(C.X["betaE"]),
        "tau_I_ms": C.LEAK_MS_Y if 4 in f else -C.SAMPLE_MS / math.log(C.X["betaI"]),
        "tap": 5 in f,
        "input30": 6 in f,
        "fine": 7 in f,
        "dynamic": 8 in f,
        "maass_input": 9 in f,
        "maass_scale": 10 in f,
    }


def build_candidate(factors, seed, n_sub, n_channels, n_sub_input_step_ms=C.SAMPLE_MS):
    """A ladder candidate: X ((a), bugs fixed) with the given factors switched on.
    n_sub: sub-steps per sample (1 without factor 7 unless a grid check asks otherwise)."""
    f = frozenset(factors)
    s = candidate_settings(f)
    N = int(np.prod(s["dims"]))
    h = n_sub_input_step_ms / n_sub
    # independent streams, so candidates with the same seed share their random numbers where they can
    r_types, r_conn, r_w = (np.random.default_rng([seed, k]) for k in (1, 2, 3))
    r_un, r_uu, r_dn, r_du, r_fn, r_fu = (np.random.default_rng([seed, k]) for k in (4, 5, 6, 7, 8, 9))
    r_inm, r_inw, r_inn = (np.random.default_rng([seed, k]) for k in (10, 11, 12))
    n_exc = int(N * C.X["excit_ratio"])
    is_exc = np.zeros(N, bool)
    is_exc[r_types.permutation(N)[:n_exc]] = True
    pos = _grid(s["dims"])
    T = _type_matrix(is_exc)
    Cm = np.vectorize(lambda t: C.X["C"][C.TYPES[t]])(T)
    d = np.sqrt(((pos[:, None, :] - pos[None, :, :]) ** 2).sum(-1))
    p = np.clip(Cm * np.exp(-(d / s["lam"]) ** 2), 0, 1)
    conn = r_conn.random((N, N)) < p
    np.fill_diagonal(conn, False)                            # no self-connections ((a); kept in Y)
    mu = np.vectorize(lambda t: s["mu"][C.TYPES[t]])(T)
    Wmag = r_w.standard_exponential((N, N)) * mu             # Gamma(shape 1, scale mu): spread = mean
    if s["normalise"]:
        fan_in = np.maximum(conn.sum(0, keepdims=True), 1)   # (a) lines 168-169: divide by the receiver's count
        Wmag = Wmag / fan_in
    W = np.where(is_exc[:, None], Wmag, -Wmag)
    pre, post = np.nonzero(conn)
    order = np.lexsort((post, pre, T[pre, post]))           # sorted by type, then pre, then post
    pre, post = pre[order], post[order]
    typ = T[pre, post]
    if s["dynamic"]:
        Ufull = _draw_udf(r_un, r_uu, T, C.MAASS["U"], True, "redraw")
        Dfull = _draw_udf(r_dn, r_du, T, {k: v * 1000 for k, v in C.MAASS["D_s"].items()}, False, "redraw")
        Ffull = _draw_udf(r_fn, r_fu, T, {k: v * 1000 for k, v in C.MAASS["F_s"].items()}, False, "redraw")
        U, D, F = Ufull[pre, post], Dfull[pre, post], Ffull[pre, post]
    else:
        U, D, F = np.ones(len(pre)), np.full(len(pre), np.inf), np.full(len(pre), np.inf)
    Ubar = np.array([C.MAASS["U"][t] for t in C.TYPES])
    # input wiring and base weights
    if s["input30"]:
        k = round_half_up(C.MAASS["in_fraction"] * N)
        in_n = np.zeros(N, bool)
        in_n[r_inn.permutation(N)[:k]] = True
        mask = np.repeat(in_n[None, :], n_channels, 0)
    else:
        mask = r_inm.random((n_channels, N)) < C.X["P_in"]
        in_n = mask.any(0)
    split = (1.0, 0.5) if (s["tap"] or s["maass_input"]) else (1.0, 1.0)
    m_in = np.where(is_exc, split[0], split[1])[None, :]
    Win = r_inw.standard_exponential((n_channels, N)) * m_in * mask
    if s["normalise"]:                                       # (a) lines 71-75: each column's mean active weight = m
        cnt = mask.sum(0, keepdims=True)
        col = Win.sum(0, keepdims=True) / (cnt + 1e-8)
        Win = Win * (m_in / (col + 1e-8))
    # time constants on the grid
    tau = np.where(is_exc, s["tau_E_ms"], s["tau_I_ms"])
    beta = np.exp(-h / tau)
    tap = C.TAP * (1 - beta) if s["tap"] else np.zeros(N)
    if s["fine"]:
        gap = np.where(is_exc, steps_of(C.MAASS["refrac_ms"]["E"], h), steps_of(C.MAASS["refrac_ms"]["I"], h))
        delay = np.array([steps_of(C.MAASS["delay_ms"][t], h) for t in C.TYPES])
        cur_a = np.exp(-h / np.array([C.MAASS["tau_s_ms"]["E"], C.MAASS["tau_s_ms"]["I"]]))
        in_a = np.full(N, math.exp(-h / C.MAASS["tau_in_ms"]))
        clamp = True
    else:
        # X: the 2.78 ms gap and the one-sample delay, in ms, converted to the grid in use
        gap = np.full(N, steps_of(C.SAMPLE_MS, h))
        delay = np.full(4, steps_of(C.SAMPLE_MS, h))
        cur_a, in_a, clamp = None, None, False
    liq = Liquid(
        N=N, dims=tuple(s["dims"]), is_exc=is_exc, pos=pos, pre=pre, post=post, typ=typ, w=W[pre, post].astype(np.float64),
        U=U, D_ms=D, F_ms=F, Ubar=Ubar, Win=Win.astype(np.float64), in_neurons=in_n,
        n_sub=n_sub, h_ms=h, beta=beta, tap=tap,
        reset_mode="value" if s["tap"] else "subtract", reset_value=C.RESET_TAP if s["tap"] else 0.0,
        start_lo=C.RESET_TAP if s["tap"] else 0.0, start_hi=1.0 if s["tap"] else 0.0,
        rest=C.TAP if s["tap"] else 0.0,
        gap=gap.astype(np.int64), clamp=clamp, strict=False, schedule="ours",
        syn_model="markram" if s["dynamic"] else "static", delay=delay.astype(np.int64), cur_a=cur_a, in_a=in_a,
        factors=f, in_split=split,
        meta={"kind": "candidate", "seed": int(seed), "n_channels": int(n_channels), "settings": s},
    )
    if liq.reset_mode == "subtract" and C.CODE["x_update"] == "authors_code":
        liq.snn_quirk, liq.strict = True, True
    m = typ == 0
    liq.g_maass = C.MAASS_EE_PUSH / float(np.abs(liq.w[m]).mean()) if m.any() else None
    if s["maass_input"]:
        liq.in_scale_fixed = C.MAASS_IN_PUSH["E"]           # base means 1 / 0.5 -> 0.12 / 0.06 per input spike
    return liq


def p_in_effective(liq):
    """Connection probability the paper's rule uses: P_in = 0.1, or 1 for the input neurons with factor 6."""
    return 1.0 if 6 in liq.factors else C.X["P_in"]


def paper_rule_scale(ain, liq):
    """(a)'s rule: mu_in = I_target / (A_in P_in), always with X's excitatory leak (plan, Input size)."""
    return C.I_TARGET / (ain * p_in_effective(liq))


# ---------------------------------------------------------------- Maass's liquid, two versions (Maass check)
def build_maass_paper(seed, n_sub=None, n_channels=1):
    """Our reading of (c), the version Y copies: Y with all 11 factors and one Poisson input channel."""
    n_sub = n_sub or C.N_SUB_REFERENCE
    liq = build_candidate(C.ALL_FACTORS, seed, n_sub, n_channels)
    liq.meta["kind"] = "maass_paper"
    return liq


def build_maass_script(seed, n_sub=None, step_ms=C.SAMPLE_MS, d_replacement="fixed"):
    """The liquid of (d)'s Brian2 script, in threshold units, with its six settings (plan, level 1e):
    input onto I neurons decays over 6 ms, bad U, D, F draws replaced by the mean, U not capped, no input delay,
    an arriving spike acts after that step's state update (Brian2 schedule), each step's synaptic current held
    constant (exact neuron update with step-constant synaptic current). Also from (d): input to all 135 neurons
    at a fixed 18 / 9 nA, self-connections allowed, E neurons are the first 108 indices with shuffled positions,
    three-state synapses (used resources recover only once the current has ended)."""
    n_sub = n_sub or C.N_SUB_REFERENCE
    N = int(np.prod(C.MAASS["dims"]))
    h = step_ms / n_sub
    rng = np.random.default_rng([seed, 1000])
    idx = rng.permutation(N)
    pos = np.stack([idx % 3, (idx // 3) % 3, idx // 9], 1).astype(np.float64)
    n_exc = int(0.8 * N)
    is_exc = np.zeros(N, bool); is_exc[:n_exc] = True
    T = _type_matrix(is_exc)
    Cm = np.vectorize(lambda t: C.X["C"][C.TYPES[t]])(T)
    d2 = ((pos[:, None, :] - pos[None, :, :]) ** 2).sum(-1)
    p = Cm * np.exp(-d2 / C.MAASS["lam"] ** 2)
    conn = np.random.default_rng([seed, 1001]).random((N, N)) < p          # self-connections allowed ((d))
    pre, post = np.nonzero(conn)
    order = np.lexsort((post, pre, T[pre, post]))
    pre, post = pre[order], post[order]
    typ = T[pre, post]
    J = C.MAASS["R_Mohm"] / C.MAASS["v_thresh_mV"]                         # nA -> thresholds (1 MOhm / 15 mV)
    A = np.array([C.MAASS["A_nA"][C.TYPES[t]] for t in typ])
    r = np.random.default_rng([seed, 1002])
    w = np.sign(A) * r.standard_exponential(len(pre)) * np.abs(A) * J       # Gamma(1, |A|) with sign ((d))
    def udf(name, scale, k):
        mean = np.array([C.MAASS[name][C.TYPES[t]] for t in typ]) * scale
        v = mean + 0.5 * mean * np.random.default_rng([seed, k]).standard_normal(len(pre))
        if d_replacement == "as_written":       # (d) as written: only the first two synapses of each group
            for ti in range(4):
                first = np.nonzero(typ == ti)[0][:2]
                v[first] = mean[first]
            return v
        return np.where(v < 0, mean, v) if name == "U" else np.where(v <= 0, mean, v)
    U, D, F = udf("U", 1.0, 1003), udf("D_s", 1000.0, 1004), udf("F_s", 1000.0, 1005)
    Ubar = np.array([C.MAASS["U"][t] for t in C.TYPES])
    Win = (np.where(is_exc, C.MAASS["in_nA"]["E"], C.MAASS["in_nA"]["I"]) * J)[None, :]   # J jump per input spike
    tau_m = C.MAASS["tau_m_ms"]
    beta = np.full(N, math.exp(-h / tau_m))
    in_tau = np.where(is_exc, 3.0, 6.0)                                     # (d): tau_stimulus 3 ms (E), 6 ms (I)
    a_in = np.exp(-h / in_tau)
    kappa = in_tau / (in_tau - tau_m) * (np.exp(-h / in_tau) - np.exp(-h / tau_m))   # Brian2's exact solution
    gap = np.where(is_exc, int((C.MAASS["refrac_ms"]["E"] + 1e-3 * h) / h), int((C.MAASS["refrac_ms"]["I"] + 1e-3 * h) / h))
    delay = np.array([int(round(C.MAASS["delay_ms"][t] / h)) for t in C.TYPES])   # Brian2 rounds delays to the grid
    syn_tau = np.array([C.MAASS["tau_s_ms"]["E"], C.MAASS["tau_s_ms"]["E"], C.MAASS["tau_s_ms"]["I"], C.MAASS["tau_s_ms"]["I"]])
    liq = Liquid(
        N=N, dims=C.MAASS["dims"], is_exc=is_exc, pos=pos, pre=pre, post=post, typ=typ, w=w, U=U, D_ms=D, F_ms=F,
        Ubar=Ubar, Win=Win, in_neurons=np.ones(N, bool), n_sub=n_sub, h_ms=h, beta=beta, tap=np.zeros(N),
        reset_mode="value", reset_value=C.RESET_TAP, start_lo=C.RESET_TAP, start_hi=1.0, rest=C.TAP,
        gap=gap.astype(np.int64), clamp=True, strict=True, schedule="brian2", syn_model="tsodyks3",
        delay=delay.astype(np.int64), cur_a=None, in_a=a_in, J_b=C.TAP, kappa=kappa, syn_tau_ms=syn_tau,
        factors=frozenset(), in_scale_fixed=1.0, in_split=(1.0, 0.5),
        meta={"kind": "maass_script", "seed": int(seed), "n_channels": 1, "d_replacement": d_replacement},
    )
    liq.g_maass = 1.0
    return liq


# ---------------------------------------------------------------- X from the authors' class
def from_authors_model(model, n_sub=1, snn_quirk=None):
    """X exactly as (a)'s class built it (weights already include its normalisations and its g_rec):
    used for the 1f regression and for E1's paper liquid at g = 1. Input weights are returned at their size
    (in_scale 1)."""
    import torch
    W = model.rsv_weights.detach().double().cpu().numpy()
    Win = model.input_rsv_weights.detach().double().cpu().numpy()
    is_exc = model.is_excit.detach().cpu().numpy().astype(bool)
    N = W.shape[0]
    T = _type_matrix(is_exc)
    pre, post = np.nonzero(W)
    order = np.lexsort((post, pre, T[pre, post]))
    pre, post = pre[order], post[order]
    h = C.SAMPLE_MS / n_sub
    tau = np.where(is_exc, -C.SAMPLE_MS / math.log(C.X["betaE"]), -C.SAMPLE_MS / math.log(C.X["betaI"]))
    liq = Liquid(
        N=N, dims=tuple(model.reservoir_dim), is_exc=is_exc, pos=_grid(model.reservoir_dim), pre=pre, post=post,
        typ=T[pre, post], w=W[pre, post], U=np.ones(len(pre)), D_ms=np.full(len(pre), np.inf),
        F_ms=np.full(len(pre), np.inf), Ubar=np.array([C.MAASS["U"][t] for t in C.TYPES]), Win=Win,
        in_neurons=(Win != 0).any(0), n_sub=n_sub, h_ms=h, beta=np.exp(-h / tau), tap=np.zeros(N),
        reset_mode="subtract", reset_value=0.0, start_lo=0.0, start_hi=0.0, rest=0.0,
        gap=np.full(N, steps_of(C.SAMPLE_MS, h)), clamp=False, strict=False, schedule="ours", syn_model="static",
        delay=np.full(4, steps_of(C.SAMPLE_MS, h)), cur_a=None, in_a=None, factors=frozenset(),
        in_scale_fixed=1.0, meta={"kind": "authors_X", "n_channels": int(Win.shape[0])},
    )
    if snn_quirk or (snn_quirk is None and C.CODE["x_update"] == "authors_code"):
        liq.snn_quirk, liq.strict = True, True
    return liq
'''
LSM2_FILES['lsm2/sim.py'] = r'''"""The simulator: one batched update for X, Y and every candidate between them (each factor is a setting of the
Liquid), plus the Brian2 schedule used only for (d)'s own liquid (level 1e and the Maass check's script version).

Units: thresholds (the voltage threshold is 1). Time: input steps (one ECG sample, or 0.1 ms for 1e), each split
into n sub-steps. Our order within a sub-step (plan, Simulation details):
  1. synapse states recover/fade over the sub-step; spikes whose delay ends now arrive (dynamic synapses
     update u, push, R in (d)'s order)
  2. currents: without fine timing a push arrives all at once; with it, it feeds a current that decays over
     3 ms (from E and from inputs) or 6 ms (from I), scaled so its total equals the push
  3. every neuron integrates: u <- beta u + input + recurrent + tap (not during a clamped refractory period)
  4. threshold test (u >= 1, outside the minimum gap), then reset (subtract 1, or set to 0.9 with the tap)
Input spikes of an input step act in its first sub-step ((a)'s Eq. 16: x[t] W_in in step t); this step's spikes
reach their targets from the next sub-step on (delay >= 1).
"""
import math
import numpy as np
import torch

from . import config as C


def _t(a, device, dtype):
    return torch.as_tensor(np.asarray(a), device=device, dtype=dtype)


class Simulator:
    def __init__(self, liq, device="cpu", dtype=torch.float64, g=1.0, in_scale=None, recurrence=True):
        self.liq, self.device, self.dtype = liq, torch.device(device), dtype
        self.g = float(g) if recurrence else 0.0
        if in_scale is None:
            if liq.in_scale_fixed is None:
                raise ValueError("this liquid needs an input size (paper rule or calibration)")
            in_scale = liq.in_scale_fixed
        self.in_scale = float(in_scale)
        d, t = self.device, dtype
        N = liq.N
        self.N = N
        self.beta = _t(liq.beta, d, t)
        self.tap = _t(liq.tap, d, t)
        self.gap = _t(liq.gap, d, torch.int64)
        self.Win = _t(liq.Win * self.in_scale, d, t)
        self.z_decay = math.exp(-liq.h_ms / C.STATE_TAU_MS)
        self.L = int(max(liq.delay.max(), 1)) + 1           # spike-history ring length
        S = liq.S
        self.S = S
        self.static = liq.syn_model == "static"
        if liq.schedule == "ours":
            if self.static:
                # one dense matrix per (delay, presynaptic class): pushes of the spikes that left `delay` sub-steps ago
                self.groups = {}
                for ti in range(4):
                    m = liq.typ == ti
                    if not m.any():
                        continue
                    key = (int(liq.delay[ti]), 0 if ti < 2 else 1)
                    W = self.groups.get(key)
                    if W is None:
                        W = torch.zeros(N, N, device=d, dtype=t)
                    W[_t(liq.pre[m], d, torch.int64), _t(liq.post[m], d, torch.int64)] += _t(liq.w[m] * self.g, d, t)
                    self.groups[key] = W
            else:
                self._prep_sparse()
                Ubar_s = liq.Ubar[liq.typ]
                self.wg = _t(liq.w * self.g / Ubar_s, d, t)        # push = (w / U-bar) u R
                self.U = _t(liq.U, d, t)
                self.eD = _t(np.exp(-liq.h_ms / liq.D_ms), d, t)
                self.eF = _t(np.exp(-liq.h_ms / liq.F_ms), d, t)
            if liq.cur_a is not None:
                self.aE, self.aI = float(liq.cur_a[0]), float(liq.cur_a[1])
            self.in_a = None if liq.in_a is None else _t(liq.in_a, d, t)
        else:   # Brian2 schedule, (d)'s three-state synapses, currents in voltage-equivalent units
            self._prep_sparse()
            self.wg = _t(liq.w * self.g, d, t)
            self.U = _t(liq.U, d, t)
            tau = liq.syn_tau_ms[liq.typ]
            D = liq.D_ms
            h = liq.h_ms
            self.ey = _t(np.exp(-h / tau), d, t)
            self.eD = _t(np.exp(-h / D), d, t)
            self.cxy = _t(tau * (np.exp(-h / tau) - np.exp(-h / D)) / (D - tau), d, t)
            self.eF = _t(np.exp(-h / liq.F_ms), d, t)
            self.kappa = _t(liq.kappa, d, t)
            self.in_a = _t(liq.in_a, d, t)
            self.J_b = float(liq.J_b)

    def _prep_sparse(self):
        liq, d, t = self.liq, self.device, self.dtype
        self.pre_idx = _t(liq.pre, d, torch.int64)
        self.syn_delay = liq.delay[liq.typ]
        self.delay_groups = [(int(dl), _t(np.nonzero(self.syn_delay == dl)[0], d, torch.int64))
                             for dl in np.unique(self.syn_delay)]
        self.post_idx = _t(liq.post, d, torch.int64)
        self.nE = int((liq.typ < 2).sum())                  # synapses are sorted by type: E-presynaptic first
        M = torch.zeros(liq.S, liq.N, device=d, dtype=t)
        if liq.S:
            M[torch.arange(liq.S, device=d), _t(liq.post, d, torch.int64)] = 1.0
        self.M_E, self.M_I, self.M = M[:self.nE], M[self.nE:], M

    # ------------------------------------------------------------------ helpers
    def _arrivals(self, hist, k, B):
        """(B, S) 0/1: which synapses receive a spike in sub-step k."""
        a = torch.zeros(B, self.S, device=self.device, dtype=self.dtype)
        for dl, idx in self.delay_groups:
            if k - dl < 0:
                continue
            src = hist[(k - dl) % self.L]
            a[:, idx] = src[:, self.pre_idx[idx]]
        return a

    def start_state(self, u0):
        return torch.as_tensor(u0, device=self.device, dtype=self.dtype).clone()

    # ------------------------------------------------------------------ main loop
    @torch.no_grad()
    def run(self, x, u0, T, rec=None, open_loop=None, trace=False, own_spikes=None):
        """x: (B, T_in, C) input spikes per input step; u0: (B, N) starting voltages; T: input steps to simulate.
        rec: {"windows": (win, nwin), "states_at": [t, ...], "states_every": (t0, t1), "neuron_counts": (t0, t1),
              "raster": bool}; times are input steps, a state is read at the end of its step.
        open_loop: (T*n, B, N) recorded spikes used as the presynaptic spikes (the liquid's own spikes are still
        computed and returned). own_spikes: (T*n, B, N) recorded spikes that also replace each neuron's own
        threshold decisions (level 1c: pure arithmetic drift, no flips); out["flips"] counts where the threshold
        test would have decided differently. trace: return u and currents every sub-step and synapse states
        every input step."""
        liq, dev, dt = self.liq, self.device, self.dtype
        rec = rec or {}
        x = torch.as_tensor(x, device=dev)
        B, T_in, Cin = x.shape
        N, n = self.N, liq.n_sub
        u = self.start_state(u0)
        ref = torch.zeros(B, N, device=dev, dtype=torch.int64)
        hist = torch.zeros(self.L, B, N, device=dev, dtype=dt)
        z = torch.zeros(B, N, device=dev, dtype=dt)
        ours = liq.schedule == "ours"
        fine = ours and liq.cur_a is not None
        IE = torch.zeros(B, N, device=dev, dtype=dt) if fine else None
        II = torch.zeros(B, N, device=dev, dtype=dt) if fine else None
        Iin = torch.zeros(B, N, device=dev, dtype=dt) if (self.in_a is not None) else None
        if not self.static or not ours:
            R = torch.ones(B, self.S, device=dev, dtype=dt)       # resources ready (x in (d))
            q = torch.zeros(B, self.S, device=dev, dtype=dt)      # utilisation u
            y = torch.zeros(B, self.S, device=dev, dtype=dt) if not ours else None
        out = {}
        win = rec.get("windows")
        if win:
            wl, nw = win
            counts = torch.zeros(B, nw, N, device=dev, dtype=torch.int32)
        st_at = sorted(set(rec.get("states_at", [])))
        st_at_set = {t: i for i, t in enumerate(st_at)}
        if st_at:
            states = torch.zeros(B, len(st_at), N, device=dev, dtype=torch.float32)
        se = rec.get("states_every")
        if se:
            states_every = torch.zeros(B, se[1] - se[0], N, device=dev, dtype=torch.float32)
        nc = rec.get("neuron_counts")
        if nc:
            ncount = torch.zeros(B, N, device=dev, dtype=torch.int64)
        raster = torch.zeros(T * n, B, N, device=dev, dtype=torch.bool) if rec.get("raster") else None
        tr = {"u": [], "I1": [], "I2": [], "I3": [], "syn": []} if trace else None
        if open_loop is not None:
            open_loop = torch.as_tensor(open_loop, device=dev)
        if own_spikes is not None:
            own_spikes = torch.as_tensor(own_spikes, device=dev).bool()
            flips = torch.zeros((), device=dev, dtype=torch.int64)
        zero_in = torch.zeros(B, N, device=dev, dtype=dt)
        k = 0
        for t in range(T):
            step_spk = torch.zeros(B, N, device=dev, dtype=dt) if (win or nc) else None
            if t < T_in:
                in_push = x[:, t, :].to(dt) @ self.Win
            else:
                in_push = zero_in
            for j in range(n):
                blocked = ref > 0
                if ours:
                    # 1. synapses
                    if self.static:
                        pE = pI = None
                        for (dl, cls), W in self.groups.items():
                            if k - dl < 0:
                                continue
                            p = hist[(k - dl) % self.L] @ W
                            if cls == 0:
                                pE = p if pE is None else pE + p
                            else:
                                pI = p if pI is None else pI + p
                        pE = zero_in if pE is None else pE
                        pI = zero_in if pI is None else pI
                    else:
                        R = 1.0 - (1.0 - R) * self.eD
                        q = q * self.eF
                        a = self._arrivals(hist, k, B)
                        q = q + a * self.U * (1.0 - q)
                        push = a * self.wg * q * R
                        R = R - a * q * R
                        pE = push[:, :self.nE] @ self.M_E if self.nE else zero_in
                        pI = push[:, self.nE:] @ self.M_I if self.S > self.nE else zero_in
                    ip = in_push if j == 0 else zero_in
                    # 2. currents
                    if fine:
                        IE = self.aE * IE + (1 - self.aE) * pE
                        II = self.aI * II + (1 - self.aI) * pI
                        Iin = self.in_a * Iin + (1 - self.in_a) * ip
                        total = IE + II + Iin
                    else:
                        total = pE + pI + ip
                    # 3. integrate
                    u_new = self.beta * u + total + self.tap
                    if liq.clamp:
                        u_new = torch.where(blocked, u, u_new)
                    # 4. threshold and reset
                    if liq.snn_quirk:       # snnTorch: the spike test sees u_new - [u_prev > 1]
                        crossed = (u_new - (u > 1.0).to(dt)) > 1.0
                    else:
                        crossed = (u_new > 1.0) if liq.strict else (u_new >= 1.0)
                    spk = crossed & ~blocked
                    if own_spikes is not None:
                        flips += (spk != own_spikes[k]).sum()
                        spk = own_spikes[k]
                    if liq.reset_mode == "subtract":
                        u = u_new - spk.to(dt)
                    else:
                        u = torch.where(spk, torch.full_like(u_new, liq.reset_value), u_new)
                else:
                    # Brian2 schedule of (d)'s network (its sorted objects): synapse state update (exact), the
                    # summed current from the updated y, exact neuron update with that current held constant over
                    # the step and the input current decaying exactly; threshold; arrivals (on_pre); reset.
                    q = q * self.eF
                    R, y = R * self.eD + (1 - self.eD) + y * self.cxy, y * self.ey
                    # scatter-add per target, as Brian2's summed variable does (a runaway synapse of (d) as written
                    # then only affects its own target, instead of spreading NaN through a matrix product)
                    Jsyn = torch.zeros_like(zero_in).index_add_(1, self.post_idx, y * self.wg) if self.S else zero_in
                    u_new = self.beta * u + (1 - self.beta) * (self.J_b + Jsyn) + self.kappa * Iin
                    u_new = torch.where(blocked, u, u_new)
                    Iin = Iin * self.in_a
                    spk = (u_new > 1.0) & ~blocked
                    if own_spikes is not None:
                        flips += (spk != own_spikes[k]).sum()
                        spk = own_spikes[k]
                    if j == 0 and t < T_in:
                        Iin = Iin + in_push
                    hist[k % self.L] = (open_loop[k] if open_loop is not None else spk).to(dt)
                    a = self._arrivals(hist, k, B)
                    q = q + a * self.U * (1.0 - q)
                    y = y + a * q * R
                    R = R - a * q * R
                    u = torch.where(spk, torch.full_like(u_new, liq.reset_value), u_new)
                ref = torch.where(spk, self.gap - 1, torch.clamp(ref - 1, min=0))
                if ours:
                    hist[k % self.L] = (open_loop[k] if open_loop is not None else spk).to(dt)
                spf = spk.to(dt)
                z = z * self.z_decay + spf
                if step_spk is not None:
                    step_spk += spf
                if raster is not None:
                    raster[k] = spk
                if trace:
                    tr["u"].append(u.detach().cpu().numpy().copy())
                    if ours:
                        tr["I1"].append((IE if fine else pE).cpu().numpy().copy())
                        tr["I2"].append((II if fine else pI).cpu().numpy().copy())
                        tr["I3"].append((Iin if fine else ip).cpu().numpy().copy())
                    else:
                        tr["I1"].append(Jsyn.cpu().numpy().copy()); tr["I3"].append(Iin.cpu().numpy().copy())
                k += 1
            # end of input step t
            if win and t < wl * nw:
                counts[:, t // wl] += step_spk.to(torch.int32)
            if nc and nc[0] <= t < nc[1]:
                ncount += step_spk.to(torch.int64)
            if t in st_at_set:
                states[:, st_at_set[t]] = z.float()
            if se and se[0] <= t < se[1]:
                states_every[:, t - se[0]] = z.float()
            if trace and (not self.static or not ours):
                syn = {"R": R.cpu().numpy().copy(), "u": q.cpu().numpy().copy()}
                if not ours:
                    syn["y"] = y.cpu().numpy().copy()
                tr["syn"].append(syn)
        if win:
            out["windows"] = counts
        if st_at:
            out["states_at"] = states
            out["states_at_times"] = st_at
        if se:
            out["states_every"] = states_every
        if nc:
            out["neuron_counts"] = ncount
        if raster is not None:
            out["raster"] = raster
        if trace:
            out["trace"] = tr
        out["u_end"] = u
        if own_spikes is not None:
            out["flips"] = int(flips)
        return out


# ---------------------------------------------------------------- starting states (CPU, from (seed, beat index))
def start_voltages(liq, seed, beat_ids, stream, random_start=None):
    """Starting voltages for each beat, drawn on the CPU from (seed, stream, beat index) so they do not depend
    on batching or the card. random_start=None: the liquid's own rule (0 in X; uniform 0.9..1 with the tap).
    random_start=True: uniform between the resting level and the threshold (level 3's two starting states)."""
    lo, hi = (liq.rest, 1.0) if random_start else (liq.start_lo, liq.start_hi)
    out = np.empty((len(beat_ids), liq.N))
    for i, b in enumerate(beat_ids):
        if hi > lo:
            out[i] = np.random.default_rng([int(seed), int(stream), int(b)]).uniform(lo, hi, liq.N)
        else:
            out[i] = lo
    return out
'''
LSM2_FILES['lsm2/measure.py'] = r'''"""Label-free measurements: batching beats through the simulator, firing, the rank scores, the starting-state
rule, twin runs (level 3) and the memory test (level 4)."""
import math
import numpy as np
import torch

from . import config as C
from .sim import Simulator, start_voltages


class Ctx:
    """Where and how to simulate: device, precision and a fixed batch size (plan, determinism)."""
    def __init__(self, device="cpu", dtype=torch.float32, batch=C.CODE["sim_batch"], log=print):
        self.device, self.dtype, self.batch, self.log = device, dtype, batch, log


def simulate(ctx, liq, g, in_scale, spikes, u0, T, rec, recurrence=True):
    """spikes: (n, T_in, C) numpy input spikes; u0: (n, N). Returns the simulator outputs as numpy, batched."""
    sm = Simulator(liq, device=ctx.device, dtype=ctx.dtype, g=g, in_scale=in_scale, recurrence=recurrence)
    parts = []
    for s in range(0, len(spikes), ctx.batch):
        o = sm.run(spikes[s:s + ctx.batch], u0[s:s + ctx.batch], T, rec=rec)
        parts.append({k: v.cpu().numpy() if torch.is_tensor(v) else v for k, v in o.items() if k != "u_end"})
    out = {}
    for k in parts[0]:
        if k == "states_at_times":
            out[k] = parts[0][k]
        elif k == "raster":
            out[k] = np.concatenate([p[k] for p in parts], 1)
        else:
            out[k] = np.concatenate([p[k] for p in parts], 0)
    return out


# ---------------------------------------------------------------- rank scores
def effective_rank(M):
    """Roy & Vetterli 2007: exp of the entropy of the normalised singular values, of the centred matrix
    (float64). 0 for an all-zero matrix."""
    M = np.asarray(M, np.float64)
    M = M - M.mean(0, keepdims=True)
    s = np.linalg.svd(M, compute_uv=False)
    tot = s.sum()
    if tot <= 0:
        return 0.0
    p = s / tot
    p = p[p > 0]
    return float(np.exp(-(p * np.log(p)).sum()))


def plain_rank(M):
    """(e)'s plain rank (reported only)."""
    M = np.asarray(M, np.float64)
    return int(np.linalg.matrix_rank(M)) if M.any() else 0


def cutoff_rank(M, frac=0.01):
    """An earlier draft's measure: singular values above 1% of the largest, uncentred (reported only)."""
    s = np.linalg.svd(np.asarray(M, np.float64), compute_uv=False)
    return int((s > frac * s[0]).sum()) if len(s) and s[0] > 0 else 0


# ---------------------------------------------------------------- input sets
def jitter_spikes(spikes, rng, sd_ms=C.JITTER_MS, step_ms=C.SAMPLE_MS):
    """Each encoded spike moved by a Gaussian amount with SD 4 ms, rounded to the nearest sample; spikes moved
    outside the beat are dropped (plan, Generalisation score). spikes: (T, C) counts."""
    T, Cn = spikes.shape
    out = np.zeros((T, Cn), np.uint8)
    tt, cc = np.nonzero(spikes)
    reps = spikes[tt, cc].astype(int)
    tt, cc = np.repeat(tt, reps), np.repeat(cc, reps)
    nt = tt + np.rint(rng.normal(0, sd_ms / step_ms, len(tt))).astype(int)
    ok = (nt >= 0) & (nt < T)
    np.add.at(out, (nt[ok], cc[ok]), 1)
    return out


def twin_spike(spikes, rng, t=C.R_SAMPLE):
    """The same input with one extra spike at sample t (the R peak), on a channel silent there."""
    s = spikes.copy()
    free = np.nonzero(s[t] == 0)[0]
    ch = rng.choice(free) if len(free) else rng.integers(s.shape[1])
    s[t, ch] = min(255, int(s[t, ch]) + 1)
    return s


def level3_sets(sep_spikes, gen_base, seed_sets=0, n_twin=None):
    """sep_spikes: (500, T, C) spikes of 500 different inputs; gen_base: (4, T, C) the 4 inputs that are jittered.
    Returns the separation, generalisation and twin input sets (fixed for every candidate and g)."""
    n_twin = C.CODE["twin_beats"] if n_twin is None else n_twin
    gen = np.stack([jitter_spikes(gen_base[i], np.random.default_rng([seed_sets, 31, i, c]))
                    for i in range(len(gen_base)) for c in range(C.GEN_COPIES)])
    twin = np.stack([twin_spike(sep_spikes[i], np.random.default_rng([seed_sets, 37, i])) for i in range(min(n_twin, len(sep_spikes)))])
    return {"sep": sep_spikes, "gen": gen, "twin": twin}


# ---------------------------------------------------------------- level 3 at one g
def firing_stats(neuron_counts, n_samples, step_ms=C.SAMPLE_MS):
    """Mean rate over all neurons (silent ones included) and the share of neurons above 100 Hz.
    neuron_counts: (beats, N) spike counts over n_samples input steps."""
    dur_s = n_samples * step_ms / 1000.0
    per_neuron = neuron_counts.mean(0) / dur_s
    return float(per_neuron.mean()), float((per_neuron > C.HIGH_HZ).mean())


def measure_at_g(ctx, liq, g, in_scale, sets, seed, T=C.BEAT_SAMPLES, read=C.STATE_READ_SAMPLE,
                 with_start_test=False, with_twins=True, extra_ranks=False):
    """Every level-3 measure at one g for one liquid. Starting voltages: uniform between the resting level and
    the threshold for every candidate, drawn from (seed, stream, input index)."""
    nb = len(sets["sep"])
    u_sep = start_voltages(liq, seed, range(nb), 1, random_start=True)
    rec = {"states_at": [read, C.START_TEST_SAMPLE], "neuron_counts": (0, T)}
    o = simulate(ctx, liq, g, in_scale, sets["sep"], u_sep, T, rec)
    rate, high = firing_stats(o["neuron_counts"], T, liq.h_ms * liq.n_sub)
    sep_states = o["states_at"][:, 0]
    u_gen = start_voltages(liq, seed, range(len(sets["gen"])), 2, random_start=True)
    og = simulate(ctx, liq, g, in_scale, sets["gen"], u_gen, T, {"states_at": [read]})
    gen_states = og["states_at"][:, 0]
    res = {"g": float(g), "abs_g": liq.ee_mean_push(g), "rate_hz": rate, "frac_above_100hz": high,
           "in_band": bool(C.BAND_HZ[0] <= rate <= C.BAND_HZ[1] and high <= C.HIGH_FRAC),
           "sep": effective_rank(sep_states), "gen": effective_rank(gen_states)}
    res["gap"] = res["sep"] - res["gen"]
    if extra_ranks:
        res.update({"sep_plain": plain_rank(sep_states), "gen_plain": plain_rank(gen_states),
                    "sep_cut1": cutoff_rank(sep_states), "gen_cut1": cutoff_rank(gen_states)})
        res["gap_plain"], res["gap_cut1"] = res["sep_plain"] - res["gen_plain"], res["sep_cut1"] - res["gen_cut1"]
    if with_twins and len(sets["twin"]):
        nt = len(sets["twin"])
        u_t = u_sep[:nt]
        rt = {"states_every": (0, T)}
        a = simulate(ctx, liq, g, in_scale, sets["sep"][:nt], u_t, T, rt)["states_every"]
        b = simulate(ctx, liq, g, in_scale, sets["twin"], u_t, T, rt)["states_every"]
        dist = np.sqrt(((a - b) ** 2).sum(-1)).mean(0)          # mean over beats of the state distance, per sample
        e0, e1 = C.CODE["twin_early"]; l0, l1 = C.CODE["twin_late"]
        early, late = dist[e0:e1 + 1].mean(), dist[l0:l1 + 1].mean()
        res["twin_early"], res["twin_late"] = float(early), float(late)
        res["twin_ratio"] = float(late / early) if early > 0 else float("nan")
    if with_start_test:
        u2 = start_voltages(liq, seed, range(nb), 3, random_start=True)
        o2 = simulate(ctx, liq, g, in_scale, sets["sep"], u2, T, {"states_at": [C.START_TEST_SAMPLE]})
        s1, s2 = o["states_at"][:, 1].astype(np.float64), o2["states_at"][:, 0].astype(np.float64)
        same = float(np.sqrt(((s1 - s2) ** 2).sum(1)).mean())
        sq = (s1 ** 2).sum(1)
        D2 = np.maximum(sq[:, None] + sq[None, :] - 2 * s1 @ s1.T, 0)
        iu = np.triu_indices(len(s1), 1)
        diff = float(np.sqrt(D2[iu]).mean())
        res.update({"start_same": same, "start_diff": diff,
                    "start_rule_pass": bool(same < diff or (same == 0 and diff == 0))})
    return res


# ---------------------------------------------------------------- level 4: memory
def surrogate_walks(steps_pool, n_seq, T, seed):
    """Random walks on [0, 1], reflected at the edges, step sizes drawn from ECG's own sample-to-sample steps
    with random signs (plan, level 4). Starting levels uniform on [0, 1]."""
    rng = np.random.default_rng([seed, 41])
    X = np.empty((n_seq, T))
    for i in range(n_seq):
        st = rng.choice(steps_pool, T - 1) * rng.choice([-1.0, 1.0], T - 1)
        v = rng.uniform(0, 1)
        X[i, 0] = v
        for t in range(1, T):
            v = v + st[t - 1]
            while v < 0 or v > 1:
                v = -v if v < 0 else 2 - v
            X[i, t] = v
    return X


class _Stats:
    """Sufficient statistics of (x, y) rows for ridge with an intercept: n, sum x, sum y, sum y^2, X'X, X'y."""
    def __init__(self, N, dev):
        z = lambda *sh: torch.zeros(*sh, dtype=torch.float64, device=dev)
        self.n, self.sx, self.sy, self.syy, self.G, self.b = 0, z(N), z(()), z(()), z(N, N), z(N)

    def add(self, X, y):
        self.n += X.shape[0]; self.sx += X.sum(0); self.sy += y.sum(); self.syy += (y * y).sum()
        self.G += X.T @ X; self.b += X.T @ y

    def __add__(self, o):
        r = _Stats.__new__(_Stats)
        r.n = self.n + o.n
        for k in ("sx", "sy", "syy", "G", "b"):
            setattr(r, k, getattr(self, k) + getattr(o, k))
        return r


def _fit(st, alpha):
    mu, my = st.sx / st.n, st.sy / st.n
    Gc = st.G - st.n * torch.outer(mu, mu)
    bc = st.b - st.n * mu * my
    w = torch.linalg.solve(Gc + alpha * torch.eye(len(mu), dtype=Gc.dtype, device=Gc.device), bc)
    return w, mu, my


def _ss_res(st, w, mu, my):
    """sum over st's rows of (y - my - (x - mu)'w)^2, from its statistics."""
    c = my - mu @ w                                     # prediction = x'w + c
    return float(st.syy - 2 * (w @ st.b) - 2 * c * st.sy + w @ st.G @ w + 2 * c * (w @ st.sx) + st.n * c * c)


def _ss_tot(st, mean=None):
    m = st.sy / st.n if mean is None else mean
    return float(st.syy - 2 * m * st.sy + st.n * m * m)


def memory_curve(ctx, liq, g, in_scale, seq_spikes, target_counts, seed, T=C.BEAT_SAMPLES,
                 lags_ms=C.MEMORY_T_MS, warmup=C.MEMORY_WARMUP, recurrence=True, batch=100,
                 alphas=C.CODE["ridge_alphas"], k=C.CODE["ridge_inner_folds"]):
    """R^2 of reconstructing target[t - T] from the 30 ms-filtered state at t, for each lag: a ridge readout
    trained on the first half of the sequences (penalty by inner CV, folds grouped by sequence) and scored on
    the second half, at every sample after the warm-up. Statistics are accumulated batch by batch."""
    n = len(seq_spikes)
    step_ms = liq.h_ms * liq.n_sub
    half = n // 2
    fold = np.full(n, k)                                      # group k = test sequences
    fold[:half] = np.random.default_rng(0).permutation(half) % k
    lagL = {lag: int(round(lag / step_ms)) for lag in lags_ms}
    dev = ctx.device
    stats = {lag: [_Stats(liq.N, dev) for _ in range(k + 1)] for lag in lags_ms}
    sm = Simulator(liq, device=dev, dtype=ctx.dtype, g=g, in_scale=in_scale, recurrence=recurrence)
    u0 = start_voltages(liq, seed, range(n), 4)               # the liquid's own starting rule, as on ECG
    counts = 0.0
    for s0 in range(0, n, batch):
        o = sm.run(seq_spikes[s0:s0 + batch], u0[s0:s0 + batch], T, rec={"states_every": (0, T), "neuron_counts": (0, T)})
        Z = o["states_every"].to(torch.float64)               # (b, T, N)
        counts += float(o["neuron_counts"].sum())
        tg = torch.as_tensor(np.asarray(target_counts[s0:s0 + batch]), dtype=torch.float64, device=dev)
        for lag, L in lagL.items():
            t0 = max(warmup, L)
            for bi in range(Z.shape[0]):
                X = Z[bi, t0:T]
                y = tg[bi, t0 - L:T - L]
                stats[lag][fold[s0 + bi]].add(X, y)
        del o, Z
    out = {"rate_hz": counts / (n * liq.N * T * step_ms / 1000.0)}
    for lag in lags_ms:
        st = stats[lag]
        train_all = st[0]
        for f in range(1, k):
            train_all = train_all + st[f]
        best, best_a = -np.inf, alphas[0]
        for a in alphas:
            res = tot = 0.0
            for f in range(k):
                tr = None
                for f2 in range(k):
                    if f2 != f:
                        tr = st[f2] if tr is None else tr + st[f2]
                w, mu, my = _fit(tr, a)
                res += _ss_res(st[f], w, mu, my)
                tot += _ss_tot(st[f], float(train_all.sy / train_all.n))
            r2 = 1 - res / tot if tot > 0 else float("nan")
            if r2 > best:
                best, best_a = r2, a
        w, mu, my = _fit(train_all, best_a)
        tot = _ss_tot(st[k])
        out[f"r2_{lag}"] = float(1 - _ss_res(st[k], w, mu, my) / tot) if tot > 0 else float("nan")
        out[f"alpha_{lag}"] = float(best_a)
    return out
'''
LSM2_FILES['lsm2/gain.py'] = r'''"""Choosing the recurrent gain g (plan, Choosing the recurrent gain; decision 5).

Grid: g = 0 (control), 0.1 and 0.3, a coarse pass at 1, 3, ..., 1000 times the base weights, extensions in 3x
steps at an edge (at most 3), then a fine pass g* x 1.4^k, k = -3..3. Each setting is measured on all seeds and
averaged. Rule: among settings above 0 inside the firing band, the largest mean gap; a setting whose mean gap lies
within the range of the best setting's seed values is tied with it, and the smallest tied g is taken."""
import numpy as np

from . import config as C


def _agg(per_seed):
    keys = [k for k in per_seed[0] if isinstance(per_seed[0][k], (int, float, bool, np.floating, np.integer))]
    m = {k: float(np.nanmean([float(r[k]) for r in per_seed])) for k in keys}
    m["gap_seeds"] = [float(r["gap"]) for r in per_seed]
    m["rate_seeds"] = [float(r["rate_hz"]) for r in per_seed]
    m["in_band"] = bool(C.BAND_HZ[0] <= m["rate_hz"] <= C.BAND_HZ[1] and m["frac_above_100hz"] <= C.HIGH_FRAC)
    m["per_seed"] = per_seed
    return m


def choose(table):
    """table: {g: aggregated measures}. Returns (chosen g or None, best g or None)."""
    cand = [g for g in table if g > 0 and table[g]["in_band"]]
    if not cand:
        return None, None
    best = max(cand, key=lambda g: (table[g]["gap"], -g))
    lo, hi = min(table[best]["gap_seeds"]), max(table[best]["gap_seeds"])
    eps = 1e-12 * max(1.0, abs(lo), abs(hi))                 # the mean of equal seed values may round
    tied = [g for g in cand if lo - eps <= table[g]["gap"] <= hi + eps] + [best]
    return min(tied), best


def edge_status(table, g):
    """'upper' / 'lower' if g sits at that edge of the measured grid (g = 0 excluded), else None. A setting next
    to one outside the firing band counts as inside the grid (plan, Grid)."""
    gs = sorted(x for x in table if x > 0)
    i = gs.index(g)
    if i == len(gs) - 1:
        return "upper"
    if i == 0:
        return "lower"
    return None


def band_limited(table, g):
    gs = sorted(x for x in table if x > 0)
    i = gs.index(g)
    up = i + 1 < len(gs) and not table[gs[i + 1]]["in_band"]
    down = i > 0 and not table[gs[i - 1]]["in_band"]
    return bool(up or down)


def sweep(measure, seeds, log=print, coarse=None):
    """measure(g, seed) -> level-3 measures at g (measure.measure_at_g). Returns the sweep record."""
    table, order = {}, []

    def run(gs, stage):
        for g in gs:
            g = float(g)
            if g in table:
                continue
            table[g] = _agg([measure(g, s) for s in seeds])
            table[g]["stage"] = stage
            order.append(g)
            t = table[g]
            log(f"    g={g:<10.4g} abs={t['abs_g']:.4g} rate={t['rate_hz']:.2f} Hz in_band={t['in_band']} "
                f"sep={t['sep']:.2f} gen={t['gen']:.2f} gap={t['gap']:.2f}"
                + (f" twin={t['twin_ratio']:.2f}" if 'twin_ratio' in t else ""))

    run([0.0] + list(C.G_LOW) + list(coarse or C.G_COARSE), "coarse")
    chosen, best = choose(table)
    ext = 0
    while best is not None and ext < C.G_EXTEND_MAX:
        e = edge_status(table, best)
        if e is None:
            break
        gs = sorted(x for x in table if x > 0)
        nxt = gs[-1] * C.G_EXTEND_STEP if e == "upper" else gs[0] / C.G_EXTEND_STEP
        run([nxt], f"extension {ext + 1}")
        ext += 1
        chosen, best = choose(table)
    at_edge = best is not None and edge_status(table, best) is not None
    if best is not None and not at_edge:
        run([best * C.G_FINE_STEP ** k for k in range(-C.G_FINE_K, C.G_FINE_K + 1) if k != 0], "fine")
        chosen, best = choose(table)
        at_edge = edge_status(table, best) is not None
    rec = {"table": table, "order": order, "chosen_g": chosen, "best_g": best, "extensions": ext,
           "no_g_in_band": best is None, "peak_at_edge": bool(at_edge),
           "band_limited": bool(chosen is not None and band_limited(table, chosen))}
    if chosen is not None:
        gs = sorted(x for x in table if x > 0)
        i = gs.index(chosen)
        rec["next_higher_g"] = gs[i + 1] if i + 1 < len(gs) else None
        rec["next_higher_gap_lower"] = (rec["next_higher_g"] is not None
                                        and table[rec["next_higher_g"]]["gap"] < table[chosen]["gap"])
    rec["gap_peak_inside_grid"] = bool(best is not None and not at_edge)
    return rec
'''
LSM2_FILES['lsm2/calib.py'] = r'''"""Input size (plan, Input and calibration): the paper's rule, or with factor 5 a bisection on one factor s so the
input-receiving neurons fire, with recurrence off, at the rate Maass's own input-receiving neurons reach in the
Maass check (paper version). Factor 9 fixes Maass's own per-spike size instead."""
import math
import numpy as np

from . import config as C
from .liquid import paper_rule_scale
from .measure import simulate, firing_stats
from .sim import start_voltages


def ain_of(spikes):
    """Mean number of input spikes per sample, summed over channels (the authors' A_in)."""
    return float(np.asarray(spikes, np.float64).sum() / (spikes.shape[0] * spikes.shape[1]))


def input_neuron_rate(ctx, liqs, s, spikes, T=C.BEAT_SAMPLES):
    """Mean rate (Hz) of the input-receiving neurons, recurrence off, averaged over the liquids (seeds)."""
    rates = []
    for liq in liqs:
        u0 = start_voltages(liq, liq.meta.get("seed", 0), range(len(spikes)), 5)
        o = simulate(ctx, liq, 0.0, s, spikes, u0, T, {"neuron_counts": (0, T)}, recurrence=False)
        cnt = o["neuron_counts"][:, liq.in_neurons]
        rates.append(firing_stats(cnt, T, liq.h_ms * liq.n_sub)[0])
    return float(np.mean(rates))


def bisect_scale(ctx, liqs, spikes, target_hz, tol=C.CODE["calib_tol"], iters=C.CODE["calib_iters"], log=print):
    """Smallest bracket on s (log scale) until the rate is within tol of the target."""
    lo, hi = 0.0, 0.05
    r_hi = input_neuron_rate(ctx, liqs, hi, spikes)
    k = 0
    while r_hi < target_hz and k < 30:
        lo, hi = hi, hi * 2
        r_hi = input_neuron_rate(ctx, liqs, hi, spikes)
        k += 1
    if r_hi < target_hz:
        raise RuntimeError(f"input calibration: rate {r_hi:.2f} Hz < target {target_hz:.2f} Hz even at s = {hi}")
    best, best_r = hi, r_hi
    for i in range(iters):
        mid = math.sqrt(lo * hi) if lo > 0 else hi / 2
        r = input_neuron_rate(ctx, liqs, mid, spikes)
        if abs(r - target_hz) < abs(best_r - target_hz):
            best, best_r = mid, r
        if abs(r - target_hz) <= tol * target_hz:
            break
        if r < target_hz:
            lo = mid
        else:
            hi = mid
    log(f"    factor-5 input size s = {best:.5g} (rate {best_r:.3f} Hz, target {target_hz:.3f} Hz); Maass's own 0.12")
    return float(best), float(best_r)


def input_scale(ctx, liqs, spikes_cal, ain, maass_target_hz, log=print):
    """The candidate's input size: (scale, how it was set)."""
    liq = liqs[0]
    if liq.in_scale_fixed is not None:
        return float(liq.in_scale_fixed), {"rule": "fixed (factor 9 or Maass's own)", "scale": float(liq.in_scale_fixed)}
    if 5 in liq.factors:
        if maass_target_hz is None:
            raise RuntimeError("factor 5 needs the Maass check's input-neuron rate")
        s, r = bisect_scale(ctx, liqs, spikes_cal, maass_target_hz, log=log)
        return s, {"rule": "factor 5 calibration", "scale": s, "rate_hz": r, "target_hz": maass_target_hz}
    s = paper_rule_scale(ain, liq)
    fill_E = C.I_TARGET / (1 - float(liq.beta[liq.is_exc][0] ** liq.n_sub))
    fill_I = C.I_TARGET / (1 - float(liq.beta[~liq.is_exc][0] ** liq.n_sub))
    return s, {"rule": "paper's rule", "scale": s, "A_in": ain, "P_in": 1.0 if 6 in liq.factors else C.X["P_in"],
               "fill_E": fill_E, "fill_I": fill_I}
'''
LSM2_FILES['lsm2/data.py'] = r'''"""Label-free inputs for the search: the DS1 halves, the 500 calibration beats, the generalisation and twin sets,
the memory test's random surrogate, and Poisson trains for the Maass check."""
from dataclasses import dataclass
import numpy as np

from . import config as C
from .measure import level3_sets, surrogate_walks


def multithreshold(X, num_levels=200):
    """The authors' Multi-threshold encoder (detect_edges='both', include_endpoints=True): a spike on threshold j
    at sample t when x crosses it between t-1 and t. X: (n, T) in [0, 1] -> (n, T, K) uint8."""
    X = np.asarray(X)
    thr = np.linspace(0.0, 1.0, num_levels, dtype=X.dtype)
    above = X[:, :, None] >= thr
    out = np.zeros(above.shape, np.uint8)
    out[:, 1:] = (above[:, 1:] != above[:, :-1])
    return out


def split_records(records, seed=C.CODE["search_split_seed"]):
    """DS1's patients halved without labels: a fixed random permutation of the record list."""
    rec = sorted(records)
    perm = np.random.default_rng(seed).permutation(len(rec))
    half = len(rec) // 2
    return sorted(rec[i] for i in perm[:half]), sorted(rec[i] for i in perm[half:])


@dataclass
class SearchData:
    name: str
    sets: dict                 # sep (500, T, C), gen (500, T, C), twin (100, T, C)
    ain: float                 # input spikes per sample (A_in) on this half, labels unused
    mem_spikes: np.ndarray     # (500, T, C) surrogate encoded
    mem_targets: np.ndarray    # (500, T) input spike count per sample
    surrogate_ain: float
    real_spikes: np.ndarray = None    # secondary memory check on real beats
    real_targets: np.ndarray = None
    meta: dict = None


def make_search_data(name, beats, encode, seed, n_cal=None, n_mem=None):
    """beats: (n, 577) normalised config-11 beats of one half, in a fixed order (labels never used).
    encode: (m, 577) -> (m, 577, C) spikes."""
    n_cal = n_cal or C.N_CAL_BEATS
    n_mem = n_mem or C.CODE["memory_sequences"]
    rng = np.random.default_rng([seed, 51])
    pick = np.sort(rng.choice(len(beats), n_cal + C.GEN_BEATS, replace=False))
    perm = rng.permutation(pick)
    sep_idx, gen_idx = np.sort(perm[:n_cal]), np.sort(perm[n_cal:])
    sep = encode(beats[sep_idx])
    gen_base = encode(beats[gen_idx])
    sets = level3_sets(sep, gen_base, seed_sets=seed)
    # A_in over the whole half (the authors' quantity: input spikes per sample, summed over channels)
    tot, cnt = 0.0, 0
    for s in range(0, len(beats), 2000):
        e = encode(beats[s:s + 2000]); tot += float(e.sum(dtype=np.int64)); cnt += e.shape[0] * e.shape[1]
    ain = tot / cnt
    steps = np.abs(np.diff(beats, axis=1)).ravel()
    walks = surrogate_walks(steps, n_mem, beats.shape[1], seed)
    mem = encode(walks)
    targets = mem.sum(2).astype(np.int64)
    s_ain = float(mem.sum(dtype=np.int64) / (mem.shape[0] * mem.shape[1]))
    # secondary check: real beats, target = deviation from the R-aligned mean beat
    mean_beat = beats.mean(0)
    real_targets = beats[sep_idx] - mean_beat[None, :]
    return SearchData(name=name, sets=sets, ain=ain, mem_spikes=mem, mem_targets=targets, surrogate_ain=s_ain,
                      real_spikes=sep, real_targets=real_targets,
                      meta={"n_beats_half": int(len(beats)), "sep_idx": sep_idx.tolist(), "gen_idx": gen_idx.tolist(),
                            "surrogate_vs_ecg": s_ain / ain})


def poisson_trains(n, T, rate_hz, seed, stream, step_ms=C.SAMPLE_MS):
    """n Poisson trains of T samples, counts per sample (one channel)."""
    rng = np.random.default_rng([seed, stream])
    return rng.poisson(rate_hz * step_ms / 1000.0, (n, T, 1)).astype(np.uint8)


def make_maass_data(seed, T=C.BEAT_SAMPLES, n_cal=None, n_mem=None):
    """The Maass check's inputs: 20 Hz Poisson trains of beat length stand in for beats."""
    n_cal = n_cal or C.N_CAL_BEATS
    n_mem = n_mem or C.CODE["memory_sequences"]
    r = C.MAASS["input_rate_hz"]
    sep = poisson_trains(n_cal, T, r, seed, 61)
    gen_base = poisson_trains(C.GEN_BEATS, T, r, seed, 62)
    sets = level3_sets(sep, gen_base, seed_sets=seed)
    mem = poisson_trains(n_mem, T, r, seed, 63)
    return SearchData(name="maass_poisson", sets=sets, ain=float(sep.mean() * 1), mem_spikes=mem,
                      mem_targets=mem[:, :, 0].astype(np.int64), surrogate_ain=float(mem.mean()), meta={"rate_hz": r})
'''
LSM2_FILES['lsm2/brian2_ref.py'] = r'''"""Level 1b and 1e references in Brian2.

brian2_ours: Brian2 runs our per-sub-step update as explicit code, in the same order of operations (level 1b).
  Brian2 step k+1 computes our sub-step k (the same one-step shift the replication's Brian2 check used):
    'start'      neurons: currents, integrate, threshold, reset (our steps 2-4), using the pushes that arrived
    'groups'     synapses: recover / fade over one sub-step (our step 1, first half), for our next sub-step
    'thresholds' our sub-step's spikes are emitted (closed loop) or replayed from a recording (open loop)
    'synapses'   arrivals for our next sub-step: dynamic synapses update u, push, R (our step 1, second half)
  A delay of d sub-steps is a Brian2 delay of d - 1 steps; input spikes of sample t are delivered in Brian2
  step t*n so they act in our sub-step t*n.
"""
import numpy as np


def _b2():
    import brian2 as b2
    b2.BrianLogger.log_level_error()
    try:
        b2.prefs.codegen.target = "cython"
        b2.start_scope()
        b2.NeuronGroup(1, "v:1").v = 0        # probe that cython compiles here
    except Exception:
        b2.prefs.codegen.target = "numpy"
    return b2


def brian2_ours(liq, x, u0, T, g=1.0, in_scale=1.0, open_loop=None, target="numpy"):
    """One beat. x: (T_in, C) input spikes per input step; u0: (N,); open_loop: (T*n, N) presynaptic spikes
    (None = closed loop). Returns traces: u, I1, I2, I3 (T*n, N), synapse states R, u at the end of each input
    step (T, S), and the spike raster (T*n, N)."""
    import brian2 as b2
    b2.BrianLogger.log_level_error()
    b2.prefs.codegen.target = target
    b2.start_scope()
    n, N, h = liq.n_sub, liq.N, liq.h_ms * 1e-3
    b2.defaultclock.dt = h * b2.second
    fine = liq.cur_a is not None
    dyn = liq.syn_model == "markram"
    eqs = """
    u : 1
    ref : 1
    IE : 1
    II : 1
    Iin : 1
    accE : 1
    accI : 1
    accIn : 1
    lastE : 1
    lastI : 1
    lastIn : 1
    spiked : 1
    beta : 1 (constant)
    tap : 1 (constant)
    gap : 1 (constant)
    a_in : 1 (constant)
    """
    G = b2.NeuronGroup(N, eqs, threshold="spiked > 0.5", reset="", method="euler", name="liquid")
    G.u = u0
    G.beta, G.tap, G.gap = liq.beta, liq.tap, liq.gap.astype(float)
    G.a_in = liq.in_a if liq.in_a is not None else 0.0
    aE = float(liq.cur_a[0]) if fine else 0.0
    aI = float(liq.cur_a[1]) if fine else 0.0
    reset = ("u_new - spk" if liq.reset_mode == "subtract" else f"spk*{liq.reset_value!r} + (1 - spk)*u_new")
    thr = "u_new - int(u > 1.0) > 1.0" if liq.snn_quirk else ("u_new > 1.0" if liq.strict else "u_new >= 1.0")
    code = f"""
    active = int(t > 0.5*dt)
    blocked = int(ref > 0.5)
    IE = active*({aE!r}*IE + (1 - {aE!r})*accE) + (1 - active)*IE
    II = active*({aI!r}*II + (1 - {aI!r})*accI) + (1 - active)*II
    Iin = active*(a_in*Iin + (1 - a_in)*accIn) + (1 - active)*Iin
    total = {'IE + II + Iin' if fine else 'accE + accI + accIn'}
    u_new = beta*u + total + tap
    {'u_new = blocked*u + (1 - blocked)*u_new' if liq.clamp else ''}
    spk = int({thr})*(1 - blocked)*active
    u_next = {reset}
    u = active*u_next + (1 - active)*u
    ref = active*(spk*(gap - 1) + (1 - spk)*clip(ref - 1, 0, 1e9)) + (1 - active)*ref
    lastE = {'IE' if fine else 'accE'}
    lastI = {'II' if fine else 'accI'}
    lastIn = {'Iin' if fine else 'accIn'}
    accE = 0
    accI = 0
    accIn = 0
    spiked = spk
    """
    G.run_regularly(code, when="start", name="update")
    objs = [G]
    # input: channel spikes of sample t delivered at Brian2 step t*n
    xs = np.asarray(x)
    tt, cc = np.nonzero(xs[:min(len(xs), T)])
    Win = liq.Win * in_scale
    if len(tt):
        inp = b2.SpikeGeneratorGroup(xs.shape[1], cc, (tt * n) * h * b2.second, name="inp")
        ci, nj = np.nonzero(Win)
        Sin = b2.Synapses(inp, G, "w : 1 (constant)", on_pre="accIn_post += w", name="sin")
        Sin.connect(i=ci, j=nj); Sin.w = Win[ci, nj]
        objs += [inp, Sin]
    # presynaptic source
    if open_loop is not None:
        kk, ii = np.nonzero(np.asarray(open_loop))
        src = b2.SpikeGeneratorGroup(N, ii, (kk + 1) * h * b2.second, name="src")
        objs.append(src)
    else:
        src = G
    syn_mons = []
    if liq.S:
        for ti in range(4):
            m = np.nonzero(liq.typ == ti)[0]
            if not len(m):
                continue
            acc = "accE_post" if ti < 2 else "accI_post"
            d = int(liq.delay[ti])
            if dyn:
                model = """
                wg : 1 (constant)
                U : 1 (constant)
                eD : 1 (constant)
                eF : 1 (constant)
                R : 1
                q : 1
                """
                on_pre = f"""
                q = q + U*(1 - q)
                {acc} += wg*q*R
                R = R - q*R
                """
            else:
                model, on_pre = "wg : 1 (constant)", f"{acc} += wg"
            S = b2.Synapses(src, G, model, on_pre=on_pre, delay=(d - 1) * h * b2.second, name=f"syn{ti}")
            S.connect(i=liq.pre[m], j=liq.post[m])
            if dyn:
                S.wg = liq.w[m] * g / liq.Ubar[ti]
                S.U, S.eD, S.eF = liq.U[m], np.exp(-liq.h_ms / liq.D_ms[m]), np.exp(-liq.h_ms / liq.F_ms[m])
                S.R, S.q = 1.0, 0.0
                S.run_regularly("R = 1 - (1 - R)*eD\nq = q*eF", when="groups", name=f"decay{ti}")
                mon = b2.StateMonitor(S, ["R", "q"], record=True, when="end", name=f"smon{ti}")
                syn_mons.append((m, mon))
                objs.append(mon)
            else:
                S.wg = liq.w[m] * g
            objs.append(S)
    M = b2.StateMonitor(G, ["u", "lastE", "lastI", "lastIn"], record=True, when="end", name="mon")
    spm = b2.SpikeMonitor(G, name="spm")
    objs += [M, spm]
    net = b2.Network(*objs)
    net.run((T * n + 1) * h * b2.second)
    out = {k: np.asarray(getattr(M, v))[:, 1:].T for k, v in (("u", "u"), ("I1", "lastE"), ("I2", "lastI"), ("I3", "lastIn"))}
    r = np.zeros((T * n, N), bool)
    k = np.round(np.asarray(spm.t / b2.second) / h).astype(int) - 1
    ok = (k >= 0) & (k < T * n)
    r[k[ok], np.asarray(spm.i)[ok]] = True
    out["raster"] = r
    if dyn and syn_mons:
        R = np.zeros((T, liq.S)); q = np.zeros((T, liq.S))
        last = np.arange(T) * n + n - 1          # Brian2 step k ('end') holds our sub-step k's synapse state
        for m, mon in syn_mons:
            R[:, m] = np.asarray(mon.R).T[last]
            q[:, m] = np.asarray(mon.q).T[last]
        out["R"], out["q"] = R, q
    return out


# ---------------------------------------------------------------- level 1e: (d)'s script, reused unchanged
D_REPLACEMENT_LINES = {
    "synapses.U[synapses.U < 0] = U": "synapses.U[np.nonzero(np.asarray(synapses.U) < 0)[0]] = U",
    "synapses.D[synapses.D / ms <= 0] = D": "synapses.D[np.nonzero(np.asarray(synapses.D / ms) <= 0)[0]] = D",
    "synapses.F[synapses.F / ms <= 0] = F": "synapses.F[np.nonzero(np.asarray(synapses.F / ms) <= 0)[0]] = F",
}


def script_module(path, fix_replacement=True):
    """Load (d)'s maass2002.py (its main block is guarded), so its own functions build and run the liquid.

    (d) means to replace bad U, D, F draws by the mean, but Brian2 reads a boolean array index as the integer
    indices 0 and 1: as written, only synapses 0 and 1 of each group get the mean and every negative draw stays
    (about 2.3% of synapses each for U, D and F; a negative D or F makes x or u grow without bound).
    fix_replacement=True patches those three lines so they do what (d) intends (and what the plan describes)."""
    import types
    src = open(path).read()
    if fix_replacement:
        for a, b in D_REPLACEMENT_LINES.items():
            if a not in src:
                raise RuntimeError(f"(d)'s script changed: line not found: {a}")
            src = src.replace(a, b)
    mod = types.ModuleType("maass2002_d")
    mod.__file__ = path
    exec(compile(src, path, "exec"), mod.__dict__)
    return mod
'''
LSM2_FILES['lsm2/stats.py'] = r'''"""Statistics and decision rules (plan, Statistics and decision rules): the paired Nadeau-Bengio corrected
t-test over the 12 CV splits (variance factor 1/12 + 0.336), Holm within a family, the DS2 difference paired by
seed, and the verdicts meaningful / comparable / inconclusive. Inputs are tables of runs with columns spec_split,
spec_seed, macro_f1 (and DS2 metrics)."""
import math
import numpy as np
from scipy import stats as sstats

NB_RATIO = 0.336          # plan: test/train ratio of the 3x4 patient-grouped CV (the replication used 0.25/0.75)


def per_split_seed(d):
    """Macro-F1 by (split, seed)."""
    return d.groupby(["spec_split", "spec_seed"], dropna=False).macro_f1.mean()

def compare(a, b, require_seed_sign=False):
    """A vs B (Macro-F1): seeds averaged per CV split; paired Nadeau-Bengio corrected t-test over the 12 splits
    with its 95% interval; the DS2 difference paired by seed."""
    if a is None or b is None or a.empty or b.empty:
        return None
    A, B = per_split_seed(a), per_split_seed(b)
    if a["spec_seed"].isna().all() or b["spec_seed"].isna().all():      # a seed-free control pairs with every seed
        A = A.groupby(level=0).mean(); B = B.groupby(level=0).mean()
        Asp, Bsp = A, B
        ds2_seed = None
    else:
        Asp, Bsp = A.groupby(level=0).mean(), B.groupby(level=0).mean()
        ds2_seed = (A.xs("final", level=0) - B.xs("final", level=0)).dropna() if "final" in A.index.get_level_values(0) and "final" in B.index.get_level_values(0) else None
    cv = [s for s in Asp.index if str(s).startswith("cv-") and s in Bsp.index]
    if len(cv) < 3:
        return None
    d = (Asp[cv] - Bsp[cv]).values
    k = len(d)
    var = np.var(d, ddof=1) * (1 / k + NB_RATIO)
    se = math.sqrt(var)
    t = d.mean() / se if se > 0 else np.inf
    p = 2 * sstats.t.sf(abs(t), k - 1)
    q = sstats.t.ppf(0.975, k - 1)
    ds2 = float(Asp.get("final", np.nan) - Bsp.get("final", np.nan)) if ("final" in Asp.index and "final" in Bsp.index) else np.nan
    out = {"n_splits": k, "cv_diff": float(d.mean()), "ci_low": float(d.mean() - q * se), "ci_high": float(d.mean() + q * se),
           "t": float(t), "p": float(p), "ds2_diff": ds2,
           "ds2_diff_by_seed": None if ds2_seed is None else {str(s): float(v) for s, v in ds2_seed.items()}}
    # DS2 without record 232, and per-class SVEB / VEB recall and precision (plan: every comparison)
    fa, fb = a[a.spec_split == "final"], b[b.spec_split == "final"]
    for m in ("macro_f1_no232", "recall_SVEB", "precision_SVEB", "recall_VEB", "precision_VEB"):
        if m in fa and m in fb and len(fa) and len(fb):
            out[f"ds2_diff_{m}"] = float(fa[m].mean() - fb[m].mean())
    if require_seed_sign and not a["spec_seed"].isna().all():
        per_seed = []
        for sd in sorted(set(a.spec_seed.dropna()) & set(b.spec_seed.dropna())):
            xa = a[(a.spec_seed == sd) & a.spec_split.astype(str).str.startswith("cv-")].groupby("spec_split").macro_f1.mean()
            xb = b[(b.spec_seed == sd) & b.spec_split.astype(str).str.startswith("cv-")].groupby("spec_split").macro_f1.mean()
            j = xa.index.intersection(xb.index)
            per_seed.append(float((xa[j] - xb[j]).mean()))
        out["cv_diff_by_seed"] = per_seed
        out["same_sign_all_seeds"] = bool(per_seed and (all(x > 0 for x in per_seed) or all(x < 0 for x in per_seed)))
    return out

def compare_diff(a_on, a_off, b_on, b_off):
    """F7: (on - off, full) - (on - off, ablated), per split (seeds averaged), corrected t-test."""
    parts = [per_split_seed(x).groupby(level=0).mean() if x is not None and not x.empty else None for x in (a_on, a_off, b_on, b_off)]
    if any(p is None for p in parts):
        return None
    D = (parts[0] - parts[1]) - (parts[2] - parts[3])
    cv = [s for s in D.index if str(s).startswith("cv-") and np.isfinite(D[s])]
    if len(cv) < 3:
        return None
    d = D[cv].values; k = len(d)
    se = math.sqrt(np.var(d, ddof=1) * (1 / k + NB_RATIO)); t = d.mean() / se if se > 0 else np.inf
    q = sstats.t.ppf(0.975, k - 1)
    return {"n_splits": k, "cv_diff": float(d.mean()), "ci_low": float(d.mean() - q * se), "ci_high": float(d.mean() + q * se),
            "t": float(t), "p": float(2 * sstats.t.sf(abs(t), k - 1)), "ds2_diff": float(D.get("final", np.nan)),
            "plain_full_minus_ablated_on": float((parts[0] - parts[2])[cv].mean())}

def holm(rows):
    ps = [r["p"] if r else np.nan for r in rows]
    idx = [i for i, p in enumerate(ps) if np.isfinite(p)]
    order = sorted(idx, key=lambda i: ps[i]); m = len(order); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * ps[i]))
        rows[i]["p_holm"] = run
    return rows

def verdict(r, need_seed_sign=False):
    if r is None:
        return "missing"
    meaningful = abs(r["cv_diff"]) >= 0.01 and r.get("p_holm", 1) < 0.05 and np.sign(r["ds2_diff"]) == np.sign(r["cv_diff"])
    if need_seed_sign:
        meaningful = meaningful and r.get("same_sign_all_seeds", False)
    if meaningful:
        return "meaningful"
    if r["ci_low"] >= -0.02 and r["ci_high"] <= 0.02 and abs(r["ds2_diff"]) < 0.02:
        return "comparable"
    return "inconclusive"

def family(name, items, need_seed_sign=False):
    rows = [it[1] for it in items]
    rows = holm([dict(r) if r else None for r in rows])
    out = []
    for (label, _), r in zip(items, rows):
        out.append({"family": name, "comparison": label, **({} if r is None else {k: v for k, v in r.items() if not isinstance(v, (dict, list))}),
                    "verdict": verdict(r, need_seed_sign)})
    return out

'''
LSM2_FILES['lsm2_tests/test_level1a.py'] = r'''"""Level 1a: every switch against its closed-form solution, to 1e-9 in float64 (plan, Correctness criteria).
One neuron under every reset, gap and tap option; one synapse static, dynamic, with and without a decaying
current; every time conversion on both grids."""
import math
import numpy as np
import pytest
import torch

from lsm2 import config as C, liquid as L, sim as S

TOL = C.TOL_1A


def tiny(factors=frozenset(), n_sub=1, N=2, exc=(True, True), w=None, typ=0, gap=None, n_channels=1):
    """A 2-neuron liquid built from a candidate's settings: neuron 0 -> neuron 1 (one synapse of type `typ`)."""
    base = L.build_candidate(factors, 1, n_sub, n_channels)
    liq = base
    is_exc = np.array(exc, bool)
    s = L.candidate_settings(factors)
    h = C.SAMPLE_MS / n_sub
    tau = np.where(is_exc, s["tau_E_ms"], s["tau_I_ms"])
    liq.N, liq.is_exc, liq.pos = N, is_exc, np.zeros((N, 3))
    liq.beta = np.exp(-h / tau)
    liq.tap = C.TAP * (1 - liq.beta) if s["tap"] else np.zeros(N)
    if s["fine"]:
        liq.gap = np.where(is_exc, L.steps_of(3.0, h), L.steps_of(2.0, h)).astype(np.int64)
        liq.in_a = np.full(N, math.exp(-h / 3.0))
    else:
        liq.gap = np.full(N, L.steps_of(C.SAMPLE_MS, h), np.int64)
    if gap is not None:
        liq.gap = np.full(N, gap, np.int64)
    if w is None:
        liq.pre = liq.post = liq.typ = np.zeros(0, np.int64)
        liq.w = liq.U = liq.D_ms = liq.F_ms = np.zeros(0)
    else:
        liq.pre, liq.post, liq.typ = np.array([0]), np.array([1]), np.array([typ])
        liq.w = np.array([w], float)
        tname = C.TYPES[typ]
        liq.U = np.array([C.MAASS["U"][tname]])
        liq.D_ms = np.array([C.MAASS["D_s"][tname] * 1000])
        liq.F_ms = np.array([C.MAASS["F_s"][tname] * 1000])
    liq.Win = np.ones((n_channels, N))
    liq.in_neurons = np.ones(N, bool)
    return liq


def run(liq, x, T, u0, g=1.0, in_scale=1.0, **kw):
    sm = S.Simulator(liq, dtype=torch.float64, g=g, in_scale=in_scale)
    return sm.run(x, u0, T, trace=True, **kw)


def U_trace(o):
    return np.array(o["trace"]["u"])        # (sub-steps, B, N)


# ---------------------------------------------------------------- neuron
@pytest.mark.parametrize("n_sub", [1, 4, 28])
@pytest.mark.parametrize("factors", [frozenset(), frozenset({4})])
def test_leak_matches_exponential_decay(n_sub, factors):
    liq = tiny(factors, n_sub)
    T = 40
    o = run(liq, np.zeros((1, T, 1)), T, np.full((1, 2), 0.7))
    u = U_trace(o)[:, 0, :]
    s = L.candidate_settings(factors)
    t_ms = (np.arange(T * n_sub) + 1) * C.SAMPLE_MS / n_sub
    want = np.stack([0.7 * np.exp(-t_ms / s["tau_E_ms"])] * 2, 1)
    assert np.abs(u - want).max() < TOL


def test_x_leak_is_papers_beta_per_sample():
    liq = tiny(frozenset(), 1, exc=(True, False))
    o = run(liq, np.zeros((1, 10, 1)), 10, np.full((1, 2), 0.5))
    u = U_trace(o)[:, 0, :]
    assert np.abs(u[:, 0] - 0.5 * 0.95 ** np.arange(1, 11)).max() < TOL
    assert np.abs(u[:, 1] - 0.5 * 0.80 ** np.arange(1, 11)).max() < TOL


@pytest.mark.parametrize("n_sub", [1, 4, 28])
def test_tap_settles_at_0_9(n_sub):
    liq = tiny(frozenset({4, 5}), n_sub)
    T = 200
    o = run(liq, np.zeros((1, T, 1)), T, np.full((1, 2), 0.2))
    u = U_trace(o)[:, 0, 0]
    k = np.arange(1, T * n_sub + 1)
    want = 0.9 + (0.2 - 0.9) * np.exp(-k * (C.SAMPLE_MS / n_sub) / 30.0)
    assert np.abs(u - want).max() < TOL
    assert abs(u[-1] - 0.9) < 1e-6


def test_reset_subtract_constant_drive_first_spike_time():
    liq = tiny(frozenset(), 1)
    c = 0.08
    T = 60
    x = np.ones((1, T, 1))
    o = run(liq, x, T, np.zeros((1, 2)), in_scale=c, rec={"raster": True})
    beta = 0.95
    k1 = math.ceil(math.log(1 - (1 - beta) / c) / math.log(beta))   # smallest k with c (1 - beta^k)/(1 - beta) >= 1
    r = o["raster"][:, 0, 0].numpy()
    assert np.nonzero(r)[0][0] == k1 - 1                            # 0-based step index
    u = U_trace(o)[:, 0, 0]
    u_peak = c * (1 - beta ** k1) / (1 - beta)
    assert abs(u[k1 - 1] - (u_peak - 1)) < TOL                      # reset by subtracting 1


def test_reset_to_tap_and_clamp_during_gap():
    n = 4
    liq = tiny(frozenset({4, 5, 7}), n, exc=(True, False))
    T = 30
    x = np.zeros((1, T, 1)); x[0, 2, 0] = 1
    o = run(liq, x, T, np.full((1, 2), 0.9), in_scale=0.5, rec={"raster": True})
    r = o["raster"][:, 0, :].numpy()
    u = U_trace(o)[:, 0, :]
    for nrn, gap in ((0, 4), (1, 3)):          # 3 ms -> 4 sub-steps, 2 ms -> 3 sub-steps on n = 4
        ks = np.nonzero(r[:, nrn])[0]
        assert len(ks) >= 1
        k = ks[0]
        assert abs(u[k, nrn] - 0.9) < TOL                           # reset to 0.9
        assert np.abs(u[k + 1:k + gap, nrn] - 0.9).max() < TOL       # clamped for gap - 1 sub-steps
        assert abs(u[k + gap, nrn] - 0.9) > 1e-6 or not r[k + gap:, nrn].any()


@pytest.mark.parametrize("n_sub,gapE,gapI", [(4, 4, 3), (28, 30, 20)])
def test_minimum_gap_with_fine_timing(n_sub, gapE, gapI):
    liq = tiny(frozenset({4, 5, 7}), n_sub, exc=(True, False))
    T = 30
    x = np.ones((1, T, 1))
    o = run(liq, x, T, np.full((1, 2), 0.9), in_scale=50.0, rec={"raster": True})
    r = o["raster"][:, 0, :].numpy()
    for nrn, gap in ((0, gapE), (1, gapI)):
        isi = np.diff(np.nonzero(r[:, nrn])[0])
        assert isi.min() == gap
    h = C.SAMPLE_MS / n_sub
    assert abs(gapE * h - (2.78 if n_sub == 4 else 2.98)) < 0.01
    assert abs(gapI * h - (2.08 if n_sub == 4 else 1.98)) < 0.01


@pytest.mark.parametrize("n_sub", [1, 4, 28])
def test_x_gap_on_any_grid_without_clamp(n_sub):
    """X keeps its 2.78 ms gap on any grid: n - 1 blocked sub-steps, voltage still integrating."""
    liq = tiny(frozenset(), n_sub)
    T = 20
    x = np.ones((1, T, 1))
    o = run(liq, x, T, np.zeros((1, 2)), in_scale=3.0, rec={"raster": True})
    r = o["raster"][:, 0, 0].numpy()
    isi = np.diff(np.nonzero(r)[0])
    assert isi.min() == n_sub
    if n_sub > 1:
        u = U_trace(o)[:, 0, 0]
        k = np.nonzero(r)[0][0]
        assert u[k + 1] != u[k]                                       # not clamped


# ---------------------------------------------------------------- synapses
@pytest.mark.parametrize("n_sub", [1, 4])
def test_static_synapse_without_current(n_sub):
    """A spike at sub-step k adds w to its target in sub-step k + delay, all at once (X: delay = one sample)."""
    liq = tiny(frozenset(), n_sub, w=0.3)
    T = 15
    k0 = 1 * n_sub + (n_sub // 2)                                     # presynaptic spike, given open loop
    pre = np.zeros((T * n_sub, 1, 2), bool); pre[k0, 0, 0] = True
    o = run(liq, np.zeros((1, T, 1)), T, np.zeros((1, 2)), open_loop=pre)
    u1 = U_trace(o)[:, 0, 1]
    d = n_sub                                                         # one sample, in sub-steps
    beta = math.exp(-(C.SAMPLE_MS / n_sub) / L.candidate_settings(frozenset())["tau_E_ms"])
    m = np.arange(len(u1))
    want = np.where(m >= k0 + d, 0.3 * beta ** (m - (k0 + d)), 0.0)
    assert np.abs(u1 - want).max() < TOL


@pytest.mark.parametrize("n_sub", [4, 28])
@pytest.mark.parametrize("typ,tau_s", [(0, 3.0), (2, 6.0)])
def test_static_synapse_with_decaying_current(n_sub, typ, tau_s):
    """Fine timing: the push w arrives after its delay as a current decaying over tau_s, total = w.
    u_k = (1-a) w (beta^(m+1) - a^(m+1)) / (beta - a), m sub-steps after arrival."""
    exc = (typ < 2, True)
    liq = tiny(frozenset({4, 7}), n_sub, exc=exc, w=0.3 if typ < 2 else -0.3, typ=typ)
    h = C.SAMPLE_MS / n_sub
    liq.delay = np.array([L.steps_of(C.MAASS["delay_ms"][t], h) for t in C.TYPES])
    liq.cur_a = np.exp(-h / np.array([3.0, 6.0]))
    T = 40
    k0 = 5
    pre = np.zeros((T * n_sub, 1, 2), bool); pre[k0, 0, 0] = True
    o = run(liq, np.zeros((1, T, 1)), T, np.zeros((1, 2)), open_loop=pre)
    d = liq.delay[typ]
    a = math.exp(-h / tau_s)
    beta = math.exp(-h / 30.0)
    u1 = U_trace(o)[:, 0, 1]
    m = np.arange(len(u1)) - (k0 + d)
    w = liq.w[0]
    want = np.where(m >= 0, (1 - a) * w * (beta ** (m + 1) - a ** (m + 1)) / (beta - a), 0.0)
    assert np.abs(u1 - want).max() < TOL
    # total push equals w: sum of the current over all sub-steps
    I = np.array(o["trace"]["I1" if typ < 2 else "I2"])[:, 0, 1]
    assert abs(I.sum() - w * (1 - a ** (len(I) - (k0 + d)))) < TOL


@pytest.mark.parametrize("typ", [0, 1, 2, 3])
def test_dynamic_synapse_markram(typ):
    """Factor 8: per spike u <- u + U (1 - u); push = (w / U-bar) u R; R <- R - u R; between spikes R recovers
    towards 1 over D and u fades towards 0 over F (closed form between spikes)."""
    tname = C.TYPES[typ]
    liq = tiny(frozenset({4, 8}), 1, exc=(typ < 2, typ in (0, 2)), w=0.02 if typ < 2 else -0.02, typ=typ)
    liq.syn_model = "markram"
    T = 120
    rng = np.random.default_rng(typ)
    times = np.sort(rng.choice(np.arange(2, 100), 12, replace=False))
    pre = np.zeros((T, 1, 2), bool); pre[times, 0, 0] = True
    o = run(liq, np.zeros((1, T, 1)), T, np.zeros((1, 2)), open_loop=pre)
    ks = times
    U, D, F, Ub = liq.U[0], liq.D_ms[0], liq.F_ms[0], C.MAASS["U"][tname]
    R, uu, last = 1.0, 0.0, None
    pushes = []
    for k in ks + 1:                                               # arrival one sample later
        if last is not None:
            dt = (k - last) * C.SAMPLE_MS
            R = 1 - (1 - R) * math.exp(-dt / D)
            uu = uu * math.exp(-dt / F)
        else:
            R, uu = 1.0, 0.0
        uu = uu + U * (1 - uu)
        pushes.append(liq.w[0] / Ub * uu * R)
        R = R - uu * R
        last = k
    beta = math.exp(-C.SAMPLE_MS / 30.0)
    u1 = U_trace(o)[:, 0, 1]
    want = np.zeros(T)
    for k, p in zip(ks + 1, pushes):
        m = np.arange(T) - k
        want += np.where(m >= 0, p * beta ** np.maximum(m, 0), 0.0)
    assert np.abs(u1 - want).max() < TOL


def test_isolated_dynamic_push_equals_w_U_over_Ubar():
    liq = tiny(frozenset({4, 8}), 1, w=0.1, typ=0)
    liq.syn_model = "markram"
    liq.U = np.array([0.35])
    pre = np.zeros((10, 1, 2), bool); pre[1, 0, 0] = True
    o = run(liq, np.zeros((1, 10, 1)), 10, np.zeros((1, 2)), open_loop=pre)
    u1 = U_trace(o)[:, 0, 1]
    assert abs(u1[2] - 0.1 * 0.35 / 0.5) < TOL


# ---------------------------------------------------------------- time conversions on both grids
@pytest.mark.parametrize("n_sub", [1, 4, 28])
def test_total_input_push_same_on_every_grid(n_sub):
    """One input spike: without fine timing its push arrives at once in its own sample's first sub-step;
    with fine timing as a 3 ms current whose total equals the same push."""
    for factors in (frozenset({4}), frozenset({4, 7})):
        if 7 in factors and n_sub == 1:
            continue
        liq = tiny(factors, n_sub)
        T = 60
        x = np.zeros((1, T, 1)); x[0, 3, 0] = 1
        o = run(liq, x, T, np.zeros((1, 2)), in_scale=0.2)
        I = np.array(o["trace"]["I3"])[:, 0, 0]
        assert np.nonzero(I)[0][0] == 3 * n_sub                       # first sub-step of sample 3
        a = math.exp(-(C.SAMPLE_MS / n_sub) / 3.0)
        tail = a ** (len(I) - 3 * n_sub) if 7 in factors else 0.0
        assert abs(I.sum() - 0.2 * (1 - tail)) < TOL


def test_delay_conversions():
    for n, (dEE, dO) in ((4, (2, 1)), (28, (15, 8))):
        h = C.SAMPLE_MS / n
        assert L.steps_of(1.5, h) == dEE and L.steps_of(0.8, h) == dO
    liq = L.build_candidate(frozenset({7}), 1, 4, 10)
    assert list(liq.delay) == [2, 1, 1, 1]
    liq = L.build_candidate(frozenset(), 1, 4, 10)
    assert list(liq.delay) == [4, 4, 4, 4]            # X: one sample on any grid


def test_constants():
    assert abs(C.MAASS_EE_PUSH - 0.1) < 1e-12
    assert abs(C.MAASS_IN_PUSH["E"] - 0.12) < 1e-12 and abs(C.MAASS_IN_PUSH["I"] - 0.06) < 1e-12
    r = C.MAASS_RATIOS
    assert abs(r["EI"] - 0.2) < 1e-12 and abs(r["IE"] - 0.6333) < 1e-3 and abs(r["II"] - 0.8107) < 1e-3
    assert abs(C.TAP - 0.9) < 1e-12 and abs(C.I_TARGET - 0.045) < 1e-12
    assert abs(math.exp(-C.SAMPLE_MS / 30.0) - 0.912) < 1e-3       # plan: 30 ms is 0.912 per step


# ---------------------------------------------------------------- Brian2 schedule (script version)
def test_script_neuron_exact_solution_with_decaying_input():
    """(d)'s neuron: exact update with the input current decaying over 3 ms (E) or 6 ms (I):
    v(t) = J tau_s / (tau_s - tau_m) (e^(-t/tau_s) - e^(-t/tau_m)) after a jump J (no tap, no recurrence)."""
    liq = L.build_maass_script(1, n_sub=1, step_ms=0.1)
    liq.J_b = 0.0
    sel = [0, liq.N - 1]                                            # one E, one I neuron
    liq.N = 2; liq.is_exc = liq.is_exc[sel]; liq.beta = liq.beta[sel]; liq.kappa = liq.kappa[sel]
    liq.in_a = liq.in_a[sel]; liq.gap = liq.gap[sel]; liq.Win = np.array([[0.5, 0.5]])
    liq.pre = liq.post = liq.typ = np.zeros(0, np.int64); liq.w = liq.U = liq.D_ms = liq.F_ms = np.zeros(0)
    T = 300
    x = np.zeros((1, T, 1)); x[0, 0, 0] = 1
    o = run(liq, x, T, np.zeros((1, 2)))
    u = U_trace(o)[:, 0, :]
    t = np.arange(T) * 0.1                                         # the jump lands after step 0's update
    for i, ts in ((0, 3.0), (1, 6.0)):
        want = 0.5 * ts / (ts - 30.0) * (np.exp(-t / ts) - np.exp(-t / 30.0))
        assert np.abs(u[:, i] - want).max() < TOL


def test_script_three_state_synapse():
    """(d)'s synapse: dx/dt = (1 - x - y)/D, dy/dt = -y/tau; on a spike u += U (1-u), y += u x, x -= u x."""
    liq = L.build_maass_script(1, n_sub=1, step_ms=0.1)
    m = np.nonzero((liq.typ == 0) & (liq.pre != liq.post))[0][0]
    liq.N = 2; liq.is_exc = np.array([True, True]); liq.beta = liq.beta[:2]; liq.kappa = liq.kappa[:2]
    liq.in_a = liq.in_a[:2]; liq.gap = np.array([1, 1]); liq.Win = np.array([[50.0, 0.0]])
    liq.pre, liq.post, liq.typ = np.array([0]), np.array([1]), np.array([0])
    for a in ("w", "U", "D_ms", "F_ms"):
        setattr(liq, a, getattr(liq, a)[[m]])
    T = 400
    x = np.zeros((1, T, 1)); spikes_in = [5, 60, 90, 250]
    x[0, spikes_in, 0] = 1
    o = run(liq, x, T, np.full((1, 2), 0.0), rec={"raster": True})
    ks = np.nonzero(o["raster"][:, 0, 0].numpy())[0]
    U, D, F, tau = liq.U[0], liq.D_ms[0], liq.F_ms[0], 3.0
    xs, ys, us = 1.0, 0.0, 0.0
    h = 0.1
    want_y = []
    arrivals = set(int(k) + liq.delay[0] for k in ks)
    for k in range(T):
        us *= math.exp(-h / F)
        xs = xs * math.exp(-h / D) + 1 - math.exp(-h / D) + ys * tau * (math.exp(-h / tau) - math.exp(-h / D)) / (D - tau)
        ys = ys * math.exp(-h / tau)
        if k in arrivals:
            us += U * (1 - us); ys += us * xs; xs -= us * xs
        want_y.append(ys)
    got_y = np.array([s["y"][0, 0] for s in o["trace"]["syn"]])
    assert np.abs(got_y - np.array(want_y)).max() < TOL


@pytest.mark.parametrize("n_sub", [4, 28])
def test_dynamic_synapse_with_decaying_current(n_sub):
    """Factors 7 + 8: each push of the dynamic synapse arrives after its delay as a 3 ms current."""
    liq = tiny(frozenset({4, 7, 8}), n_sub, w=0.02, typ=0)
    liq.syn_model = "markram"
    h = C.SAMPLE_MS / n_sub
    liq.delay = np.array([L.steps_of(C.MAASS["delay_ms"][t], h) for t in C.TYPES])
    liq.cur_a = np.exp(-h / np.array([3.0, 6.0]))
    T = 60
    times = np.array([7, 30, 41, 90, 95, 150]) * (n_sub // 4)
    pre = np.zeros((T * n_sub, 1, 2), bool); pre[times, 0, 0] = True
    o = run(liq, np.zeros((1, T, 1)), T, np.zeros((1, 2)), open_loop=pre)
    U, D, F, Ub = liq.U[0], liq.D_ms[0], liq.F_ms[0], 0.5
    d = liq.delay[0]
    a, beta = math.exp(-h / 3.0), math.exp(-h / 30.0)
    R, uu, last, want = 1.0, 0.0, None, np.zeros(T * n_sub)
    for k in times + d:
        if last is not None:
            dt = (k - last) * h
            R = 1 - (1 - R) * math.exp(-dt / D); uu *= math.exp(-dt / F)
        uu += U * (1 - uu)
        p = liq.w[0] / Ub * uu * R
        R -= uu * R
        last = k
        m = np.arange(T * n_sub) - k
        want += np.where(m >= 0, (1 - a) * p * (beta ** (m + 1) - a ** (m + 1)) / (beta - a), 0.0)
    assert np.abs(U_trace(o)[:, 0, 1] - want).max() < TOL
'''
LSM2_FILES['lsm2_tests/test_script_mode.py'] = r'''"""Our code in the script's settings against (d)'s own Brian2 script on the same realised liquid, the same
stimulus and the same initial voltages: the spike trains must be identical (a deterministic version of 1e)."""
import os
import numpy as np
import pytest
import torch

from lsm2 import config as C, liquid as L, sim as S, brian2_ref

SCRIPT = os.environ.get("MAASS_SCRIPT", "/home/claude/ext/brian2ex/maass2002.py")


@pytest.mark.skipif(not os.path.exists(SCRIPT), reason="(d)'s script not available")
def test_script_mode_reproduces_d_exactly():
    mod = brian2_ref.script_module(SCRIPT)
    mod.prefs.codegen.target = "numpy"
    np.random.seed(5)
    import brian2; brian2.start_scope()
    neurons = mod.get_neurons()
    N_exc = int(0.8 * len(neurons))
    exc, inh = neurons[:N_exc], neurons[N_exc:]
    exc.tau_refrac = 3 * mod.ms; exc.tau_stimulus = 3 * mod.ms
    inh.tau_refrac = 2 * mod.ms; inh.tau_stimulus = 6 * mod.ms
    s, nA, ms = mod.second, mod.nA, mod.ms
    specs = [("ee_synapses", exc, exc, 0.3, 3, 30, 0.5, 1.1, 0.05, 1.5, 0), ("ei_synapses", exc, inh, 0.2, 3, 60, 0.05, 0.125, 1.2, 0.8, 1),
             ("ie_synapses", inh, exc, 0.4, 6, -19, 0.25, 0.7, 0.02, 0.8, 2), ("ii_synapses", inh, inh, 0.1, 6, -19, 0.32, 0.144, 0.06, 0.8, 3)]
    syns = [mod.get_synapses(nm, a, b, C=c, l=2, tau_I=ti * ms, A=A * nA, U=U, D=D * s, F=F * s, delay=dl * ms)
            for nm, a, b, c, ti, A, U, D, F, dl, _ in specs]
    stim = mod.SpikeGeneratorGroup(1, [], [] * ms, name="stimulus")
    se = mod.Synapses(stim, exc, "A : ampere (shared, constant)", on_pre="I_stimulus += A"); se.connect(p=1); se.A = 18 * nA
    si = mod.Synapses(stim, inh, "A : ampere (shared, constant)", on_pre="I_stimulus += A"); si.connect(p=1); si.A = 9 * nA
    me = mod.SpikeMonitor(exc, name="spike_monitor_exc"); mi = mod.SpikeMonitor(inh, name="spike_monitor_inh")
    mod.defaultclock.dt = mod.DT
    net = mod.Network([neurons, *syns, se, si, stim, me, mi])
    v0 = np.random.uniform(13.5, 15, len(neurons))
    neurons.v = v0 * mod.mV
    spikes = np.sort(np.random.uniform(0, 200, 6))
    spikes = np.round(spikes, 1)
    stim.set_spikes([0] * len(spikes), spikes * ms)
    T = 2000                                   # 200 ms at 0.1 ms
    net.run(T * 0.1 * ms)
    b_r = np.zeros((T, len(neurons)), bool)
    for mon, off in ((me, 0), (mi, N_exc)):
        k = np.round(np.asarray(mon.t / ms) / 0.1).astype(int)
        b_r[k, np.asarray(mon.i) + off] = True
    # our liquid with the same realised parameters
    liq = L.build_maass_script(1, n_sub=1, step_ms=0.1)
    pos = np.stack([np.asarray(neurons.x_pos), np.asarray(neurons.y_pos), np.asarray(neurons.z_pos)], 1)
    pre, post, typ, w, U, D, F = [], [], [], [], [], [], []
    for (nm, a, b, *_, ti_), syn in zip(specs, syns):
        oi = 0 if a is exc else N_exc
        oj = 0 if b is exc else N_exc
        pre.append(np.asarray(syn.i) + oi); post.append(np.asarray(syn.j) + oj); typ.append(np.full(len(syn), ti_))
        w.append(np.asarray(syn.A / nA) / 15.0); U.append(np.asarray(syn.U))
        D.append(np.asarray(syn.D / ms)); F.append(np.asarray(syn.F / ms))
    liq.pos = pos
    liq.pre, liq.post, liq.typ = (np.concatenate(v) for v in (pre, post, typ))
    liq.w, liq.U, liq.D_ms, liq.F_ms = (np.concatenate(v) for v in (w, U, D, F))
    x = np.zeros((1, T, 1), np.uint8)
    x[0, np.round(spikes / 0.1).astype(int), 0] = 1
    sm = S.Simulator(liq, dtype=torch.float64, g=1.0, in_scale=1.0)
    o = sm.run(x, (v0 / 15.0)[None], T, rec={"raster": True})
    r = o["raster"][:, 0, :].numpy()
    assert b_r.sum() > 50
    assert (r != b_r).sum() == 0, f"{(r != b_r).sum()} spike mismatches of {b_r.sum()}"
'''
LSM2_FILES['lsm2_tests/test_ytest.py'] = r'''"""Checks behind the Y vs Y-off notebook's own claims."""
import numpy as np
import torch

from lsm2 import config as C, gain, liquid as L, sim as S


def test_cube_structure_does_not_depend_on_input_channels():
    """y_cube: the Poisson (1 channel) and ECG (200 channels) versions of a seed are the same recurrent liquid."""
    a = L.build_candidate(C.ALL_FACTORS, 3, 2, 1)
    b = L.build_candidate(C.ALL_FACTORS, 3, 2, 200)
    assert a.dims == (15, 3, 3) and a.N == 135
    for k in ("pre", "post", "typ", "w", "U", "D_ms", "F_ms", "is_exc", "in_neurons"):
        assert np.array_equal(getattr(a, k), getattr(b, k)), k
    assert a.g_maass == b.g_maass and a.in_scale_fixed == b.in_scale_fixed


def test_y_off_equals_zero_gain():
    """Y-off (recurrence=False) is the same run as g = 0: same input, no recurrent push."""
    liq = L.build_candidate(C.ALL_FACTORS, 1, 2, 5)
    rng = np.random.default_rng(0)
    x = (rng.random((3, 40, 5)) < 0.3).astype(np.uint8)
    u0 = S.start_voltages(liq, 1, range(3), 4)
    rec = {"windows": (4, 10), "states_at": [39]}
    a = S.Simulator(liq, dtype=torch.float64, g=liq.g_maass, in_scale=liq.in_scale_fixed, recurrence=False).run(x, u0, 40, rec=rec)
    b = S.Simulator(liq, dtype=torch.float64, g=0.0, in_scale=liq.in_scale_fixed).run(x, u0, 40, rec=rec)
    assert torch.equal(a["windows"], b["windows"]) and torch.allclose(a["states_at"], b["states_at"])


def test_recurrence_changes_the_run():
    """With the same input and start, Y and Y-off differ (the recurrent synapses do something)."""
    liq = L.build_candidate(C.ALL_FACTORS, 1, 2, 5)
    rng = np.random.default_rng(1)
    x = (rng.random((4, 200, 5)) < 0.3).astype(np.uint8)
    u0 = S.start_voltages(liq, 1, range(4), 4)
    rec = {"neuron_counts": (0, 200)}
    on = S.Simulator(liq, dtype=torch.float64, g=liq.g_maass, in_scale=liq.in_scale_fixed).run(x, u0, 200, rec=rec)
    off = S.Simulator(liq, dtype=torch.float64, g=liq.g_maass, in_scale=liq.in_scale_fixed, recurrence=False).run(x, u0, 200, rec=rec)
    assert not torch.equal(on["neuron_counts"], off["neuron_counts"])


def test_gain_rule_without_band():
    """With the band switched off (the notebook's setting), the rule takes the largest mean gap; ties go to the
    smallest g within the best setting's seed range."""
    old = C.BAND_HZ, C.HIGH_FRAC
    C.BAND_HZ, C.HIGH_FRAC = (0.0, float("inf")), 1.0
    try:
        def row(gap, rate, seeds):
            m = {"gap": gap, "rate_hz": rate, "frac_above_100hz": 0.0, "gap_seeds": seeds}
            m["in_band"] = True
            return m
        t = {0.0: row(1.0, 0.1, [1, 1]), 1.0: row(3.0, 0.5, [2.9, 3.1]), 3.0: row(5.0, 80.0, [4.0, 6.0]),
             10.0: row(4.5, 150.0, [4.4, 4.6])}
        chosen, best = gain.choose(t)
        assert best == 3.0 and chosen == 3.0
        t[2.0] = row(4.2, 30.0, [4.1, 4.3])         # inside [4, 6]: tied, and smaller
        chosen, best = gain.choose(t)
        assert best == 3.0 and chosen == 2.0
    finally:
        C.BAND_HZ, C.HIGH_FRAC = old
'''
LSM2_FILES['lsm2_tests/test_stats.py'] = r'''"""Statistics and decision rules on synthetic run tables."""
import math
import numpy as np
import pandas as pd
import pytest
from scipy import stats as sstats

from lsm2 import stats as S

SPLITS = [f"cv-r{r}f{f}" for r in (1, 2, 3) for f in (1, 2, 3, 4)]


def runs(base, shift=0.0, seeds=(2026, 69, 420), noise=0.01, seed_free=False, rng=None, ds2=None):
    rng = rng or np.random.default_rng(0)
    rows = []
    for sd in ([None] if seed_free else seeds):
        for i, sp in enumerate(SPLITS + ["final"]):
            v = base[i] + shift + noise * rng.standard_normal()
            if sp == "final" and ds2 is not None:
                v = ds2
            rows.append({"spec_split": sp, "spec_seed": sd, "macro_f1": v, "macro_f1_no232": v,
                         "recall_SVEB": v, "precision_SVEB": v, "recall_VEB": v, "precision_VEB": v})
    return pd.DataFrame(rows)


def test_nb_test_matches_formula():
    rng = np.random.default_rng(1)
    base = 0.6 + 0.05 * rng.standard_normal(13)
    a, b = runs(base, 0.03, rng=rng), runs(base, 0.0, rng=rng)
    r = S.compare(a, b)
    A = a.groupby(["spec_split", "spec_seed"]).macro_f1.mean().groupby(level=0).mean()
    B = b.groupby(["spec_split", "spec_seed"]).macro_f1.mean().groupby(level=0).mean()
    d = (A[SPLITS] - B[SPLITS]).values
    se = math.sqrt(np.var(d, ddof=1) * (1 / 12 + 0.336))
    assert r["n_splits"] == 12 and r["cv_diff"] == pytest.approx(d.mean())
    assert r["t"] == pytest.approx(d.mean() / se)
    assert r["p"] == pytest.approx(2 * sstats.t.sf(abs(d.mean() / se), 11))
    assert r["ci_high"] - r["cv_diff"] == pytest.approx(sstats.t.ppf(0.975, 11) * se)
    assert r["ds2_diff"] == pytest.approx(A["final"] - B["final"])


def test_verdicts():
    big = {"cv_diff": 0.05, "ci_low": 0.03, "ci_high": 0.07, "ds2_diff": 0.04, "p_holm": 0.001, "same_sign_all_seeds": True}
    assert S.verdict(big) == "meaningful"
    assert S.verdict({**big, "ds2_diff": -0.01}) == "inconclusive"            # DS2 must have the same sign
    assert S.verdict({**big, "same_sign_all_seeds": False}, need_seed_sign=True) == "inconclusive"
    assert S.verdict({**big, "cv_diff": 0.008, "ci_low": 0.007, "ci_high": 0.009, "ds2_diff": 0.005}) == "comparable"
    small = {"cv_diff": 0.001, "ci_low": -0.015, "ci_high": 0.017, "ds2_diff": 0.01, "p_holm": 0.8}
    assert S.verdict(small) == "comparable"
    assert S.verdict({**small, "ds2_diff": 0.03}) == "inconclusive"
    assert S.verdict(None) == "missing"


def test_holm():
    rows = S.holm([{"p": 0.01}, {"p": 0.04}, {"p": 0.03}, None])
    assert rows[0]["p_holm"] == pytest.approx(0.03)
    assert rows[2]["p_holm"] == pytest.approx(0.06)
    assert rows[1]["p_holm"] == pytest.approx(0.06)          # monotone


def test_seed_sign_and_seed_free_control():
    rng = np.random.default_rng(2)
    base = 0.6 + 0.05 * rng.standard_normal(13)
    a = runs(base, 0.02, rng=rng, noise=0.001)
    c = runs(base, 0.0, rng=rng, noise=0.001, seed_free=True)
    r = S.compare(a, c, require_seed_sign=True)
    assert r is not None and r["cv_diff"] == pytest.approx(0.02, abs=0.002)


def test_compare_diff_f7():
    rng = np.random.default_rng(3)
    base = 0.6 + 0.05 * rng.standard_normal(13)
    on_full, off_full = runs(base, 0.04, rng=rng, noise=0.001), runs(base, 0.0, rng=rng, noise=0.001)
    on_abl, off_abl = runs(base, 0.01, rng=rng, noise=0.001), runs(base, 0.0, rng=rng, noise=0.001)
    r = S.compare_diff(on_full, off_full, on_abl, off_abl)
    assert r["cv_diff"] == pytest.approx(0.03, abs=0.002) and r["p"] < 1e-6


def test_family_applies_holm():
    rng = np.random.default_rng(4)
    base = 0.6 + 0.05 * rng.standard_normal(13)
    items = [(f"c{i}", S.compare(runs(base, 0.03 * i, rng=rng), runs(base, 0.0, rng=rng))) for i in range(3)]
    fam = S.family("F", items)
    assert [f["comparison"] for f in fam] == ["c0", "c1", "c2"] and all("p_holm" in f for f in fam)
'''

import shutil
for _d in ("lsm2", "lsm2_tests"):                 # only this notebook's files (an older lsm2 may be there)
    shutil.rmtree(os.path.join(SCRATCH_DIR, _d), ignore_errors=True)
for _rel, _src in LSM2_FILES.items():
    _p = os.path.join(SCRATCH_DIR, _rel)
    os.makedirs(os.path.dirname(_p), exist_ok=True)
    with open(_p, "w") as _f:
        _f.write(_src)
open(os.path.join(SCRATCH_DIR, "lsm2", "__init__.py"), "w").close()
if SCRATCH_DIR not in sys.path:
    sys.path.insert(0, SCRATCH_DIR)
for _m in [m for m in sys.modules if m == "lsm2" or m.startswith("lsm2.")]:
    del sys.modules[_m]
import lsm2
print(f"lsm2: {len(LSM2_FILES)} files at {os.path.join(SCRATCH_DIR, 'lsm2')}")


## 4. Utilities: logging, metrics, the authors' readout

In [ ]:
# Utilities: output folders, logging, cache.txt, resume, time/disk guards, metrics, the authors' readout.
CODE_VERSION = "v1"   # part of every run's hash; bump only if experiment code changes meaning
for sub in ["runs", "reservoirs", "states", "tables", "figures", "logs", "checks"]:
    os.makedirs(os.path.join(WORK_DIR, sub), exist_ok=True)

def elapsed_h():
    return (time.time() - SESSION_START) / 3600

LOCK = threading.RLock()     # cache.txt, the log, speeds and disk counters are shared with the fit workers

def log(*parts):
    line = f"[{elapsed_h():6.2f} h] " + " ".join(str(p) for p in parts)
    with LOCK:
        print(line, flush=True)               # the tee writes it to LOG_PATH

def _json_default(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, torch.Tensor): return o.detach().cpu().tolist()
    return str(o)

def atomic_write_text(path, text):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        f.write(text)
    os.replace(tmp, path)

def jdump(obj, path):
    atomic_write_text(path, json.dumps(obj, indent=2, default=_json_default))

def stable_hash(obj, n=8):
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=_json_default).encode()).hexdigest()[:n]

# ---------------- cache.txt ----------------
CACHE_PATH = os.path.join(WORK_DIR, "cache.txt")
MACHINE_ID = f"{MACHINE}|{ENV.get('gpu', DEVICE.type)}|{N_CPU} cores"      # identifies the machine in merged outputs

def out_tag(share, machine_id):
    """Folder-safe name of an output: its share and machine."""
    return re.sub(r"[^A-Za-z0-9.]+", "_", f"{share if share not in (None, 'None') else 'all'}__{machine_id or 'unknown'}").strip("_")

class RunCache:
    """cache.txt = the record of what has run. Rewritten (atomically) after every run.
    Sections: [DONE] run_id, finished_utc, seconds, macro_f1, accuracy, group
              [FAILED] run_id, when_utc, error
              [PENDING] run_id, group"""
    def __init__(self):
        self.done, self.failed, self.pending = {}, {}, []

    def merge_file(self, path):
        if not os.path.exists(path):
            return
        section = None
        for line in open(path):
            line = line.rstrip("\n")
            if line.startswith("[") and line.endswith("]"):
                section = line[1:-1]
                continue
            if not line or line.startswith("#"):
                continue
            parts = line.split("\t")
            if section == "DONE":
                self.done[parts[0]] = parts[1:]
                self.failed.pop(parts[0], None)
            elif section == "FAILED" and parts[0] not in self.done:
                self.failed[parts[0]] = parts[1:]

    def is_done(self, rid):
        return rid in self.done

    def mark_done(self, rid, seconds, metrics, group):
        with LOCK:
            self._mark_done(rid, seconds, metrics, group)

    def _mark_done(self, rid, seconds, metrics, group):
        mf1 = f"{metrics['macro_f1']:.4f}" if metrics and "macro_f1" in metrics else "-"
        acc = f"{metrics['accuracy']:.4f}" if metrics and "accuracy" in metrics else "-"
        self.done[rid] = [datetime.datetime.utcnow().isoformat(timespec="seconds"), f"{seconds:.1f}", mf1, acc, group]
        self.failed.pop(rid, None)
        self.write()

    def mark_failed(self, rid, err):
        with LOCK:
            self._mark_failed(rid, err)

    def _mark_failed(self, rid, err):
        self.failed[rid] = [datetime.datetime.utcnow().isoformat(timespec="seconds"), err.replace("\t", " ").replace("\n", " | ")[:500]]
        self.write()

    def write(self):
        with LOCK:
            self._write()

    def _write(self):
        lines = ["# LSM-ECG replication: run cache. Rewritten after every run; read by the next session to skip finished work.",
                 f"# updated_utc: {datetime.datetime.utcnow().isoformat(timespec='seconds')}   session_elapsed_h: {elapsed_h():.2f}",
                 f"# totals: DONE {len(self.done)} | FAILED {len(self.failed)} | PENDING {len(self.pending)}",
                 f"# smoke_test: {SMOKE_TEST}",
                 f"# share: {SHARE}",
                 f"# machine_id: {MACHINE_ID}",
                 f"# code_version: {CODE_VERSION}",
                 f"# notebook_build: {NOTEBOOK_BUILD}",
                 "# DONE columns: run_id, finished_utc, seconds, macro_f1, accuracy, group", "[DONE]"]
        lines += ["\t".join([k] + v) for k, v in self.done.items()]
        lines += ["", "# FAILED columns: run_id, when_utc, error", "[FAILED]"]
        lines += ["\t".join([k] + v) for k, v in self.failed.items()]
        lines += ["", "# PENDING columns: run_id, group", "[PENDING]"]
        lines += [f"{rid}\t{g}" for rid, g in self.pending]
        atomic_write_text(CACHE_PATH, "\n".join(lines) + "\n")

CACHE = RunCache()

# ---------------- resume (previous session) and merge mode (two machines' outputs) ----------------
RESUME_SUBDIRS = ["runs", "reservoirs", "states", "logs", "checks"]   # outputs worth carrying; tables/figures are rebuilt

def _cache_is_smoke(p):
    return any(l.strip() == "# smoke_test: True" for l in itertools.islice(open(p), 12))

def cache_header(p):
    h = {}
    for l in itertools.islice(open(p), 12):
        m = re.match(r"^# (smoke_test|share|machine_id|code_version|notebook_build): (.*)$", l.rstrip("\n"))
        if m:
            h[m.group(1)] = m.group(2).strip()
    return h

def copy_forward(src_root):
    n = 0
    for sub in RESUME_SUBDIRS:
        for root, _, files in os.walk(os.path.join(src_root, sub)):
            rel = os.path.relpath(root, src_root)
            for fn in files:
                dst = os.path.normpath(os.path.join(WORK_DIR, rel, fn))
                if not os.path.exists(dst):
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    shutil.copy2(os.path.join(root, fn), dst)
                    n += 1
    return n

if MERGE_MODE:
    for md in MERGE_DIRS:
        if os.path.abspath(md) == os.path.abspath(WORK_DIR):
            raise ValueError("merge mode: WORK_DIR must be a new folder, not one of MERGE_DIRS")
        if not os.path.exists(os.path.join(md, "cache.txt")):
            raise FileNotFoundError(f"merge mode: no cache.txt in {md} (give the output folders themselves)")
    _hdrs = [cache_header(os.path.join(md, "cache.txt")) for md in MERGE_DIRS]
    if len({h.get("notebook_build") for h in _hdrs}) > 1:
        print("WARNING: the folders were made by different notebook builds: "
              + ", ".join(f"{md}: {h.get('notebook_build', '?')}" for md, h in zip(MERGE_DIRS, _hdrs)))

if RESUME_DIRS == "auto":
    RESUME_DIRS = sorted({os.path.dirname(p) for p in glob.glob(os.path.join(AUTO_RESUME_ROOT, "**", "cache.txt"), recursive=True)})
for rd in (MERGE_DIRS if MERGE_MODE else RESUME_DIRS):
    cp = os.path.join(rd, "cache.txt")
    if os.path.abspath(rd) == os.path.abspath(WORK_DIR):
        continue
    if not os.path.exists(cp):
        print(f"resume: no cache.txt in {rd} -> fresh start")
        continue
    if _cache_is_smoke(cp) != SMOKE_TEST:
        msg = f"{rd}: its smoke_test flag differs from this session's SMOKE_TEST={SMOKE_TEST}"
        if MERGE_MODE:
            raise ValueError("merge mode: " + msg)
        print(f"resume: skipping {msg}")
        continue
    before = len(CACHE.done)
    _o = RunCache(); _o.merge_file(cp); n_there = len(_o.done)
    CACHE.merge_file(cp)
    n = copy_forward(rd)
    n_dirs = len(os.listdir(os.path.join(rd, "runs"))) if os.path.isdir(os.path.join(rd, "runs")) else 0
    h = cache_header(cp)
    print(f"{'merged' if MERGE_MODE else 'resumed'} from {rd} (share {h.get('share', '?')}, {h.get('machine_id', '?')}): "
          f"{n_there} finished runs there, {len(CACHE.done) - before} of them new here, {n} files copied")
    if n_dirs < len(CACHE.done) - before - 2:
        print(f"WARNING: its cache.txt lists {len(CACHE.done) - before} finished runs but only {n_dirs} run folders are there. "
              "They will be skipped, but their results will be missing from tables and figures. "
              "Bring the whole output folder, not only cache.txt.")
if os.path.exists(CACHE_PATH) and _cache_is_smoke(CACHE_PATH) != SMOKE_TEST:
    raise RuntimeError(f"{WORK_DIR} holds {'smoke-test' if _cache_is_smoke(CACHE_PATH) else 'real'} results but SMOKE_TEST = "
                       f"{SMOKE_TEST}. Use a separate WORK_DIR for the smoke test so the two never mix.")
CACHE.merge_file(CACHE_PATH)       # this machine's earlier sessions (same folder, reused in place)
print(f"cache: {len(CACHE.done)} done, {len(CACHE.failed)} failed")

# ---------------- time and disk guards ----------------
def seconds_left():
    return SESSION_BUDGET_HOURS * 3600 - (time.time() - SESSION_START) - ANALYSIS_RESERVE_MIN * 60

_DISK = {"bytes": None, "since": 0}
def work_dir_gb(force=False):
    if force or _DISK["bytes"] is None or _DISK["since"] >= 50:
        total = 0
        for root, _, files in os.walk(WORK_DIR):
            for fn in files:
                try:
                    total += os.path.getsize(os.path.join(root, fn))
                except OSError:
                    pass
        _DISK.update(bytes=total, since=0)
    return _DISK["bytes"] / 1e9

def disk_add(nbytes):
    with LOCK:
        if _DISK["bytes"] is not None:
            _DISK["bytes"] += nbytes
            _DISK["since"] += 1

# Throughput model, measured as the session runs (and carried over between sessions in checks/speeds.json).
# Units: beats/s for simulation and encoding; seconds per (training beats x features / 1e9) for a classifier fit;
# seconds per epoch per 1,000 training beats for the CNN/GRU.
_GPU = DEVICE.type == "cuda"
# GPU defaults: roughly what a Kaggle T4 measured (used only until this machine has measured its own)
SPEED = {"sim_beats_per_s": {e: (2000.0 if _GPU else 300.0) for e in ("multi", "rate", "latency")},
         "encode_beats_per_s": {e: (20000.0 if _GPU else 1500.0) for e in ("multi", "rate", "latency")},
         "fit_s_per_Gcell": {"1": 400.0, "2": 220.0, "3": 160.0, "pipe": 400.0}, "fit_threads": float(READOUT_JOBS),
         "nn_s_per_epoch_kbeat": {"cnn": 0.04 if _GPU else 3.0, "gru": 0.025 if _GPU else 8.0},
         "nn_unit_s_per_epoch_kbeat": {"cnn": 0.03 if _GPU else 3.0, "gru": 0.04 if _GPU else 8.0},
         "lsm_job_overhead_s": 3.0, "huang_s_per_run": 1200.0}
SPEED_SAMPLES = {}                # recent measurements of the robust (median) entries
# fit_s_per_Gcell: "1"/"2"/"3" = one fit using that many cores (older runner); "pipe" = one fit per core while all
# the pipeline's workers are busy (includes their competition for memory bandwidth).
# nn_unit_*: seconds per epoch per 1,000 beats for a tuned architecture, divided by its relative cost (see nn_cost).
SPEED_MEASURED = set()            # which entries were measured (vs default guesses)
SPEEDS_PATH = os.path.join(WORK_DIR, "checks", "speeds.json")
if os.path.exists(SPEEDS_PATH):
    try:
        _sp = json.load(open(SPEEDS_PATH))
        if _sp.get("device") == DEVICE.type and _sp.get("gpu") == ENV.get("gpu") and _sp.get("cpu_count") == N_CPU:
            for k, v in _sp["speed"].items():
                if isinstance(SPEED.get(k), dict) and isinstance(v, dict):
                    SPEED[k].update(v)
                elif k in SPEED and not isinstance(SPEED[k], dict) and not isinstance(v, dict):
                    SPEED[k] = v                                               # entries whose format changed are skipped
            SPEED_MEASURED.update(m for m in _sp["measured"] if not (m == "fit_s_per_Gcell"))
            SPEED_SAMPLES.update(_sp.get("samples", {}))
            print("speeds carried over from an earlier session:", sorted(SPEED_MEASURED))
    except Exception as ex:
        print("could not read speeds.json:", ex)

def update_speed(key, value, sub=None, alpha=0.3, robust=False):
    with LOCK:
        if robust:            # median of the last 40 measurements: one odd fit does not move the projection
            tag = f"{key}:{sub}" if sub else key
            xs = SPEED_SAMPLES.setdefault(tag, []); xs.append(float(value)); del xs[:-40]
            value, alpha = float(np.median(xs)), 1.0
        _update_speed(key, value, sub, alpha)

def _update_speed(key, value, sub, alpha):
    tag = f"{key}:{sub}" if sub else key
    if sub is None:
        SPEED[key] = value if tag not in SPEED_MEASURED else alpha * value + (1 - alpha) * SPEED[key]
    else:
        SPEED[key][sub] = value if tag not in SPEED_MEASURED else alpha * value + (1 - alpha) * SPEED[key][sub]
    SPEED_MEASURED.add(tag)

def save_speeds():
    with LOCK:
        jdump({"device": DEVICE.type, "gpu": ENV.get("gpu"), "cpu_count": N_CPU, "speed": SPEED, "samples": SPEED_SAMPLES, "measured": sorted(SPEED_MEASURED)}, SPEEDS_PATH)

# ---------------- metrics ----------------
CLASS_NAMES = ["N", "SVEB", "VEB"]

def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    tp = np.diag(cm).astype(float)
    support, predicted, total = cm.sum(1).astype(float), cm.sum(0).astype(float), cm.sum()
    rec = np.divide(tp, support, out=np.zeros(3), where=support > 0)
    prec = np.divide(tp, predicted, out=np.zeros(3), where=predicted > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(3), where=(prec + rec) > 0)
    fp = predicted - tp
    tn = total - support - fp
    spec = np.divide(tn, tn + fp, out=np.zeros(3), where=(tn + fp) > 0)
    m = {"accuracy": float(tp.sum() / total), "macro_f1": float(f1.mean()), "macro_recall": float(rec.mean()),
         "macro_precision": float(prec.mean()), "n": int(total), "confusion_matrix": cm.tolist()}
    for i, c in enumerate(CLASS_NAMES):
        m.update({f"recall_{c}": float(rec[i]), f"precision_{c}": float(prec[i]), f"f1_{c}": float(f1[i]),
                  f"specificity_{c}": float(spec[i]), f"fpr_{c}": float(1 - spec[i]), f"support_{c}": int(support[i])})
    return m

# ---------------- the authors' readout (exact settings) ----------------
def soft_class_weight(y, p=0.6):
    classes = np.unique(y)
    base = compute_class_weight(class_weight="balanced", classes=classes, y=y).astype(np.float64)
    return dict(zip(classes.astype(int), (base ** p).astype(float)))

CURVE_EVAL_EPOCHS = {1, 3, 5, 10, 15}      # epochs at which the validation curve is measured (CV runs only)

class Cancelled(Exception):
    """Raised inside a fit when the session's time is up; the run stays pending for the next session."""

def prepare_train(Xtr_fn, ntr, prescale):
    """StandardScaler fitted with batched partial_fit (authors' way); optionally the whole scaled training matrix."""
    bs = 4096
    scaler = StandardScaler(with_mean=True, with_std=True)
    for i in range(0, ntr, bs):
        scaler.partial_fit(Xtr_fn(np.arange(i, min(i + bs, ntr))))
    XS = None
    if prescale:
        XS = np.concatenate([scaler.transform(Xtr_fn(np.arange(i, min(i + bs, ntr)))).astype(np.float32) for i in range(0, ntr, bs)])
    return scaler, XS

def fit_readout(Xtr_fn, ntr, ytr, Xev_fn, nev, loss, rng_seed, yev_for_curve=None, curve_sub=4000, prescale=False, n_jobs=None,
                prepared=None, cancel=None, epochs=None):
    """Authors' readout: StandardScaler (batched partial_fit) + SGDClassifier(hinge|log_loss, l2, alpha=1e-4,
    optimal LR, average=True, random_state=2026, soft class weights ^0.6), 15 epochs, batch 4096,
    shuffling with np.random.default_rng(rng_seed). Batches are scaled on the fly as in the authors' code, or, when
    prescale=True and RAM allows, the whole matrix is scaled once (row-wise identical values, faster).
    prepared=(scaler, XS) reuses a scaler / scaled matrix shared by several fits on the same training rows.
    Xtr_fn(positions) / Xev_fn(positions) return float32 feature rows."""
    bs = 4096
    scaler, XS = prepared if prepared is not None else prepare_train(Xtr_fn, ntr, prescale)
    tf = lambda X: scaler.transform(X).astype(np.float32)
    get_tr = (lambda p: XS[p]) if XS is not None else (lambda p: tf(Xtr_fn(p)))
    clf = SGDClassifier(loss=loss, penalty="l2", alpha=1e-4, fit_intercept=True, learning_rate="optimal",
                        max_iter=1, tol=None, class_weight=soft_class_weight(ytr), average=True, random_state=2026,
                        n_jobs=n_jobs)   # n_jobs only parallelises the 3 one-vs-rest problems: results are bit-identical
    classes = np.unique(ytr)
    rng = np.random.default_rng(rng_seed)
    sub = np.sort(np.random.default_rng(0).choice(ntr, min(curve_sub, ntr), replace=False))
    Xsub = get_tr(sub)
    def predict_eval():
        preds, scores = [], []
        for i in range(0, nev, bs):
            Xe = tf(Xev_fn(np.arange(i, min(i + bs, nev))))
            preds.append(clf.predict(Xe)); scores.append(clf.decision_function(Xe).astype(np.float32))
        return np.concatenate(preds), np.concatenate(scores)
    curve = []
    for ep in range(epochs or READOUT_EPOCHS):          # epochs: only the core search uses fewer
        idx = rng.permutation(ntr)
        for i in range(0, ntr, bs):
            if cancel is not None and cancel.is_set():
                raise Cancelled()
            sel = idx[i:i + bs]
            Xb = get_tr(sel)
            if ep == 0 and i == 0:
                clf.partial_fit(Xb, ytr[sel], classes=classes)
            else:
                clf.partial_fit(Xb, ytr[sel])
        row = {"epoch": ep + 1, "train_sub_macro_f1": compute_metrics(ytr[sub], clf.predict(Xsub))["macro_f1"]}
        if loss == "hinge":
            dec = clf.decision_function(Xsub)
            onehot = -np.ones_like(dec); onehot[np.arange(len(sub)), ytr[sub]] = 1
            row["train_sub_hinge_loss"] = float(np.maximum(0, 1 - onehot * dec).mean())
        if yev_for_curve is not None and (ep + 1) in CURVE_EVAL_EPOCHS:
            row["eval_macro_f1"] = compute_metrics(yev_for_curve, predict_eval()[0])["macro_f1"]
        curve.append(row)
    y_pred, scores = predict_eval()
    return scaler, clf, y_pred, scores, curve

# ---------------- saving a run ----------------
def save_run(rid, spec, group, metrics, eval_idx, eval_pid, y_true, y_pred, scores, model_obj,
             curve, liquid_stats, timing, extra=None):
    d = os.path.join(WORK_DIR, "runs", rid)
    os.makedirs(d, exist_ok=True)
    jdump({"run_id": rid, "group": group, "spec": spec, "env": ENV, "code_version": CODE_VERSION,
           "session": SESSION_TAG, **(extra or {})}, os.path.join(d, "config.json"))
    jdump(metrics, os.path.join(d, "metrics.json"))
    np.savez_compressed(os.path.join(d, "predictions.npz"), eval_index=np.asarray(eval_idx, np.int32),
                        patient_id=np.asarray(eval_pid).astype("U4"), y_true=np.asarray(y_true, np.int8),
                        y_pred=np.asarray(y_pred, np.int8),
                        scores=np.asarray(scores, np.float32) if scores is not None else np.zeros((0, 3), np.float32))
    if model_obj is not None:
        if isinstance(model_obj, dict) and model_obj.get("_torch_state"):
            torch.save(model_obj, os.path.join(d, "model.pt"))
        else:
            joblib.dump(model_obj, os.path.join(d, "readout.joblib"), compress=3)
    if curve:
        pd.DataFrame(curve).to_csv(os.path.join(d, "train_curve.csv"), index=False)
    if liquid_stats:
        jdump(liquid_stats, os.path.join(d, "liquid_stats.json"))
    jdump(timing, os.path.join(d, "timing.json"))
    with open(os.path.join(d, "log.txt"), "w") as f:
        f.write(f"{rid}\nmacro_f1={metrics.get('macro_f1')}\naccuracy={metrics.get('accuracy')}\n"
                f"confusion_matrix={metrics.get('confusion_matrix')}\ntiming={timing}\n")
    disk_add(sum(os.path.getsize(os.path.join(d, f)) for f in os.listdir(d)))


## 5. Data: MIT-BIH → DS1 / DS2 and the authors' folds

In [ ]:
# Data: locate + checksum MIT-BIH, build DS1/DS2 exactly as the authors do (both filters),
# track each beat's position for RR-interval features, verify against the authors' released files.
def locate_input():
    cands = [INPUT_DIR] + sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/100.hea", recursive=True)})
    for c in cands:
        if os.path.exists(os.path.join(c, "100.hea")) and os.path.exists(os.path.join(c, "234.atr")):
            return c
    raise FileNotFoundError(f"MIT-BIH not found at {INPUT_DIR} or anywhere under /kaggle/input")

DATA_DIR = locate_input()
print("MIT-BIH at:", DATA_DIR)

CHECKS = {}
_recs = pd.read_csv(os.path.join(REPO_DIR, "splits/ds1_ds2_records.csv")).set_index("set")["records"]
DS_RECORDS = {"DS1": _recs["DS1"].split(","), "DS2": _recs["DS2"].split(",")}
DS_RECORDS_ALL = DS_RECORDS["DS1"] + DS_RECORDS["DS2"]
sums_path = os.path.join(DATA_DIR, "SHA256SUMS.txt")
if os.path.exists(sums_path):
    bad, missing, n = [], [], 0
    for line in open(sums_path):
        h, fn = line.strip().split(maxsplit=1)
        fp = os.path.join(DATA_DIR, fn)
        if os.path.isfile(fp):
            n += 1
            if hashlib.sha256(open(fp, "rb").read()).hexdigest() != h:
                bad.append(fn)
        else:
            missing.append(fn)
    used = {f"{r}.{x}" for r in DS_RECORDS_ALL for x in ("dat", "hea", "atr")}
    CHECKS["checksums"] = {"files_listed": n + len(missing), "files_checked": n, "mismatches": bad, "missing": missing,
                           "missing_files_used_by_pipeline": sorted(set(missing) & used)}
    print(f"checksums: {n} files checked, {len(bad)} mismatches, {len(missing)} missing"
          + (f" ({', '.join(missing[:12])}{' ...' if len(missing) > 12 else ''})" if missing else ""))
    if set(missing) & used:
        raise FileNotFoundError(f"record files needed by the pipeline are missing: {sorted(set(missing) & used)}")
else:
    CHECKS["checksums"] = {"files_checked": 0, "note": "SHA256SUMS.txt not present"}


NORMAL_S, SVEB_S, VEB_S = ['N', 'L', 'R', 'e', 'j'], ['A', 'a', 'J', 'S'], ['V', 'E']
BEAT_SYMBOLS = set("NLRBAaJSVrFejnE/fQ?")      # all beat annotations (for RR intervals)
RR_NAMES = ["pre", "post", "local10", "rec_median", "pre_over_local", "post_over_local", "pre_over_rec", "pre_over_post"]

def rr_features(all_beats, pos, fs):
    """RR features (seconds / ratios) for beats at sample positions `pos`, from every beat annotation."""
    all_beats = np.sort(np.asarray(all_beats))
    rr = np.diff(all_beats) / fs
    rec_med = float(np.median(rr))
    i = np.searchsorted(all_beats, pos)
    pre = np.where(i > 0, rr[np.clip(i - 1, 0, len(rr) - 1)], rec_med)
    post = np.where(i < len(all_beats) - 1, rr[np.clip(i, 0, len(rr) - 1)], rec_med)
    csum = np.concatenate([[0], np.cumsum(rr)])
    lo = np.clip(i - 10, 0, None)
    cnt = np.maximum(i - lo, 1)
    local = np.where(i > 0, (csum[np.clip(i, 0, len(rr))] - csum[lo]) / cnt, rec_med)
    f = np.stack([pre, post, local, np.full_like(pre, rec_med), pre / local, post / local, pre / rec_med, pre / post], 1)
    return f.astype(np.float32)

def rr_mean_ratio(all_beats, pos, fs):
    """pre-RR / mean RR of the whole recording (Huang et al. 2014); the same pre-RR as the "pre" column."""
    rr = np.diff(np.sort(np.asarray(all_beats))) / fs
    return (rr_features(all_beats, pos, fs)[:, 0] / rr.mean()).astype(np.float32)

def huang_windows(sig, pos, half=100):
    """Huang et al.: 200 raw (unfiltered) samples, R-100 .. R+99 at 360 Hz, minus their mean, divided by their SD."""
    W = np.stack([sig[p - half:p + half] for p in pos]).astype(np.float64)
    return ((W - W.mean(1, keepdims=True)) / W.std(1, keepdims=True)).astype(np.float32)

def build_set(records, filt):
    Xs, syms, pids, poss, rrs = [], [], [], [], []
    rrms, hws = [], []          # extensions: pre-RR / record mean RR, Huang windows
    for rec in records:
        sig, atr_sym, atr_sample, fs = A03.load_ecg(os.path.join(DATA_DIR, rec), target_fs=360)
        if filt == "median":
            sig_f, _ = A_sig.median_filters(sig, fs=fs, kernel_size_ms1=200, kernel_size_ms2=600, mode='reflect')
        else:   # the authors' (commented-out) morphological line
            _, sig_f, _, _ = A_sig.morph_filter_baseline(sig, fs=360, k1_ratio=0.20, k2_ratio=0.35, k3_ratio=0.005, mode='reflect')
        df_all = pd.DataFrame({'atr_sym': atr_sym, 'atr_sample': atr_sample})
        df = df_all[df_all.atr_sym.isin(NORMAL_S + SVEB_S + VEB_S)].reset_index(drop=True)
        Xp, _, sp, pp = A03.build_XY(sig_f, df, 0.8, 0.8, fs, NORMAL_S, SVEB_S + VEB_S, patient_id=rec)
        L = int(round(0.8 * fs))
        keep = ((df.atr_sample - L) >= 0) & ((df.atr_sample + L + 1) <= len(sig_f))
        pos = df.atr_sample[keep].to_numpy()
        assert len(pos) == len(Xp) and list(df.atr_sym[keep]) == list(sp)
        beats_all = [s for s, y in zip(atr_sample, atr_sym) if y in BEAT_SYMBOLS]
        Xs.append(Xp); syms += list(sp); pids += list(pp); poss.append(pos); rrs.append(rr_features(beats_all, pos, fs))
        rrms.append(rr_mean_ratio(beats_all, pos, fs)); hws.append(huang_windows(sig, pos))
    X = np.vstack(Xs); pos = np.concatenate(poss); rr = np.vstack(rrs); pid = np.array(pids)
    num = A03.Numerical_Labels(syms)
    X_sh, y_sh, pid_sh = A03.Shuffle_data(X, num, pid, seed=42)
    np.random.seed(42); perm = np.random.permutation(len(X))           # the same permutation Shuffle_data used
    assert np.array_equal(X[perm], X_sh)
    with contextlib.redirect_stdout(io.StringIO()):     # silence the authors' "taking all of them" prints
        Xd, yd, pidd = A_strat.Make_Data_Stratified(X_sh, y_sh, pid_sh, num_data=50000, Type_Class='SuperC')
        idx_d, _, _ = A_strat.Make_Data_Stratified(np.arange(len(X))[:, None], y_sh, pid_sh, num_data=50000, Type_Class='SuperC')
    src = perm[np.asarray(idx_d)[:, 0].astype(int)]
    assert np.array_equal(X[src], Xd), "beat-order tracking failed"
    return {"X": np.asarray(Xd, np.float64), "y": np.asarray(yd, int), "pid": np.asarray(pidd).astype(str),
            "pos": pos[src], "rr": rr[src], "sym": np.array(syms)[src],
            "rr_mean_ratio": np.concatenate(rrms)[src], "huang_beats": np.vstack(hws)[src]}

t0 = time.time()
DATA = {f: {ds: build_set(DS_RECORDS[ds], f) for ds in ("DS1", "DS2")} for f in ("morph", "median")}
print(f"datasets built in {time.time()-t0:.0f}s")
for ds in ("DS1", "DS2"):
    assert np.array_equal(DATA["morph"][ds]["y"], DATA["median"][ds]["y"])
    assert np.array_equal(DATA["morph"][ds]["pid"], DATA["median"][ds]["pid"])

# verify against the authors' released DS1 order and counts
_auth = pd.read_csv(os.path.join(REPO_DIR, "splits/patient_ids_train.csv"))
CHECKS["ds1_matches_authors_order"] = bool(np.array_equal(_auth.patient_id.astype(str).values, DATA["morph"]["DS1"]["pid"])
                                           and np.array_equal(_auth.label.values, DATA["morph"]["DS1"]["y"]))
CHECKS["class_counts"] = {ds: np.bincount(DATA["morph"][ds]["y"], minlength=3).tolist() for ds in ("DS1", "DS2")}
CHECKS["expected_counts"] = {"DS1": [45827, 943, 3787], "DS2": [44216, 1837, 3219]}
print("DS1 order identical to authors' file:", CHECKS["ds1_matches_authors_order"], "| counts:", CHECKS["class_counts"])

# folds: the authors' exact indices, and a check that they regenerate
_fd = json.load(open(os.path.join(REPO_DIR, "splits/fold_definitions.json")))
FOLDS = [{"rep": f["rep"], "fold": f["fold"], "train_idx": np.array(f["train_idx"]), "val_idx": np.array(f["val_idx"])}
         for f in _fd["folds"]]
_ok, _k = True, 0
_y1, _p1 = DATA["morph"]["DS1"]["y"], DATA["morph"]["DS1"]["pid"]
for rep, sd in enumerate([11, 23, 37], 1):
    for fold, (tr, va) in enumerate(StratifiedGroupKFold(4, shuffle=True, random_state=sd).split(np.zeros(len(_y1)), _y1, _p1), 1):
        f = FOLDS[_k]; _k += 1
        _ok &= (f["rep"], f["fold"]) == (rep, fold) and np.array_equal(np.sort(f["train_idx"]), tr) and np.array_equal(np.sort(f["val_idx"]), va)
CHECKS["folds_regenerate_exactly"] = bool(_ok)
print("authors' fold indices regenerate exactly:", _ok)

# smoke test: keep a few beats per record and class, remap folds
if SMOKE_TEST:
    def _keep(D):
        keep = []
        for p in np.unique(D["pid"]):
            for c, k in SMOKE_PER_CLASS.items():
                keep += list(np.where((D["pid"] == p) & (D["y"] == c))[0][:k])
        return np.sort(np.array(keep))
    _k1, _k2 = _keep(DATA["morph"]["DS1"]), _keep(DATA["morph"]["DS2"])
    for f in DATA:
        for ds, kk in (("DS1", _k1), ("DS2", _k2)):
            DATA[f][ds] = {k: v[kk] for k, v in DATA[f][ds].items()}
    _remap_arr = -np.ones(int(_k1.max()) + 1, int); _remap_arr[_k1] = np.arange(len(_k1))
    def _remap(idx):
        r = _remap_arr[idx[idx <= _k1.max()]]
        return r[r >= 0]
    FOLDS = [{"rep": f["rep"], "fold": f["fold"], "train_idx": _remap(f["train_idx"]), "val_idx": _remap(f["val_idx"])}
             for f in FOLDS]
    print("SMOKE subset:", {ds: np.bincount(DATA['morph'][ds]['y'], minlength=3).tolist() for ds in ("DS1", "DS2")})

CV_SPLITS = [(f["rep"], f["fold"]) for f in FOLDS]
if SMOKE_TEST:
    CV_SPLITS = SMOKE_CV_SPLITS
FOLD_BY = {(f["rep"], f["fold"]): f for f in FOLDS}

# ---------------- normalization (two paths, as the authors have them) ----------------
def normalize_cv(X, tr_idx, norm):
    """CV path (script 03 get_fold_data + float32 cast before encoding)."""
    if norm == "standard":
        return np.asarray(A_strat.minmax_per_beat(X), np.float32), {}
    mn, mx = A_strat.robust_minmax_fit(X[tr_idx], low=0.1, high=99.9)
    return np.asarray(A_strat.robust_minmax_apply(X, mn, mx), np.float32), {"p_low": mn, "p_high": mx}

def normalize_final(X1, X2, norm):
    """Final DS1 -> DS2 path (script 01): float64, global DS1 percentiles for robust."""
    if norm == "standard":
        return A_strat.Rowwise_Normalization(X1), A_strat.Rowwise_Normalization(X2), {}
    lo, hi = np.percentile(X1, [0.1, 99.9])
    return A_strat.minmax_apply(X1, lo, hi), A_strat.minmax_apply(X2, lo, hi), {"p_low": float(lo), "p_high": float(hi)}


## 6. Spike encoders

In [ ]:
# Encoders (the authors' functions, device passed explicitly), RR spike channels, A_in measurement.
PAPER_AIN = {"multi": 0.05892, "rate": 20.8245, "latency": 0.9473}   # values hard-coded by the authors

def multithreshold_t(Xb, num_levels=200):
    """GPU version of the authors' MultiThreshold (detect_edges='both', include_endpoints=True,
    separate_polarity=False). Same thresholds (np.linspace in the data's dtype) and the same comparisons:
    rise = x_t >= thr & x_{t-1} < thr, fall = x_t < thr & x_{t-1} >= thr, spike = rise | fall."""
    X = np.ascontiguousarray(np.asarray(Xb))
    thr = torch.from_numpy(np.linspace(0.0, 1.0, num_levels, dtype=X.dtype)).to(DEVICE)
    x = torch.from_numpy(X).to(DEVICE)
    out = torch.zeros((X.shape[0], X.shape[1], num_levels), dtype=torch.uint8, device=DEVICE)
    above = x[:, :, None] >= thr                     # (B, T, K)
    out[:, 1:] = (above[:, 1:] != above[:, :-1]).to(torch.uint8)
    return out

def roundrobin_t(counts, bin_pop=10):
    """GPU version of the authors' counts_to_binary_population_roundrobin. Their loop keeps a pointer p per
    (beat, channel), sets sub-channels (p + j) % bin_pop for j < min(k, bin_pop), then p = (p + k) % bin_pop.
    So p_t = (counts before t) % bin_pop, and sub-channel b fires iff (b - p_t) % bin_pop < min(k_t, bin_pop)."""
    c = torch.as_tensor(np.asarray(counts).astype(np.int64), device=DEVICE)          # (B, L, C)
    p = torch.remainder(torch.cumsum(c, dim=1) - c, bin_pop)
    keff = torch.clamp(c, max=bin_pop)
    b = torch.arange(bin_pop, device=DEVICE)
    rel = torch.remainder(b[None, None, None, :] - p[..., None], bin_pop)
    out = (rel < keff[..., None]).to(torch.uint8)
    return out.reshape(c.shape[0], c.shape[1], c.shape[2] * bin_pop)

def encode_t(Xb, enc, num_levels=200):
    """(B, 577) normalized beats -> (B, T_in, C) uint8 spikes as a tensor on DEVICE."""
    if enc == "multi":
        return multithreshold_t(Xb, num_levels)
    if enc == "rate":
        s = A_enc.TemporalRateCoding(Xb, None, gain=1.0, offset=0.0, pop_size=64, batch_size=1024, device=DEVICE,
                                     out_dir=None, spike_filename=None, labels_filename=None, save_to_disk=False)
        return torch.from_numpy(np.asarray(s, dtype=np.uint8)).to(DEVICE)
    if enc == "latency":
        cnt = A_enc.BoundedDelayLatencyCoding(Xb, Dmax=64, pop_size=1, batch_size=1024, device=DEVICE)
        return roundrobin_t(cnt, bin_pop=10)
    raise ValueError(enc)

def encode(Xb, enc, num_levels=200):
    """Same as encode_t, returned as a NumPy array."""
    return encode_t(Xb, enc, num_levels).cpu().numpy()

N_IN = {"multi": 200, "rate": 64, "latency": 10}

# RR timing as extra input channels: thermometer code of 3 ratio features, 10 levels each.
RR_LIQ_FEATS = [RR_NAMES.index(n) for n in ("pre_over_local", "post_over_local", "pre_over_rec")]
RR_LEVELS, RR_PERIOD = 10, 5            # an active channel spikes every 5th step

def rr_liq_feats(rr_rows):
    """Columns turned into RR channels: the 3 ratios; with the 9-column array of rr_inputs(..., True) also the
    9th column, pre-RR / recording mean RR (40 channels instead of 30)."""
    return RR_LIQ_FEATS + ([len(RR_NAMES)] if rr_rows.shape[1] > len(RR_NAMES) else [])

def rr_thresholds(rr_train):
    q = np.linspace(0.05, 0.95, RR_LEVELS)
    return np.stack([np.quantile(rr_train[:, j], q) for j in rr_liq_feats(rr_train)])     # (3 or 4, 10)

def rr_channels(rr_rows, thr, T):
    active = (rr_rows[:, rr_liq_feats(rr_rows)][:, :, None] >= thr[None]).reshape(len(rr_rows), -1)   # (B, 30 or 40)
    pulse = (np.arange(T) % RR_PERIOD == 0).astype(np.uint8)
    return (pulse[None, :, None] * active[:, None, :]).astype(np.uint8)                   # (B, T, 30)

def rr_window_counts(rr_rows, thr, T):
    """rr_channels(rr_rows, thr, T) summed over the 32 windows of 20 steps (steps >= T are silent), (B, 32, 30) uint8."""
    active = (rr_rows[:, rr_liq_feats(rr_rows)][:, :, None] >= thr[None]).reshape(len(rr_rows), -1)
    steps = np.arange(NWIN * WIN)
    per_window = ((steps % RR_PERIOD == 0) & (steps < T)).reshape(NWIN, WIN).sum(1)
    return (per_window[None, :, None] * active[:, None, :]).astype(np.uint8)

def measure_ain(Xn, rows, enc, num_levels=200, rr=None, rr_thr=None, n=5000):
    """Average number of active input spikes per time step (summed over channels), over a random
    subset of training beats. Same quantity as the authors' estimate_K_spikes_per_timestep."""
    rows = np.asarray(rows)
    torch.manual_seed(12345)           # the Rate encoder is stochastic: make A_in independent of job order
    pick = np.sort(np.random.default_rng(0).choice(rows, min(n, len(rows)), replace=False))
    tot, cnt = 0.0, 0
    for i in range(0, len(pick), 1000):
        r = pick[i:i + 1000]
        S = encode(Xn[r], enc, num_levels)
        if rr is not None:
            S = np.concatenate([S, rr_channels(rr[r], rr_thr, S.shape[1])], axis=2)
        tot += float(S.sum(dtype=np.int64)); cnt += S.shape[0] * S.shape[1]
    return tot / cnt


## 7. The paper's liquid class (for X)

In [ ]:
# The liquid: built with the authors' class; modes patch the two bugs. GPU simulator with diagnostics.
LIQ = dict(P_in=0.1, betaE=0.95, betaI=0.8, Uthr=1.0, rho=0.9, reservoir_dim=(7, 7, 7), excit_ratio=0.8,
           dt=1.0, num_sim_steps=650, C_vals={"exc_exc": 0.3, "exc_inh": 0.2, "inh_exc": 0.4, "inh_inh": 0.1},
           lambda_val=0.8, scl_v=1, mu_rsv=[0.020, 0.025, 0.050, 0.040], g_rec=1, tau_syn=0.01)
I_TARGET = LIQ["rho"] * (1 - LIQ["betaE"]) * LIQ["Uthr"]
MODES = {   # gamma_fix: Gamma(shape, scale) as intended; ain: "paper" = hard-coded value, "measured" = Eq. as described
    "published": {"gamma_fix": False, "ain": "paper"},
    "fixed":     {"gamma_fix": True,  "ain": "measured"},
    "fix_gamma": {"gamma_fix": True,  "ain": "paper"},
    "fix_ain":   {"gamma_fix": False, "ain": "measured"},
}
_TorchGamma = torch.distributions.Gamma

def build_liquid(input_size, mu_in, seed, gamma_fix, lambda_val=0.8, recurrence=True, g_rec=None):
    """seed_everything(seed) then the authors' constructor, exactly as in their scripts."""
    seed_everything(seed, deterministic=False)
    if gamma_fix:
        torch.distributions.Gamma = lambda conc, scale: _TorchGamma(conc, 1.0 / scale)   # (shape, scale) as intended
    try:
        model = LSM3D(input_size, LIQ["P_in"], mu_in, LIQ["betaE"], LIQ["betaI"], LIQ["reservoir_dim"],
                      LIQ["excit_ratio"], LIQ["dt"], LIQ["num_sim_steps"], LIQ["C_vals"], lambda_val, LIQ["scl_v"],
                      np.array(LIQ["mu_rsv"]), LIQ["g_rec"] if g_rec is None else g_rec, LIQ["tau_syn"], DEVICE).to(DEVICE)
    finally:
        torch.distributions.Gamma = _TorchGamma
    if not recurrence:
        model.rsv_weights = torch.zeros_like(model.rsv_weights)
    return model

def structure_stats(model):
    W = model.rsv_weights.detach().cpu().numpy(); Win = model.input_rsv_weights.detach().cpu().numpy()
    E = model.is_excit.cpu().numpy()
    out = {"n_recurrent_synapses": int((W != 0).sum()), "mean_recurrent_in_degree": float((W != 0).sum(0).mean()),
           "neurons_without_recurrent_input": int(((W != 0).sum(0) == 0).sum()),
           "n_input_synapses": int((Win != 0).sum()), "mean_input_in_degree": float((Win != 0).sum(0).mean()),
           "mean_nonzero_input_weight": float(Win[Win != 0].mean()) if (Win != 0).any() else 0.0,
           "threshold": LIQ["Uthr"], "n_excitatory": int(E.sum())}
    for name, pre, post in (("EE", E, E), ("EI", E, ~E), ("IE", ~E, E), ("II", ~E, ~E)):
        blk = W[np.ix_(pre, post)]
        nz = blk[blk != 0]
        out[f"{name}_synapses"] = int(nz.size)
        out[f"{name}_mean_abs_weight"] = float(np.abs(nz).mean()) if nz.size else 0.0
    return out

def save_reservoir(model, meta):
    rid = stable_hash(meta, 12)
    path = os.path.join(WORK_DIR, "reservoirs", f"{rid}.pt")
    if not os.path.exists(path):
        Win = model.input_rsv_weights.detach().cpu(); W = model.rsv_weights.detach().cpu()
        iw, ir = Win.nonzero().T, W.nonzero().T
        torch.save({"meta": meta, "input_size": Win.shape[0], "n_neurons": W.shape[0],
                    "win_index": iw.to(torch.int16), "win_value": Win[iw[0], iw[1]].float(),
                    "wrec_index": ir.to(torch.int16), "wrec_value": W[ir[0], ir[1]].float(),
                    "is_excit": model.is_excit.cpu(), "beta": torch.where(model.is_excit.cpu(), LIQ["betaE"], LIQ["betaI"]).float(),
                    "structure": structure_stats(model), "torch": torch.__version__, "device": str(DEVICE)}, path)
        disk_add(os.path.getsize(path))
    return rid

WIN, NWIN, N_NEUR = 20, 32, 343          # fr_windows(win=20, stride=20) over 650 steps -> 32 windows

def simulate(model, Xn, enc, batch, num_levels=200, rr=None, rr_thr=None, y=None, enc_seed=0):
    """Encode + run the liquid exactly as the authors' loop does. Returns spike counts per (window, neuron)
    as uint8 (N, 32*343) in the authors' feature order (window-major), plus liquid diagnostics."""
    torch.manual_seed(enc_seed)
    N = len(Xn)
    out = np.empty((N, NWIN * N_NEUR), np.uint8)
    Win, Wr = model.input_rsv_weights, model.rsv_weights
    neur = torch.zeros(N_NEUR, dtype=torch.float64, device=DEVICE)
    per_beat = np.empty(N, np.float32)
    acc = {"in_spikes": 0.0, "in_steps": 0, "Iin_sum": 0.0, "Iin_steps": 0, "Irec_abs_sum": 0.0}
    t_enc = t_sim = 0.0
    with torch.no_grad():
        for s in range(0, N, batch):
            t0 = time.time()
            x = encode_t(Xn[s:s + batch], enc, num_levels)
            if rr is not None:
                x = torch.cat([x, torch.from_numpy(rr_channels(rr[s:s + batch], rr_thr, x.shape[1])).to(DEVICE)], dim=2)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t_enc += time.time() - t0; t0 = time.time()
            B, Tin, C = x.shape
            spk = torch.zeros(B, N_NEUR, device=DEVICE); mem = torch.zeros(B, N_NEUR, device=DEVICE)
            zero_in = torch.zeros(B, C, device=DEVICE)
            cnt = torch.zeros(B, NWIN, N_NEUR, dtype=torch.int16, device=DEVICE)
            Iin_acc = torch.zeros((), dtype=torch.float64, device=DEVICE); Irec_acc = torch.zeros((), dtype=torch.float64, device=DEVICE)
            for t in range(LIQ["num_sim_steps"]):
                x_t = x[:, t, :].float() if t < Tin else zero_in
                I_in = x_t @ Win
                I_rec = spk @ Wr
                I = (I_in + I_rec) * model.dt
                spk, mem = model.lif(I, mem)
                if t < NWIN * WIN:
                    cnt[:, t // WIN] += spk.to(torch.int16)
                if t < Tin:
                    Iin_acc += I_in.mean(dtype=torch.float64)
                Irec_acc += I_rec.abs().mean(dtype=torch.float64)
            c = cnt.reshape(B, -1)
            out[s:s + B] = c.to(torch.uint8).cpu().numpy()
            neur += cnt.sum((0, 1), dtype=torch.float64)
            per_beat[s:s + B] = c.sum(1).float().cpu().numpy()
            acc["in_spikes"] += float(x.sum(dtype=torch.int64)); acc["in_steps"] += B * Tin
            acc["Iin_sum"] += float(Iin_acc) * B; acc["Iin_steps"] += Tin * B
            acc["Irec_abs_sum"] += float(Irec_acc) * B
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t_sim += time.time() - t0
            del x, cnt, spk, mem
    neur = neur.cpu().numpy()
    steps = NWIN * WIN
    E = model.is_excit.cpu().numpy()
    stats = {"liquid_firing_prob_per_step": float(neur.sum() / (N * steps * N_NEUR)),
             "excitatory_firing_prob": float(neur[E].sum() / (N * steps * E.sum())),
             "inhibitory_firing_prob": float(neur[~E].sum() / (N * steps * (~E).sum())),
             "silent_neurons": int((neur == 0).sum()), "silent_fraction": float((neur == 0).mean()),
             "liquid_spikes_per_beat_mean": float(per_beat.mean()),
             "input_spikes_per_step": acc["in_spikes"] / max(acc["in_steps"], 1),
             "mean_input_current_per_neuron_step": acc["Iin_sum"] / max(acc["Iin_steps"], 1),
             "input_current_over_target": (acc["Iin_sum"] / max(acc["Iin_steps"], 1)) / I_TARGET,
             "mean_abs_recurrent_current": acc["Irec_abs_sum"] / max(N * LIQ["num_sim_steps"], 1),
             "n_beats": N, "encode_seconds": t_enc, "simulate_seconds": t_sim}
    if y is not None:
        for c, nm in enumerate(CLASS_NAMES):
            if (y == c).any():
                stats[f"liquid_spikes_per_beat_{nm}"] = float(per_beat[y == c].mean())
    return out, stats

def liquid_raster(model, S, batch=256):
    """Full spike raster (B, 650, 343) with the same update as simulate(), on the device. Also returns the mean
    |recurrent current| and mean input current per neuron and step (input steps only)."""
    Win, Wr = model.input_rsv_weights, model.rsv_weights
    T = LIQ["num_sim_steps"]; out = np.zeros((len(S), T, N_NEUR), bool); irec = iin = 0.0; tin = 1
    with torch.no_grad():
        for s in range(0, len(S), batch):
            x = torch.from_numpy(np.ascontiguousarray(S[s:s + batch])).to(DEVICE).float()
            B, tin, Cin = x.shape
            spk = torch.zeros(B, N_NEUR, device=DEVICE); mem = torch.zeros(B, N_NEUR, device=DEVICE)
            zero_in = torch.zeros(B, Cin, device=DEVICE)
            r = torch.zeros(B, T, N_NEUR, dtype=torch.bool, device=DEVICE)
            for t in range(T):
                I_in = (x[:, t, :] if t < tin else zero_in) @ Win
                I_rec = spk @ Wr
                spk, mem = model.lif((I_in + I_rec) * model.dt, mem)
                r[:, t] = spk > 0
                irec += float(I_rec.abs().mean()) * B
                if t < tin:
                    iin += float(I_in.mean()) * B
            out[s:s + B] = r.cpu().numpy()
    return out, irec / (len(S) * T), iin / (len(S) * tin)

def counts_to_rates(counts_rows):
    """Authors' features: window mean of 0/1 spikes in float64, stored float32."""
    return (counts_rows.astype(np.float64) / WIN).astype(np.float32)


## 8. Build check: Y on Maass's own input

In [ ]:
# Stages "tests" and "build": Y on Maass's own input. No ECG, no labels.
# Y is Maass's liquid as we read (c): the 15 x 3 x 3 cube (135 neurons), lambda 2, (c)'s synapse types, dynamic
# synapses, 30 ms membranes, background at 0.9 of threshold, input to 30% of the neurons at (c)'s size (0.12 / 0.06).
# Y-off is the same liquid with every recurrent synapse removed (same neurons, same input wiring, same input).
import urllib.request
from lsm2 import config as LC, liquid as LL, sim as LS, measure as LMe, gain as LGa, data as LDa, calib as LCa

Y_DIR = os.path.join(WORK_DIR, "ytest")
for _d in ("", "states", "figures"):
    os.makedirs(os.path.join(Y_DIR, _d), exist_ok=True)
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#8a8a8a"]       # the replication's chart colours (+ grey)
MAASS_SCRIPT_SHA256 = "81f50fec6001c76b16ba548f2e03e915de9a9799c554ca33aa0598e8bb47ac4e"   # the copy the tests were written for

# The gain is chosen by (e)'s rank measure only. The old plan's firing band (2-40 Hz) is off: (e) found firing rates
# poorly correlated with performance, so rates are reported, not used to choose.
LC.BAND_HZ = (0.0, float("inf"))
LC.HIGH_FRAC = 1.0
SEGMENT_MS, SEGMENTS = 250.0, 4          # (c), Figures 4-5: four 250 ms segments
SEGMENT_TRAIN, SEGMENT_TEST = 1000, 500  # (c): 1000 training and 500 test inputs
SEGMENT_RATE_HZ = 20.0                   # (c): each template is a 20 Hz Poisson train

if SMOKE_TEST:     # a logic test of every step on a few inputs and a coarse grid; the numbers mean nothing
    LC.N_CAL_BEATS, LC.GEN_COPIES, LC.N_SUB_REFERENCE = 24, 6, 4
    LC.CODE.update(twin_beats=4, memory_sequences=16)
    LC.G_COARSE, LC.G_LOW, LC.G_FINE_K = (1, 10, 100), (0.3,), 1
    SEGMENT_TRAIN, SEGMENT_TEST = 40, 20
    Y_SEEDS, X_SEEDS = (1, 2), (2026, 69)

Y_CTX = LMe.Ctx(DEVICE, torch.float32, NEW_SIM_BATCH, log)


def y_path(step):
    return os.path.join(Y_DIR, f"{step}.json")

def y_done(step):
    return os.path.exists(y_path(step))

def y_save(step, obj):
    jdump(obj, y_path(step))

def y_load(step):
    return json.load(open(y_path(step)))

def y_gate(ok, msg):
    """A stop point. In the smoke test the numbers mean nothing, so it only warns."""
    if ok:
        return
    if SMOKE_TEST:
        log("SMOKE TEST: would stop here: " + msg)
        return
    raise RuntimeError(msg)


def y_cube(seed, n_channels=1):
    """Y for one seed. Its recurrent structure depends only on the seed (the input wiring has its own random
    streams), so the Poisson version and the ECG version of a seed are the same liquid."""
    liq = LL.build_candidate(LC.ALL_FACTORS, seed, LC.N_SUB_REFERENCE, n_channels)
    liq.meta["seed"] = int(seed)
    return liq


def maass_script_path():
    if MAASS_SCRIPT_PATH:
        p = os.path.expanduser(MAASS_SCRIPT_PATH)
    else:
        p = os.path.join(SCRATCH_DIR, "maass2002.py")
        if not os.path.exists(p):
            urllib.request.urlretrieve(MAASS_SCRIPT_URL, p)
    h = hashlib.sha256(open(p, "rb").read()).hexdigest()
    if h != MAASS_SCRIPT_SHA256:
        log(f"WARNING: (d)'s script differs from the tested copy (sha256 {h[:12]} vs {MAASS_SCRIPT_SHA256[:12]})")
    return p


def sweep_rows(sw):
    """The gain sweep as a list of rows (JSON-safe)."""
    rows = []
    for g in sorted(sw["table"]):
        t = sw["table"][g]
        rows.append({"g": float(g), "abs_g": t["abs_g"], "rate_hz": t["rate_hz"], "sep": t["sep"], "gen": t["gen"],
                     "gap": t["gap"], "gap_seeds": t["gap_seeds"], "rate_seeds": t["rate_seeds"],
                     "twin_ratio": t.get("twin_ratio"), "gap_plain": t.get("gap_plain"), "stage": t["stage"]})
    return {"rows": rows, "chosen_g": sw["chosen_g"], "best_g": sw["best_g"], "peak_at_edge": sw["peak_at_edge"],
            "gap_peak_inside_grid": sw["gap_peak_inside_grid"], "extensions": sw["extensions"]}


def run_sweep(liqs, sets, label):
    """(e)'s measure at every g of the grid (old plan's grid rule), on every seed. Returns the sweep record."""
    def measure(g, s):
        r = LMe.measure_at_g(Y_CTX, liqs[s], g, liqs[s].in_scale_fixed, sets, s, extra_ranks=True)
        r["seed"] = s
        return r
    log(f"  gain sweep on {label}, seeds {list(liqs)}")
    return LGa.sweep(measure, list(liqs), log=log)


# ---------------------------------------------------------------- step: tests
def y_tests():
    try:
        import pytest   # noqa: F401
    except ImportError:
        return {"pass": None, "note": "pytest not installed: the tests were run by the build, not on this machine"}
    tdir = os.path.join(SCRATCH_DIR, "lsm2_tests")
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", tdir], capture_output=True, text=True,
                       env={**os.environ, "PYTHONPATH": SCRATCH_DIR, "MAASS_SCRIPT": maass_script_path()})
    return {"pass": r.returncode == 0, "tail": r.stdout[-2000:]}


# ---------------------------------------------------------------- step: build (Maass's own input)
def segment_inputs(seed, n):
    """(c)'s Figures 4-5 task: four 250 ms segments, each one of two templates (20 Hz Poisson), the whole train
    jittered by 4 ms (the 4 ms jitter of the plan's generalisation set). Labels: which template, per segment."""
    rng = np.random.default_rng([seed, 71])
    seg = int(round(SEGMENT_MS / LC.SAMPLE_MS))
    tmpl = rng.poisson(SEGMENT_RATE_HZ * LC.SAMPLE_MS / 1000.0, (SEGMENTS, 2, seg, 1)).astype(np.uint8)
    lab = rng.integers(0, 2, (n, SEGMENTS))
    X = np.zeros((n, SEGMENTS * seg, 1), np.uint8)
    for i in range(n):
        base = np.concatenate([tmpl[k, lab[i, k]] for k in range(SEGMENTS)])
        X[i] = LMe.jitter_spikes(base, np.random.default_rng([seed, 72, i]))
    return X, lab


def segment_task(liq, g, seed, recurrence):
    """Accuracy of a linear readout (least squares to +-1, as (c)) of the 30 ms-filtered state at the end of the
    1 s input, for each segment (oldest first). Trained on the first 1000 inputs, scored on the next 500."""
    n_tr, n_te = SEGMENT_TRAIN, SEGMENT_TEST
    X, lab = segment_inputs(seed, n_tr + n_te)
    T = X.shape[1]
    u0 = LS.start_voltages(liq, seed, range(len(X)), 4)
    o = LMe.simulate(Y_CTX, liq, g, liq.in_scale_fixed, X, u0, T, {"states_at": [T - 1], "neuron_counts": (0, T)},
                     recurrence=recurrence)
    Z = o["states_at"][:, 0].astype(np.float64)
    Zb = np.hstack([Z, np.ones((len(Z), 1))])
    acc = []
    for k in range(SEGMENTS):
        y = 2.0 * lab[:, k] - 1
        w = np.linalg.lstsq(Zb[:n_tr], y[:n_tr], rcond=None)[0]
        acc.append(float(((Zb[n_tr:] @ w >= 0) == (lab[n_tr:, k] == 1)).mean()))
    rate = float(o["neuron_counts"].sum() / (len(X) * liq.N * T * LC.SAMPLE_MS / 1000.0))
    return {"accuracy": acc, "rate_hz": rate}


def y_build():
    data = LDa.make_maass_data(seed=1)
    liqs = {s: y_cube(s) for s in Y_SEEDS}
    sw = run_sweep(liqs, data.sets, "Maass's 20 Hz Poisson input")
    own = {s: float(liqs[s].g_maass) for s in Y_SEEDS}
    out = {"sweep": sweep_rows(sw), "g_maass": own,
           "abs_g_maass": {s: liqs[s].ee_mean_push(own[s]) for s in Y_SEEDS},
           "input_neuron_rate_hz": LCa.input_neuron_rate(Y_CTX, list(liqs.values()), liqs[Y_SEEDS[0]].in_scale_fixed,
                                                         data.sets["sep"])}
    settings = {"maass_scale": own}
    if sw["chosen_g"] is not None:
        settings["rule"] = {s: float(sw["chosen_g"]) for s in Y_SEEDS}
    seg = {}
    for name, gs in settings.items():
        seg[name] = {}
        for s in Y_SEEDS:
            on = segment_task(liqs[s], gs[s], s, True)
            off = segment_task(liqs[s], gs[s], s, False)
            older_on, older_off = np.mean(on["accuracy"][:-1]), np.mean(off["accuracy"][:-1])
            seg[name][s] = {"g": gs[s], "on": on, "off": off, "older_on": float(older_on), "older_off": float(older_off),
                            "Y_beats_Y_off": bool(older_on > older_off)}
            log(f"    segment task, {name} (g = {gs[s]:.3g}), seed {s}: Y {np.round(on['accuracy'], 3).tolist()} "
                f"({on['rate_hz']:.2f} Hz), Y-off {np.round(off['accuracy'], 3).tolist()} ({off['rate_hz']:.2f} Hz)")
    out["segment_task"] = seg
    # the stop point: at Maass's own scale, Y keeps the three older segments better than Y-off, on every seed
    out["pass"] = all(seg["maass_scale"][s]["Y_beats_Y_off"] for s in Y_SEEDS)
    return out


# ---------------------------------------------------------------- run the stages of this cell
if "tests" in Y_STAGES:
    if not y_done("tests"):
        y_save("tests", y_tests())
    _t = y_load("tests")
    log(f"tests: {_t}")
    y_gate(_t["pass"] is not False, "the lsm2 tests failed (see WORK_DIR/ytest/tests.json)")

if "build" in Y_STAGES:
    if not y_done("build"):
        y_save("build", y_build())
    Y_BUILD = y_load("build")
    log(f"build: Maass's own scale g = {Y_BUILD['g_maass']}; (e)'s rule on Maass's input g = {Y_BUILD['sweep']['chosen_g']}; "
        f"segment task at Maass's scale, Y beats Y-off on every seed: {Y_BUILD['pass']}")
    y_gate(Y_BUILD["pass"], "Y does not keep (c)'s older segments better than Y-off at Maass's own scale: "
                            "the build does not behave like Maass's liquid (see WORK_DIR/ytest/build.json)")


## 9. Y on ECG beats: input size and gain (no labels)

In [ ]:
# Stage "ecg_setup": Y on ECG beats, labels unused. The front end is config 11's ((a)'s selected configuration:
# morphological filter, robust normalisation, Multi-threshold encoder, 200 channels).
# Input size (Y_INPUT_RULE): "match_maass_rate" scales the input so Y's input-receiving neurons fire, with recurrence
# off, at the rate they fire on Maass's own input (the old plan's factor-5 rule: Maass's operating point). The
# encoded ECG brings many more input spikes per second than (c)'s single 20 Hz train, so (c)'s per-spike size
# ("maass_per_spike", 0.12 / 0.06) would drive those neurons far harder than in (c). Both rates are reported.
# Then the recurrent gain is chosen by (e)'s rank measure on these beats.
Y_FILT, Y_NORM, Y_ENC = "morph", "robust", "multi"
Y_N_CH = 200
Y_START_STREAM = 101                      # per-beat starting voltages from (seed, stream, beat index), as stage 2 had

def y_ds1_normalised():
    """All DS1 beats through config 11's filter and robust normalisation. The 0.1 / 99.9 percentiles are fitted on
    all of DS1 (no labels; DS2 is not used). In the authors' CV they are fitted on each fold's training beats: one
    fit for all folds here means each liquid is simulated once instead of once per fold (same for every liquid)."""
    D = DATA[Y_FILT]["DS1"]
    mn, mx = A_strat.robust_minmax_fit(D["X"], low=0.1, high=99.9)
    return np.asarray(A_strat.robust_minmax_apply(D["X"], mn, mx), np.float32), {"p_low": float(mn), "p_high": float(mx)}

def enc_np(X):
    return np.concatenate([encode(X[s:s + 2000], Y_ENC) for s in range(0, len(X), 2000)])

Y_XN, Y_NORMP = y_ds1_normalised()
Y_SETUP = LDa.make_search_data("ecg_setup", Y_XN, enc_np, seed=1)
log(f"ECG set-up beats: {len(Y_SETUP.sets['sep'])} different beats, {len(Y_SETUP.sets['gen'])} jittered copies of "
    f"{LC.GEN_BEATS} beats (DS1, labels unused); A_in {Y_SETUP.ain:.4f} input spikes per sample")


def y_cube_ecg(seed, scale):
    """Y for ECG input (200 channels) with the input size of this stage."""
    liq = y_cube(seed, Y_N_CH)
    liq.in_scale_fixed = float(scale)
    return liq

def y_ecg_setup():
    liqs = {s: y_cube(s, Y_N_CH) for s in Y_SEEDS}
    per_spike = liqs[Y_SEEDS[0]].in_scale_fixed
    target = Y_BUILD["input_neuron_rate_hz"]
    rate_per_spike = LCa.input_neuron_rate(Y_CTX, list(liqs.values()), per_spike, Y_SETUP.sets["sep"])
    log(f"  input-receiving neurons, recurrence off, at (c)'s size per spike ({per_spike:.3f}): {rate_per_spike:.2f} Hz "
        f"on ECG beats; on Maass's input: {target:.2f} Hz")
    if Y_INPUT_RULE == "match_maass_rate":
        scale, rate_in = LCa.bisect_scale(Y_CTX, list(liqs.values()), Y_SETUP.sets["sep"], target, log=log)
    elif Y_INPUT_RULE == "maass_per_spike":
        scale, rate_in = per_spike, rate_per_spike
    else:
        raise ValueError(f"Y_INPUT_RULE = {Y_INPUT_RULE!r}")
    liqs = {s: y_cube_ecg(s, scale) for s in Y_SEEDS}
    sw = run_sweep(liqs, Y_SETUP.sets, "ECG beats")
    out = {"sweep": sweep_rows(sw), "input_rule": Y_INPUT_RULE, "input_scale": scale, "input_neuron_rate_hz": rate_in,
           "input_scale_maass_per_spike": per_spike, "input_neuron_rate_at_maass_per_spike_hz": rate_per_spike,
           "input_neuron_rate_maass_hz": target, "A_in": Y_SETUP.ain, "normalisation": Y_NORMP,
           "g_maass": {s: float(liqs[s].g_maass) for s in Y_SEEDS}}
    g = sw["chosen_g"]
    if g is None:          # cannot happen with the band off unless every gap is NaN; fall back to Maass's own scale
        out["g"] = {s: float(liqs[s].g_maass) for s in Y_SEEDS}
        out["g_rule"] = "Maass's own scale (the rank measure gave no choice)"
    else:
        out["g"] = {s: float(g) for s in Y_SEEDS}
        out["g_rule"] = "(e)'s rank measure on ECG beats"
    out["abs_g"] = {s: liqs[s].ee_mean_push(out["g"][s]) for s in Y_SEEDS}
    if sw["peak_at_edge"]:
        log("  WARNING: the rank measure peaks at the edge of the grid even after the extensions; that g is used")
    return out


if "ecg_setup" in Y_STAGES:
    if not y_done("ecg_setup"):
        y_save("ecg_setup", y_ecg_setup())
    Y_ECG = y_load("ecg_setup")
    Y_G = {int(s): float(v) for s, v in Y_ECG["g"].items()}
    Y_IN_SCALE = float(Y_ECG["input_scale"])
    log(f"ECG set-up: input size {Y_IN_SCALE:.4g} ({Y_ECG['input_rule']}; input-receiving neurons {Y_ECG['input_neuron_rate_hz']:.2f} Hz, "
        f"Maass's input {Y_ECG['input_neuron_rate_maass_hz']:.2f} Hz); g = {Y_G} ({Y_ECG['g_rule']}); Maass's own scale {Y_ECG['g_maass']}")


## 10. Potential check: A. memory on ECG; B. quick classification on DS1

In [ ]:
# Stage "potential": does Y beat Y-off? DS1 only; DS2 is never touched.
#   A. Memory on ECG (no labels): from the 30 ms-filtered state at time t, a linear readout reconstructs the ECG
#      (its deviation from the mean beat) at t - lag. Y vs Y-off, same beats, same g.
#   B. Quick classification: (a)'s readout (config 11: linear SVM, (a)'s exact settings) on one repetition of the
#      authors' grouped 4-fold CV of DS1, from two readouts: the snapshot (state at R + 300 ms) and the whole beat
#      ((a)'s 32 windows of spike counts). Liquids: Y, Y-off, X (the paper's liquid, bugs fixed) and the input alone.
# The rule fixed before running (agreed): Y "has potential" if its snapshot Macro-F1 (mean of the 4 folds) is higher
# than Y-off's on every seed. Everything else is reported, not used to decide.
Y_SNAP_T = list(LC.SNAPSHOT_SAMPLES)          # R, R + 150 ms, R + 300 ms (plan, Readouts)
Y_SNAP_RULE = Y_SNAP_T.index(LC.STATE_READ_SAMPLE)   # R + 300 ms: where (e)'s measure was taken (plan, decision 8)
Y_FOLDS = [f for f in FOLDS if f["rep"] == Y_CV_REP and (f["rep"], f["fold"]) in set(CV_SPLITS)]
Y_MEM_LAG_RULE, Y_MEM_MIN_GAIN = LC.MEMORY_PASS_T_MS, LC.MEMORY_MIN_GAIN   # 100 ms, R^2 + 0.05 (plan, level 4)


# ---------------------------------------------------------------- A: memory on ECG
def y_memory():
    out = {}
    for s in Y_SEEDS:
        liq = y_cube_ecg(s, Y_IN_SCALE)
        on = LMe.memory_curve(Y_CTX, liq, Y_G[s], liq.in_scale_fixed, Y_SETUP.real_spikes, Y_SETUP.real_targets, s)
        off = LMe.memory_curve(Y_CTX, liq, Y_G[s], liq.in_scale_fixed, Y_SETUP.real_spikes, Y_SETUP.real_targets, s,
                               recurrence=False)
        gain = on[f"r2_{Y_MEM_LAG_RULE}"] - off[f"r2_{Y_MEM_LAG_RULE}"]
        out[s] = {"Y": on, "Y_off": off, "gain_at_rule_lag": float(gain), "pass": bool(np.isfinite(gain) and gain >= Y_MEM_MIN_GAIN)}
        log(f"    A, seed {s}: R^2 by lag {LC.MEMORY_T_MS} ms: Y {[round(on[f'r2_{l}'], 3) for l in LC.MEMORY_T_MS]}, "
            f"Y-off {[round(off[f'r2_{l}'], 3) for l in LC.MEMORY_T_MS]}")
    return {"seeds": out, "lags_ms": list(LC.MEMORY_T_MS), "rule": f"R^2(Y) - R^2(Y-off) >= {Y_MEM_MIN_GAIN} at {Y_MEM_LAG_RULE} ms, every seed",
            "pass": all(v["pass"] for v in out.values())}


# ---------------------------------------------------------------- B: simulate all of DS1 once per liquid and seed
def y_state_path(name, seed):
    return os.path.join(Y_DIR, "states", f"{name}_seed{seed}{'_smoke' if SMOKE_TEST else ''}.npz")

def y_x_liquid(seed):
    """X as the replication's fixed runs build it (the authors' class, A_in measured on these beats), at g = 1."""
    ain = measure_ain(Y_XN, np.arange(len(Y_XN)), Y_ENC)
    model = build_liquid(Y_N_CH, I_TARGET / (ain * LIQ["P_in"]), seed, True)
    return LL.from_authors_model(model), ain

def y_simulate(name, seed):
    """Window counts (32 x N), the 3 snapshots (3 x N) and spike counts per neuron for every DS1 beat."""
    p = y_state_path(name, seed)
    if os.path.exists(p):
        return
    extra = {}
    if name == "X":
        liq, ain = y_x_liquid(seed)
        g, scale, rec_on = 1.0, 1.0, True
        extra["A_in"] = ain
    else:
        liq = y_cube_ecg(seed, Y_IN_SCALE)
        g, scale, rec_on = Y_G[seed], liq.in_scale_fixed, name == "Y"
    sm = LS.Simulator(liq, device=DEVICE, dtype=torch.float32, g=g, in_scale=scale, recurrence=rec_on)
    n, N = len(Y_XN), liq.N
    W = np.zeros((n, LC.NWIN, N), np.uint8)
    S = np.zeros((n, len(Y_SNAP_T), N), np.float32)
    cnt = np.zeros((n, N), np.int32)
    t0 = time.time()
    for s in range(0, n, NEW_SIM_BATCH):
        x = encode_t(Y_XN[s:s + NEW_SIM_BATCH], Y_ENC)
        u0 = LS.start_voltages(liq, seed, np.arange(s, s + len(x)), Y_START_STREAM)
        o = sm.run(x, u0, LIQ["num_sim_steps"], rec={"windows": (LC.WIN, LC.NWIN), "states_at": Y_SNAP_T,
                                                     "neuron_counts": (0, LC.BEAT_SAMPLES)})
        W[s:s + len(x)] = o["windows"].clamp(max=255).to(torch.uint8).cpu().numpy()
        S[s:s + len(x)] = o["states_at"].cpu().numpy()
        cnt[s:s + len(x)] = o["neuron_counts"].cpu().numpy()
    rate = float(cnt.sum() / (n * N * LC.BEAT_SAMPLES * LC.SAMPLE_MS / 1000.0))
    np.savez(p, windows=W, snaps=S, rate_hz=rate, g=g, in_scale=scale, seconds=time.time() - t0, **extra)
    log(f"    simulated {name} seed {seed}: {n} beats, {rate:.2f} Hz, {time.time() - t0:.0f} s")

def y_input_windows():
    """The input alone: encoder spike counts in the same 32 windows (the 'no liquid' control of the replication)."""
    p = os.path.join(Y_DIR, "states", f"input_only{'_smoke' if SMOKE_TEST else ''}.npz")
    if not os.path.exists(p):
        n, T = len(Y_XN), LIQ["num_sim_steps"]
        W = np.zeros((n, LC.NWIN, Y_N_CH), np.uint8)
        for s in range(0, n, 2000):
            e = encode(Y_XN[s:s + 2000], Y_ENC)
            e = np.pad(e, ((0, 0), (0, max(0, T - e.shape[1])), (0, 0)))[:, :LC.WIN * LC.NWIN]
            W[s:s + len(e)] = e.reshape(len(e), LC.NWIN, LC.WIN, Y_N_CH).sum(2)
        np.savez(p, windows=W)
    return np.load(p)["windows"]


# ---------------------------------------------------------------- B: the readout
def y_readout(feat, fold):
    """(a)'s readout on one fold: the replication's fit_readout (StandardScaler + SGD linear SVM, alpha 1e-4,
    15 epochs, soft class weights) with the replication's readout seed for that split."""
    y = DATA[Y_FILT]["DS1"]["y"]
    tr, va = fold["train_idx"], fold["val_idx"]
    ftr = lambda p: feat[tr[p]].reshape(len(p), -1).astype(np.float32)
    fev = lambda p: feat[va[p]].reshape(len(p), -1).astype(np.float32)
    rng_seed = 2026 + fold["rep"] * 10 + fold["fold"]
    _, _, yp, _, _ = fit_readout(ftr, len(tr), y[tr], fev, len(va), "hinge", rng_seed, n_jobs=READOUT_JOBS)
    return compute_metrics(y[va], yp)

def y_rows_path():
    return os.path.join(Y_DIR, f"potential_B_rows{'_smoke' if SMOKE_TEST else ''}.json")

def y_classify():
    rows_path = y_rows_path()
    rows = json.load(open(rows_path)) if os.path.exists(rows_path) else []
    done = {(r["liquid"], r["seed"], r["readout"], r["fold"]) for r in rows}
    jobs = [("Y", s) for s in Y_SEEDS] + [("Y_off", s) for s in Y_SEEDS] + [("X", s) for s in X_SEEDS]
    for name, seed in jobs + [("input_only", 0)]:
        if name == "input_only":
            feats = {"whole_beat": y_input_windows()}
            rate = None
        else:
            y_simulate(name, seed)
            z = np.load(y_state_path(name, seed))
            feats = {"snapshot": z["snaps"][:, Y_SNAP_RULE], "whole_beat": z["windows"]}
            rate = float(z["rate_hz"])
        for ro, F in feats.items():
            for f in Y_FOLDS:
                key = (name, seed, ro, f["fold"])
                if key in done:
                    continue
                m = y_readout(F, f)
                rows.append({"liquid": name, "seed": seed, "readout": ro, "fold": f["fold"], "rate_hz": rate,
                             **{k: v for k, v in m.items() if k != "confusion_matrix"}, "confusion_matrix": m["confusion_matrix"]})
                done.add(key)
                jdump(rows, rows_path)
                log(f"    B {name} seed {seed} {ro} fold {f['fold']}: Macro-F1 {m['macro_f1']:.4f}, "
                    f"SVEB recall {m['recall_SVEB']:.3f}, VEB recall {m['recall_VEB']:.3f}")
    return rows

def y_verdict_B(rows):
    df = pd.DataFrame(rows)
    per_seed = df.groupby(["liquid", "readout", "seed"])["macro_f1"].mean().reset_index()
    def seed_means(liq, ro):
        d = per_seed[(per_seed.liquid == liq) & (per_seed.readout == ro)]
        return {int(s): float(v) for s, v in zip(d.seed, d.macro_f1)}
    out = {}
    for ro in ("snapshot", "whole_beat"):
        y_, off = seed_means("Y", ro), seed_means("Y_off", ro)
        out[ro] = {"Y": y_, "Y_off": off, "difference": {s: y_[s] - off[s] for s in y_ if s in off},
                   "Y_beats_Y_off_every_seed": bool(y_) and all(y_[s] > off.get(s, np.inf) for s in y_)}
    out["pass"] = out["snapshot"]["Y_beats_Y_off_every_seed"]
    out["rule"] = "snapshot (R + 300 ms) Macro-F1, mean of the 4 folds: Y > Y-off on every seed"
    return out


if "potential" in Y_STAGES:
    if not y_done("potential_A"):
        log("A. memory on ECG")
        y_save("potential_A", y_memory())
    Y_A = y_load("potential_A")
    log(f"A: Y keeps the ECG {Y_MEM_LAG_RULE} ms back better than Y-off by >= {Y_MEM_MIN_GAIN} R^2 on every seed: {Y_A['pass']}")
    log("B. quick classification on DS1")
    Y_B_ROWS = y_classify()
    Y_B = y_verdict_B(Y_B_ROWS)
    y_save("potential_B", Y_B)
    log(f"B: snapshot, Y beats Y-off on every seed: {Y_B['snapshot']['Y_beats_Y_off_every_seed']} "
        f"(Y {Y_B['snapshot']['Y']}, Y-off {Y_B['snapshot']['Y_off']}); whole beat: "
        f"{Y_B['whole_beat']['Y_beats_Y_off_every_seed']}")


## 11. Two of (a)'s claims: normalisation (claim 3) and the encoder without a liquid (claim 7)

In [ ]:
# Two of (a)'s claims, checked cheaply (no liquid simulation). DS2 is not used by claim 7; claim 3 reads the
# replication's own runs (which include its DS2 runs).
#   Claim 3 ("robust normalisation provides an additional positive contribution"): robust - standard Macro-F1 in the
#     replication's 24-configuration runs, paired within each (filter, encoder, readout) setting; one test pooled
#     over the 12 settings and one per setting (Holm). Needs REPLICATION_TABLE (the replication's output folder, or its run table tidy.csv).
#   Claim 7 ("Multi-threshold coding ... may facilitate the generation of discriminative transient dynamics within
#     the liquid"): the encoder comparison repeated with NO liquid (the encoder's own spikes in (a)'s 32 windows),
#     on the 8 front-end x readout settings and the authors' 12 CV splits. If Multi-threshold still leads without a
#     liquid, its lead comes from the encoding, not from liquid dynamics.
# Tests: the replication's paired Nadeau-Bengio corrected t-test over the CV splits (lsm2.stats), Holm within each
# family. Verdict on CV only: "A better" / "B better" if |difference| >= 0.01 and Holm p < 0.05; "comparable" if the
# 95% interval lies within +-0.02; otherwise "inconclusive" (the replication's thresholds, without its DS2 condition).
from lsm2 import stats as LSt
C7_SETTINGS = [(f, n, r) for f in ("morph", "median") for n in ("robust", "standard") for r in ("svm", "lr")]
C7_ENCODERS = ("multi", "rate", "latency")
C7_RATE_SEEDS = tuple(X_SEEDS)            # the Rate encoder is random: one run per seed (as the replication)


def cv_verdict(r):
    if r is None:
        return "missing"
    if abs(r["cv_diff"]) >= 0.01 and r.get("p_holm", 1.0) < 0.05:
        return "A better" if r["cv_diff"] > 0 else "B better"
    if r["ci_low"] >= -0.02 and r["ci_high"] <= 0.02:
        return "comparable"
    return "inconclusive"

def cv_family(items):
    rows = LSt.holm([dict(r) if r else None for _, r in items])
    return [{"comparison": lab, **({} if r is None else {k: v for k, v in r.items() if not isinstance(v, (dict, list))}),
             "verdict": cv_verdict(r)} for (lab, _), r in zip(items, rows)]

def _as_runs(df):
    """A table in lsm2.stats' format (spec_split, spec_seed, macro_f1)."""
    return df.rename(columns={"split": "spec_split", "seed": "spec_seed"})

def _pooled(df, by_cols):
    """Average over settings within each (split, seed): one pooled run per split and seed."""
    return df.groupby(["spec_split", "spec_seed"], dropna=False).macro_f1.mean().reset_index()


# ---------------------------------------------------------------- claim 7: encoders without a liquid
def c7_rows_path():
    return os.path.join(Y_DIR, f"claim7_rows{'_smoke' if SMOKE_TEST else ''}.json")

def encoder_windows(Xn, enc):
    """The encoder's own spikes counted in (a)'s 32 windows of 20 steps (the replication's 'no liquid' control)."""
    Cn = 200 if enc == "multi" else N_IN[enc]
    out = np.zeros((len(Xn), LC.NWIN, Cn), np.uint8)
    for s in range(0, len(Xn), 2000):
        S = encode(Xn[s:s + 2000], enc)
        T = min(S.shape[1], LC.NWIN * LC.WIN)
        Sp = np.zeros((S.shape[0], LC.NWIN * LC.WIN, Cn), np.uint8)
        Sp[:, :T] = S[:, :T]
        out[s:s + len(S)] = Sp.reshape(len(S), LC.NWIN, LC.WIN, Cn).sum(2)
    return out

def y_claim7_runs():
    rows = json.load(open(c7_rows_path())) if os.path.exists(c7_rows_path()) else []
    done = {(r["filter"], r["norm"], r["encoder"], r["readout"], r["split"], r["seed"]) for r in rows}
    for filt in sorted({s[0] for s in C7_SETTINGS}):
        D = DATA[filt]["DS1"]
        for norm in sorted({s[1] for s in C7_SETTINGS if s[0] == filt}):
            readouts = [s[2] for s in C7_SETTINGS if s[0] == filt and s[1] == norm]
            for rep, fold in CV_SPLITS:
                f = FOLD_BY[(rep, fold)]
                split = f"cv-r{rep}f{fold}"
                Xn = None
                for enc in C7_ENCODERS:
                    for seed in (C7_RATE_SEEDS if enc == "rate" else (None,)):
                        todo = [r for r in readouts if (filt, norm, enc, r, split, seed) not in done]
                        if not todo:
                            continue
                        if Xn is None:
                            Xn, _ = normalize_cv(D["X"], f["train_idx"], norm)      # the authors' CV path
                        torch.manual_seed(int(stable_hash(("claim7", filt, norm, enc, split, seed), 6), 16))
                        F = encoder_windows(Xn, enc)
                        tr, va = f["train_idx"], f["val_idx"]
                        ftr = lambda p: F[tr[p]].reshape(len(p), -1).astype(np.float32)
                        fev = lambda p: F[va[p]].reshape(len(p), -1).astype(np.float32)
                        for ro in todo:
                            _, _, yp, _, _ = fit_readout(ftr, len(tr), D["y"][tr], fev, len(va),
                                                         "hinge" if ro == "svm" else "log_loss", 2026 + rep * 10 + fold,
                                                         n_jobs=READOUT_JOBS)
                            m = compute_metrics(D["y"][va], yp)
                            rows.append({"filter": filt, "norm": norm, "encoder": enc, "readout": ro, "split": split,
                                         "seed": seed, **{k: v for k, v in m.items() if k != "confusion_matrix"}})
                            done.add((filt, norm, enc, ro, split, seed))
                            jdump(rows, c7_rows_path())
                            log(f"    claim 7: {filt}/{norm}/{enc}/{ro} {split} seed {seed}: Macro-F1 {m['macro_f1']:.4f}")
                        del F
    return rows

def encoder_tests(df, label):
    """multi - rate and multi - latency: pooled over the settings present, then per setting; plus Friedman over
    (setting x split) blocks with the seeds averaged."""
    df = _as_runs(df)
    keyc = ["filter", "norm", "readout"]
    items = []
    for other in ("rate", "latency"):
        a, b = df[df.encoder == "multi"], df[df.encoder == other]
        items.append((f"{label}: multi - {other}, pooled", LSt.compare(_pooled(a, keyc), _pooled(b, keyc))))
    pooled = cv_family(items)
    per = []
    for other in ("rate", "latency"):
        its = []
        for st, g in df.groupby(keyc):
            its.append((f"{label}: multi - {other}, {'/'.join(st)}", LSt.compare(g[g.encoder == "multi"], g[g.encoder == other])))
        per += cv_family(its)
    blocks = df.groupby(keyc + ["spec_split", "encoder"]).macro_f1.mean().unstack("encoder").dropna()
    fr = None
    if len(blocks) >= 3 and blocks.shape[1] == 3:
        st = sstats.friedmanchisquare(*[blocks[e].values for e in C7_ENCODERS])
        ranks = blocks[list(C7_ENCODERS)].rank(axis=1, ascending=False).mean()
        fr = {"label": label, "blocks": int(len(blocks)), "chi2": float(st.statistic), "p": float(st.pvalue),
              **{f"mean_rank_{e}": float(ranks[e]) for e in C7_ENCODERS},
              **{f"mean_macro_f1_{e}": float(blocks[e].mean()) for e in C7_ENCODERS}}
    return pooled, per, fr


# ---------------------------------------------------------------- the replication's runs (claims 3 and 7)
def _runs_from_folder(root):
    """One row per run of the replication's output folder: runs/<run_id>/config.json (spec, group) and metrics.json.
    The same table as tidy.csv (the replication's run table), only the columns used here."""
    rows = []
    for d in sorted(glob.glob(os.path.join(root, "runs", "*"))):
        try:
            c = json.load(open(os.path.join(d, "config.json")))
            m = json.load(open(os.path.join(d, "metrics.json")))
        except Exception:
            continue
        if c.get("group") not in ("main_cv", "main_final"):
            continue
        r = {"run_id": os.path.basename(d), "group": c.get("group")}
        for k, v in c.get("spec", {}).items():
            r["s_" + k] = json.dumps(v, sort_keys=True) if isinstance(v, (dict, list)) else v
        r.update({k: v for k, v in m.items() if not isinstance(v, (dict, list))})
        rows.append(r)
    return pd.DataFrame(rows)

def replication_runs():
    """The replication's 24-configuration runs (CV and DS2). REPLICATION_TABLE is either the replication's output
    folder (the one holding runs/) or a run table built from it (tidy.csv)."""
    if not REPLICATION_TABLE:
        return None
    p = os.path.expanduser(REPLICATION_TABLE)
    if os.path.isdir(p):
        t = _runs_from_folder(p)
        if t.empty:
            log(f"claims: no replication runs found in {os.path.join(p, 'runs')}")
            return None
        t.to_csv(os.path.join(Y_DIR, "replication_main_runs.csv"), index=False)
        log(f"claims: {len(t)} replication runs read from {os.path.join(p, 'runs')}")
    elif os.path.exists(p):
        t = pd.read_csv(p, low_memory=False)
    else:
        log(f"claims: REPLICATION_TABLE {p} does not exist")
        return None
    t = t[t.group.isin(["main_cv", "main_final"])]
    return t.rename(columns={"s_split": "split", "s_seed": "seed", "s_filter": "filter", "s_norm": "norm",
                             "s_encoder": "encoder", "s_readout": "readout", "s_mode": "mode"})

def y_claim3(rep_runs):
    out = []
    for mode in sorted(rep_runs["mode"].dropna().unique()):
        m = _as_runs(rep_runs[rep_runs["mode"] == mode])
        A, B = m[m.norm == "robust"], m[m.norm == "standard"]
        keyc = ["filter", "encoder", "readout"]
        pooled = LSt.family(f"claim3 {mode} pooled", [(f"{mode}: robust - standard, pooled 12 settings",
                                                      LSt.compare(_pooled(A, keyc), _pooled(B, keyc)))])
        its = []
        for st, g in m.groupby(keyc):
            its.append((f"{mode}: robust - standard, {'/'.join(st)}", LSt.compare(g[g.norm == "robust"], g[g.norm == "standard"])))
        out += [{"mode": mode, **r} for r in pooled + LSt.family(f"claim3 {mode}", its)]
    return out


if "claim3" in Y_STAGES or "claim7" in Y_STAGES:
    from scipy import stats as sstats
    Y_REP = replication_runs()
    if Y_REP is None:
        log("claims: REPLICATION_TABLE not set or not found: claim 3 and the with-liquid side of claim 7 are skipped")

if "claim3" in Y_STAGES and Y_REP is not None:
    c3 = pd.DataFrame(y_claim3(Y_REP))
    c3.to_csv(os.path.join(Y_DIR, "claim3_normalisation.csv"), index=False)
    for _, r in c3[c3.comparison.str.contains("pooled")].iterrows():
        log(f"claim 3, {r['comparison']}: CV {r['cv_diff']:+.4f} [{r['ci_low']:+.4f}, {r['ci_high']:+.4f}], "
            f"DS2 {r['ds2_diff']:+.4f}: {r['verdict']}")

if "claim7" in Y_STAGES:
    log("claim 7: the encoders without a liquid")
    c7 = pd.DataFrame(y_claim7_runs())
    p7, s7, f7 = encoder_tests(c7, "no liquid")
    res = p7 + s7
    fr = [f7] if f7 else []
    if Y_REP is not None:          # the same tests on the replication's liquid runs, same settings and splits
        for mode in ("published", "fixed"):
            L = Y_REP[(Y_REP["mode"] == mode) & Y_REP.split.astype(str).str.startswith("cv-")
                      & Y_REP.split.isin(c7.split.unique())]
            if len(L):
                pl, sl, fl = encoder_tests(L, f"{mode} liquid")
                res += pl + sl
                fr += [fl] if fl else []
    pd.DataFrame(res).to_csv(os.path.join(Y_DIR, "claim7_encoders.csv"), index=False)
    pd.DataFrame(fr).to_csv(os.path.join(Y_DIR, "claim7_friedman.csv"), index=False)
    for r in res:
        if "pooled" in r["comparison"]:
            log(f"claim 7, {r['comparison']}: CV {r.get('cv_diff', np.nan):+.4f} "
                f"[{r.get('ci_low', np.nan):+.4f}, {r.get('ci_high', np.nan):+.4f}]: {r['verdict']}")


## 12. Summary

In [ ]:
# Summary: WORK_DIR/ytest/summary_ytest.md and the figures in WORK_DIR/ytest/figures/. Tolerates missing stages.
def _fmt(x, d=3):
    return "-" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{x:.{d}f}"

def _sweep_fig(rec, title, fn, g_marks):
    rows = [r for r in rec["rows"] if r["g"] > 0]
    if not rows:
        return
    ab = [r["abs_g"] for r in rows]
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
    for a, key, lab in zip(ax, ("rate_hz", "gap", "twin_ratio"),
                           ("firing (Hz)", "(e)'s measure: separation - generalisation", "extra-spike distance, late / early")):
        a.plot(ab, [np.nan if r.get(key) is None else r[key] for r in rows], "o-", color=SERIES[0])
        a.set_xscale("log"); a.set_xlabel("mean EE push per spike (thresholds)"); a.set_ylabel(lab)
        for v, c, ls in g_marks:
            a.axvline(v, color=c, ls=ls, lw=1.3)
    ax[2].axhline(1.0, color="#999", lw=0.8)
    fig.suptitle(title + " (orange: chosen g; dashed: Maass's own scale)")
    fig.tight_layout(); fig.savefig(os.path.join(Y_DIR, "figures", fn), dpi=110); plt.close(fig)

lines = ["# Y vs Y-off: does a liquid built by Maass's recipe beat its own recurrence-off version?", "",
         f"Notebook build {NOTEBOOK_BUILD}; {datetime.datetime.utcnow():%Y-%m-%d %H:%M} UTC; "
         f"{'SMOKE TEST: the numbers mean nothing' if SMOKE_TEST else 'full run'}. DS2 is not used.", ""]

if y_done("build"):
    b = y_load("build")
    lines += ["## 1. Build check on Maass's own input (20 Hz Poisson)", "",
              f"Maass's own weight scale: g = {', '.join(f'{float(v):.3g}' for v in b['g_maass'].values())} (seeds {', '.join(b['g_maass'])}). "
              f"(e)'s measure chose g = {_fmt(b['sweep']['chosen_g'], 3)} on this input. "
              f"Input-receiving neurons fire {b['input_neuron_rate_hz']:.2f} Hz with recurrence off.", "",
              "(c)'s four-segment task: accuracy of the template in each segment, read at the end of the 1 s input "
              "(segment 1 = 750-1000 ms back; 0.5 is chance).", "",
              "| weights | seed | g | Y: seg 1-4 | Y-off: seg 1-4 | Y rate (Hz) | older 3, Y vs Y-off |", "|---|---|---|---|---|---|---|"]
    for name, per in b["segment_task"].items():
        for s, r in per.items():
            lines.append(f"| {name.replace('_', ' ')} | {s} | {r['g']:.3g} | {' / '.join(f'{a:.3f}' for a in r['on']['accuracy'])} | "
                         f"{' / '.join(f'{a:.3f}' for a in r['off']['accuracy'])} | {r['on']['rate_hz']:.2f} | "
                         f"{r['older_on']:.3f} vs {r['older_off']:.3f} |")
    lines += ["", f"**Stop point** (Y keeps the 3 older segments better than Y-off at Maass's scale, every seed): "
              f"{'PASS' if b['pass'] else 'FAIL'}", ""]
    gm = float(np.mean([b["abs_g_maass"][s] for s in b["abs_g_maass"]]))
    ch = [r["abs_g"] for r in b["sweep"]["rows"] if r["g"] == b["sweep"]["chosen_g"]]
    _sweep_fig(b["sweep"], "Y on Maass's input", "sweep_maass.png",
               [(gm, "#999", "--")] + ([(ch[0], SERIES[1], "-")] if ch else []))
    fig, ax = plt.subplots(figsize=(6, 3.4))
    per = b["segment_task"]["maass_scale"]
    xs = np.arange(1, SEGMENTS + 1)
    for i, (s, r) in enumerate(per.items()):
        ax.plot(xs, r["on"]["accuracy"], "o-", color=SERIES[0], alpha=0.8, label="Y" if i == 0 else None)
        ax.plot(xs, r["off"]["accuracy"], "s--", color=SERIES[1], alpha=0.8, label="Y-off" if i == 0 else None)
    ax.axhline(0.5, color="#999", lw=0.8); ax.set_xticks(xs); ax.set_xticklabels(["750-1000", "500-750", "250-500", "0-250"])
    ax.set_xlabel("segment, ms before the readout"); ax.set_ylabel("accuracy"); ax.legend(frameon=False)
    ax.set_title("(c)'s segment task at Maass's scale (one line per seed)")
    fig.tight_layout(); fig.savefig(os.path.join(Y_DIR, "figures", "segment_task.png"), dpi=110); plt.close(fig)

if y_done("ecg_setup"):
    e = y_load("ecg_setup")
    lines += ["## 2. Y on ECG beats (config 11's front end, DS1, no labels)", "",
              f"A_in {e['A_in']:.4f} input spikes per sample. Input size {e['input_scale']:.4g} ({e['input_rule']}): input-receiving "
              f"neurons fire {e['input_neuron_rate_hz']:.2f} Hz with recurrence off (on Maass's input: {e['input_neuron_rate_maass_hz']:.2f} Hz; "
              f"at (c)'s size per spike, {e['input_scale_maass_per_spike']:.3f}, they would fire {e['input_neuron_rate_at_maass_per_spike_hz']:.2f} Hz).", "",
              f"Gain: g = {', '.join(f'{v:.3g}' for v in e['g'].values())} ({e['g_rule']}); Maass's own scale would be "
              f"{', '.join(f'{v:.3g}' for v in e['g_maass'].values())}. Peak at the grid's edge: {e['sweep']['peak_at_edge']}.", "",
              "| g | EE push | rate (Hz) | separation | generalisation | gap | extra-spike late/early |", "|---|---|---|---|---|---|---|"]
    for r in e["sweep"]["rows"]:
        lines.append(f"| {r['g']:.4g} | {r['abs_g']:.4f} | {r['rate_hz']:.2f} | {r['sep']:.2f} | {r['gen']:.2f} | {r['gap']:.2f} | {_fmt(r.get('twin_ratio'), 2)} |")
    lines.append("")
    gm = float(np.mean([e["g_maass"][s] for s in e["g_maass"]]))
    _ref = y_cube_ecg(Y_SEEDS[0], e["input_scale"])
    _sweep_fig(e["sweep"], "Y on ECG beats", "sweep_ecg.png",
               [(_ref.ee_mean_push(gm), "#999", "--"), (_ref.ee_mean_push(float(list(e["g"].values())[0])), SERIES[1], "-")])

if y_done("potential_A"):
    a = y_load("potential_A")
    lines += ["## 3A. Memory on ECG (no labels)", "",
              "R^2 of reconstructing the ECG (deviation from the mean beat) `lag` ms back from the current state.", "",
              "| seed | liquid | " + " | ".join(f"{l} ms" for l in a["lags_ms"]) + " |", "|---|---|" + "---|" * len(a["lags_ms"])]
    for s, r in a["seeds"].items():
        for nm, key in (("Y", "Y"), ("Y-off", "Y_off")):
            lines.append(f"| {s} | {nm} | " + " | ".join(_fmt(r[key][f'r2_{l}']) for l in a["lags_ms"]) + " |")
    lines += ["", f"**A** ({a['rule']}): {'PASS' if a['pass'] else 'FAIL'}", ""]
    fig, ax = plt.subplots(figsize=(6, 3.4))
    for i, (s, r) in enumerate(a["seeds"].items()):
        ax.plot(a["lags_ms"], [r["Y"][f"r2_{l}"] for l in a["lags_ms"]], "o-", color=SERIES[0], label="Y" if i == 0 else None)
        ax.plot(a["lags_ms"], [r["Y_off"][f"r2_{l}"] for l in a["lags_ms"]], "s--", color=SERIES[1], label="Y-off" if i == 0 else None)
    ax.set_xlabel("lag (ms)"); ax.set_ylabel("R^2"); ax.legend(frameon=False); ax.set_title("Memory on ECG (one line per seed)")
    fig.tight_layout(); fig.savefig(os.path.join(Y_DIR, "figures", "memory_ecg.png"), dpi=110); plt.close(fig)

if y_done("potential_B") and os.path.exists(y_rows_path()):
    vb = y_load("potential_B")
    df = pd.DataFrame(json.load(open(y_rows_path())))
    lines += ["## 3B. Quick classification on DS1 (authors' folds, repetition 1)", "",
              "Mean (SD) over seeds of the per-seed mean over folds.", "",
              "| liquid | readout | Macro-F1 | accuracy | SVEB recall | VEB recall | seeds |", "|---|---|---|---|---|---|---|"]
    order = {"Y": 0, "Y_off": 1, "X": 2, "input_only": 3}
    ps = df.groupby(["liquid", "readout", "seed"])[["macro_f1", "accuracy", "recall_SVEB", "recall_VEB"]].mean().reset_index()
    for (lq, ro), d in sorted(ps.groupby(["liquid", "readout"]), key=lambda kv: (order.get(kv[0][0], 9), kv[0][1])):
        cell = lambda c: f"{d[c].mean():.4f} ({d[c].std(ddof=1):.4f})" if len(d) > 1 else f"{d[c].mean():.4f}"
        lines.append(f"| {lq.replace('_', '-')} | {ro.replace('_', ' ')} | {cell('macro_f1')} | {cell('accuracy')} | "
                     f"{cell('recall_SVEB')} | {cell('recall_VEB')} | {len(d)} |")
    lines += ["", "Y minus Y-off, Macro-F1 per seed:", ""]
    for ro in ("snapshot", "whole_beat"):
        dd = vb[ro]["difference"]
        lines.append(f"- {ro.replace('_', ' ')}: " + ", ".join(f"seed {s}: {v:+.4f}" for s, v in dd.items())
                     + f" (every seed positive: {vb[ro]['Y_beats_Y_off_every_seed']})")
    lines += ["", f"**B, the agreed rule** ({vb['rule']}): {'PASS' if vb['pass'] else 'FAIL'}", ""]
    fig, ax = plt.subplots(figsize=(7, 3.4))
    labs, k = [], 0
    for (lq, ro), d in sorted(ps.groupby(["liquid", "readout"]), key=lambda kv: (kv[0][1], order.get(kv[0][0], 9))):
        ax.scatter(np.full(len(d), k), d["macro_f1"], color=SERIES[order.get(lq, 3) % 4], zorder=3)
        ax.bar(k, d["macro_f1"].mean(), color=SERIES[order.get(lq, 3) % 4], alpha=0.35)
        labs.append(f"{lq.replace('_', '-')}\n{ro.replace('_', ' ')}"); k += 1
    ax.set_xticks(range(k)); ax.set_xticklabels(labs, fontsize=8); ax.set_ylabel("Macro-F1 (DS1 CV)")
    ax.set_title("Quick classification (dots: seeds)")
    fig.tight_layout(); fig.savefig(os.path.join(Y_DIR, "figures", "classification_ds1.png"), dpi=110); plt.close(fig)

_c3p = os.path.join(Y_DIR, "claim3_normalisation.csv")
if os.path.exists(_c3p):
    c3 = pd.read_csv(_c3p)
    lines += ["## 4. Claim 3: robust vs standard normalisation (the replication's runs)", "",
              "Robust minus standard Macro-F1, paired within each filter/encoder/readout setting; CV over the 12 splits "
              "(corrected t-test), DS2 difference alongside; Holm within each family.", "",
              "| comparison | CV difference | 95% interval | Holm p | DS2 difference | verdict |", "|---|---|---|---|---|---|"]
    for _, r in c3.iterrows():
        lines.append(f"| {r['comparison']} | {r['cv_diff']:+.4f} | [{r['ci_low']:+.4f}, {r['ci_high']:+.4f}] | "
                     f"{r.get('p_holm', np.nan):.3f} | {r['ds2_diff']:+.4f} | {r['verdict']} |")
    lines.append("")
_c7p = os.path.join(Y_DIR, "claim7_encoders.csv")
if os.path.exists(_c7p):
    c7r = pd.read_csv(_c7p)
    lines += ["## 5. Claim 7: does Multi-threshold still lead without a liquid? (DS1 CV)", "",
              "Multi minus the other encoder, Macro-F1. 'no liquid' = the encoder's spikes in (a)'s 32 windows; "
              "'published / fixed liquid' = the replication's runs on the same settings and splits.", "",
              "| comparison | CV difference | 95% interval | Holm p | verdict |", "|---|---|---|---|---|"]
    for _, r in c7r[c7r.comparison.str.contains("pooled")].iterrows():
        lines.append(f"| {r['comparison']} | {r.get('cv_diff', np.nan):+.4f} | [{r.get('ci_low', np.nan):+.4f}, "
                     f"{r.get('ci_high', np.nan):+.4f}] | {r.get('p_holm', np.nan):.3f} | {r['verdict']} |")
    _fp = os.path.join(Y_DIR, "claim7_friedman.csv")
    if os.path.exists(_fp) and os.path.getsize(_fp) > 1:
        fr = pd.read_csv(_fp)
        lines += ["", "| encoders compared on | blocks | Friedman p | mean rank multi / rate / latency | mean Macro-F1 multi / rate / latency |",
                  "|---|---|---|---|---|"]
        for _, r in fr.iterrows():
            lines.append(f"| {r['label']} | {r['blocks']} | {r['p']:.2e} | {r['mean_rank_multi']:.2f} / {r['mean_rank_rate']:.2f} / "
                         f"{r['mean_rank_latency']:.2f} | {r['mean_macro_f1_multi']:.4f} / {r['mean_macro_f1_rate']:.4f} / {r['mean_macro_f1_latency']:.4f} |")
    lines += ["", "Per-setting results: `claim7_encoders.csv`.", ""]

lines += ["## Figures", ""] + [f"![{f}](figures/{f})" for f in sorted(os.listdir(os.path.join(Y_DIR, "figures")))]
atomic_write_text(os.path.join(Y_DIR, "summary_ytest.md"), "\n".join(lines) + "\n")
print("\n".join(lines))


## 13. End of session

In [ ]:
# End of session: where the results are.
print("results:", os.path.join(WORK_DIR, "ytest"))
print("summary:", os.path.join(WORK_DIR, "ytest", "summary_ytest.md"))
print(f"session time: {(time.time() - SESSION_START) / 3600:.2f} h")
